# DL TCN 벤치마크
Purpose: define a safe dual-T4 causal TCN benchmark using the immutable shared validation contract.

> Warning: this is an oracle/sanity-only synthetic-data benchmark. Real accuracy is NOT VERIFIED and this is not medical or diagnostic evidence.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
from bisect import bisect_right
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    recall_score,
    roc_auc_score,
)
from torch.nn.parallel import DistributedDataParallel
from torch.utils.data import DataLoader, Dataset, DistributedSampler

SERIES_ID = "mvp3-oracle-v1"
EXPECTED_SPLIT_COUNTS = {"train": 24, "validation": 6, "locked_test": 6}
DATA_STATUS = "oracle/sanity"
REAL_ACCURACY_STATUS = "NOT VERIFIED"
DEVICE_SYNCHRONIZATION_STATUS = "NOT_AVAILABLE_TRUTH_ONLY"
RUN_TRAINING = False
RUN_LOCKED_TEST = False

SEQUENCE_OUTPUT_ROOT = Path("/kaggle/working/goal15_dl_sequences")
ML_VIEW_ROOT = Path('/kaggle/working/goal15_ml_view')
DL_BENCHMARK_OUTPUT_ROOT = Path('/kaggle/working/goal15_dl_tcn_benchmark')
ML_BENCHMARK_OUTPUT_ROOT = Path('/kaggle/working/goal15_ml_benchmark')
SEQUENCE_LENGTHS_SECONDS = (300, 600)
PATTERN_TARGET = "pattern_binary"
ONSET_EVENT_TARGET = "event_binary"
STAGE_TARGET = 'stage_code'
STAGE_CODES = ('LOW', 'MEDIUM', 'HIGH', 'DECREASING', 'RECOVERY')
BEHAVIOR_CODES = (
    'ear_covering',
    'exit_attempt',
    'head_turn_away',
    'motion_freeze',
    'movement_reduction',
    'repetitive_body_movement',
    'repetitive_hand_movement',
    'repetitive_object_contact',
    'sustained_pressure_or_contact',
    'withdrawal_movement',
)
CAUSAL_FACTORS = (
    'autonomic_arousal', 'motor_activation', 'cognitive_load',
    'sleep_pressure', 'sensory_context', 'recovery_capacity', 'social_context',
)
ROLLING_STATISTICS = ('mean', 'std', 'slope')
ROLLING_WINDOWS_SECONDS = (5, 15, 30, 60, 180, 300)
TIME_FEATURE_COLUMNS = ('time_sin', 'time_cos', 'weekday_sin', 'weekday_cos', 'is_awake')
CONTEXT_FEATURE_COLUMNS = (
    'context__sleep', 'context__transition', 'context__meal_context',
    'context__focused_task', 'context__moderate_activity',
    'context__light_activity', 'context__wake_rest', 'context__sedentary_activity',
)
APPROVED_CONTEXTS = (
    'sleep', 'transition', 'meal_context', 'focused_task',
    'moderate_activity', 'light_activity', 'wake_rest', 'sedentary_activity',
)
ALLOWED_FEATURE_COLUMNS = tuple(
    [
        feature
        for factor in CAUSAL_FACTORS
        for feature in (
            f'{factor}__robust_z',
            *(
                f'{factor}__{statistic}_{window_seconds}s'
                for window_seconds in ROLLING_WINDOWS_SECONDS
                for statistic in ROLLING_STATISTICS
            ),
        )
    ]
    + list(TIME_FEATURE_COLUMNS)
    + list(CONTEXT_FEATURE_COLUMNS)
)
PREDICTION_COLUMNS = [
    'model_family', 'model_name', 'series_id', 'dataset_id', 'run_id',
    'person_key', 'canonical_time', 'split_role', 'target', 'label',
    'probability', 'threshold',
]
METRIC_COLUMNS = [
    'model_family', 'model_name', 'series_id', 'split_role', 'target',
    'metric', 'value', 'support', 'data_status',
]
SEQUENCE_INDEX_COLUMNS = {
    'person_key', 'run_id', 'dataset_id', 'context', 'split_role',
    PATTERN_TARGET, ONSET_EVENT_TARGET, 'hard_negative', STAGE_TARGET,
    *BEHAVIOR_CODES, 'window_start', 'window_end', 'prediction_time',
    'length_seconds', 'window_id', 'sample_type',
}
SEED = 20260728
WANDB_PROJECT = 'multisensor-goal15-benchmark'
WANDB_GROUP = 'deep-learning-tcn'
WANDB_TAGS = ['oracle-sanity', 'mvp3', 'split-24-6-6', 'not-real-verified']


## 1. 입력 무결성 검증
Task 4가 만든 manifest, train-only 정규화, 여섯 sequence index와 원본 full causal timeline을 독립적으로 검증합니다.

In [ ]:
@dataclass(frozen=True)
class VerifiedSequenceInputs:
    source_dataset_hash: str
    split_hash: str
    index_paths: Mapping[str, Path]
    timeline_paths: Mapping[str, Path]
    normalization: Mapping[str, Any]
    source_root: Path = Path('.')


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def _require_sha256(value: Any, field: str) -> str:
    if not isinstance(value, str) or len(value) != 64:
        raise ValueError(f'invalid SHA-256 for {field}')
    try:
        int(value, 16)
    except ValueError as exc:
        raise ValueError(f'invalid SHA-256 for {field}') from exc
    return value


def _safe_child(root: Path, name: Any, field: str) -> Path:
    if not isinstance(name, str) or not name or Path(name).name != name:
        raise ValueError(f'invalid relative path for {field}')
    path = root / name
    if not path.is_file():
        raise FileNotFoundError(f'missing {field}: {path}')
    return path


def _validate_timeline_schema(parquet: pq.ParquetFile, split_role: str) -> None:
    columns = list(parquet.schema_arrow.names)
    ordered_features = [column for column in columns if column in ALLOWED_FEATURE_COLUMNS]
    if ordered_features != list(ALLOWED_FEATURE_COLUMNS):
        raise ValueError(f'exact feature schema/order mismatch: {split_role}')
    for feature in ALLOWED_FEATURE_COLUMNS:
        data_type = parquet.schema_arrow.field(feature).type
        if not (pa.types.is_boolean(data_type) or pa.types.is_integer(data_type) or pa.types.is_floating(data_type)):
            raise ValueError(f'non-numeric causal feature: {feature}')


def _stream_role_people(parquet: pq.ParquetFile, split_role: str) -> set[str]:
    required = {'person_key', 'split_role'}
    if not required.issubset(parquet.schema_arrow.names):
        raise ValueError(f'timeline identity schema mismatch: {split_role}')
    people: set[str] = set()
    for row_group in range(parquet.num_row_groups):
        frame = parquet.read_row_group(row_group, columns=['person_key', 'split_role']).to_pandas()
        if frame[['person_key', 'split_role']].isna().any().any():
            raise ValueError(f'null timeline identity: {split_role}')
        if not frame['split_role'].eq(split_role).all():
            raise ValueError(f'timeline split binding mismatch: {split_role}')
        if not frame['person_key'].map(lambda value: isinstance(value, str) and bool(value.strip())).all():
            raise ValueError(f'invalid person identity: {split_role}')
        people.update(frame['person_key'])
    return people


def _verify_sequence_index(
    path: Path,
    split_role: str,
    expected_length: int,
    metadata: Mapping[str, Any],
) -> None:
    parquet = pq.ParquetFile(path)
    if metadata.get('row_count') != parquet.metadata.num_rows:
        raise ValueError(f'sequence row_count mismatch: {path.name}')
    if not SEQUENCE_INDEX_COLUMNS.issubset(parquet.schema_arrow.names):
        raise ValueError(f'sequence index schema mismatch: {path.name}')
    allowed_sample_types = (
        {'positive_centered', 'hard_negative', 'matched_baseline'}
        if split_role == 'train'
        else {'sliding'}
    )
    seen_people: set[str] = set()
    previous_order: tuple[str, str, str, int, int, str] | None = None
    for row_group in range(parquet.num_row_groups):
        frame = parquet.read_row_group(
            row_group,
            columns=list(SEQUENCE_INDEX_COLUMNS),
        ).to_pandas()
        identity_columns = ['person_key', 'run_id', 'dataset_id', 'context', 'window_id', 'sample_type']
        if frame[identity_columns].isna().any().any():
            raise ValueError(f'null sequence identity: {path.name}')
        for column in identity_columns:
            if not frame[column].map(lambda value: isinstance(value, str) and bool(value) and value == value.strip()).all():
                raise ValueError(f'invalid sequence identity {column}: {path.name}')
        if not frame['context'].isin(APPROVED_CONTEXTS).all():
            raise ValueError(f'invalid context domain: {path.name}')
        if not frame['split_role'].eq(split_role).all():
            raise ValueError(f'sequence index split mismatch: {path.name}')
        if not frame['sample_type'].isin(allowed_sample_types).all():
            raise ValueError(f'sampled or wrong sequence manifest: {path.name}')
        if not frame['length_seconds'].eq(expected_length).all():
            raise ValueError(f'sequence length mismatch: {path.name}')
        binary_columns = [PATTERN_TARGET, ONSET_EVENT_TARGET, 'hard_negative', *BEHAVIOR_CODES]
        if frame[binary_columns].isna().any().any() or not all(frame[column].isin((0, 1)).all() for column in binary_columns):
            raise ValueError(f'sequence labels must be exact binary: {path.name}')
        if frame[STAGE_TARGET].isna().any() or not frame[STAGE_TARGET].isin(('NO_EVENT', *STAGE_CODES)).all():
            raise ValueError(f'invalid stage labels: {path.name}')
        expected_pattern = frame[STAGE_TARGET].ne('NO_EVENT').astype(np.int8)
        if not frame[PATTERN_TARGET].astype(np.int8).eq(expected_pattern).all():
            raise ValueError(f'pattern/stage mismatch: {path.name}')
        if (frame[PATTERN_TARGET].eq(1) & frame['hard_negative'].eq(1)).any():
            raise ValueError('pattern and hard_negative cannot overlap')
        behavior_positive = frame.loc[:, list(BEHAVIOR_CODES)].eq(1).any(axis=1)
        if (behavior_positive & frame[PATTERN_TARGET].eq(0) & frame['hard_negative'].eq(0)).any():
            raise ValueError('behavior-positive ordinary baseline is forbidden')
        parsed_times: dict[str, pd.Series] = {}
        for column in ('window_start', 'window_end', 'prediction_time'):
            times = pd.to_datetime(frame[column], errors='raise')
            if not isinstance(times.dtype, pd.DatetimeTZDtype) or str(times.dtype.tz) != 'UTC':
                raise ValueError(f'sequence time must be timezone-aware UTC: {column}')
            parsed_times[column] = times
        if not parsed_times['window_end'].eq(parsed_times['prediction_time']).all():
            raise ValueError(f'window_end must equal prediction_time: {path.name}')
        expected_start = parsed_times['prediction_time'] - pd.Timedelta(seconds=expected_length - 1)
        if not parsed_times['window_start'].eq(expected_start).all():
            raise ValueError(f'causal window_start mismatch: {path.name}')
        for offset, row in enumerate(frame.itertuples(index=False)):
            prediction_time = pd.Timestamp(parsed_times['prediction_time'].iloc[offset])
            identity = f'{row.person_key}|{row.run_id}|{row.dataset_id}|{row.context}|{prediction_time.isoformat()}|{expected_length}'
            expected_window_id = hashlib.sha256(identity.encode()).hexdigest()
            if row.window_id != expected_window_id:
                raise ValueError('deterministic window_id mismatch')
            order = (row.person_key, row.run_id, row.dataset_id, expected_length, prediction_time.value, row.context)
            if previous_order is not None and order <= previous_order:
                raise ValueError(f'sequence index is not globally ordered and unique: {path.name}')
            previous_order = order
        seen_people.update(frame['person_key'])
    if split_role != 'train' and len(seen_people) != EXPECTED_SPLIT_COUNTS[split_role]:
        raise ValueError(f'full evaluation membership mismatch: {split_role}')


def verify_sequence_inputs(
    sequence_root: Path = SEQUENCE_OUTPUT_ROOT,
    timeline_root: Path = ML_VIEW_ROOT,
) -> VerifiedSequenceInputs:
    sequence_manifest_path = sequence_root / 'sequence_manifest.json'
    timeline_manifest_path = timeline_root / 'view_manifest.json'
    if not sequence_manifest_path.is_file() or not timeline_manifest_path.is_file():
        raise FileNotFoundError('sequence and full timeline manifests are required')
    sequence_manifest = json.loads(sequence_manifest_path.read_text())
    timeline_manifest = json.loads(timeline_manifest_path.read_text())
    for manifest in (sequence_manifest, timeline_manifest):
        if manifest.get('series_id') != SERIES_ID or manifest.get('data_status') != DATA_STATUS:
            raise ValueError('shared Dataset identity/status mismatch')
    source_dataset_hash = _require_sha256(sequence_manifest.get('source_dataset_hash'), 'source_dataset_hash')
    split_hash = _require_sha256(sequence_manifest.get('split_hash'), 'split_hash')
    if source_dataset_hash != _require_sha256(timeline_manifest.get('source_dataset_hash'), 'timeline source_dataset_hash'):
        raise ValueError('source_dataset_hash mismatch')
    if split_hash != _require_sha256(timeline_manifest.get('split_hash'), 'timeline split_hash'):
        raise ValueError('split_hash mismatch')

    normalization_metadata = sequence_manifest.get('normalization')
    if not isinstance(normalization_metadata, dict):
        raise ValueError('normalization metadata is required')
    normalization_path = _safe_child(sequence_root, normalization_metadata.get('path'), 'normalization')
    if sha256_file(normalization_path) != _require_sha256(normalization_metadata.get('sha256'), 'normalization'):
        raise ValueError('normalization hash mismatch')
    normalization = json.loads(normalization_path.read_text())
    if normalization.get('fit_split_role') != 'train':
        raise ValueError('normalization must be fitted on train only')
    if normalization.get('series_id') != SERIES_ID or normalization.get('source_hash') != source_dataset_hash:
        raise ValueError('normalization source identity mismatch')
    feature_statistics = normalization.get('features')
    if not isinstance(feature_statistics, dict) or set(feature_statistics) != set(ALLOWED_FEATURE_COLUMNS):
        raise ValueError('normalization exact feature schema mismatch')
    for feature in ALLOWED_FEATURE_COLUMNS:
        statistic = feature_statistics[feature]
        if not isinstance(statistic, dict) or set(statistic) != {'median', 'iqr'}:
            raise ValueError(f'invalid train normalization: {feature}')
        values = np.asarray([statistic['median'], statistic['iqr']], dtype=np.float64)
        if not np.isfinite(values).all() or values[1] <= 0:
            raise ValueError(f'invalid train normalization values: {feature}')

    timeline_files = timeline_manifest.get('dl_timeline_files')
    if not isinstance(timeline_files, dict) or set(timeline_files) != set(EXPECTED_SPLIT_COUNTS):
        raise ValueError('full causal timeline roles are incomplete')
    timeline_paths: dict[str, Path] = {}
    role_people: dict[str, set[str]] = {}
    expected_feature_types: tuple[str, ...] | None = None
    for split_role, metadata in timeline_files.items():
        if metadata.get('view_kind') != 'full_causal_timeline' or metadata.get('sampled') is not False:
            raise ValueError(f'sampled full timeline is forbidden: {split_role}')
        if metadata.get('feature_columns') != list(ALLOWED_FEATURE_COLUMNS):
            raise ValueError(f'timeline feature schema/order mismatch: {split_role}')
        path = _safe_child(timeline_root, metadata.get('path'), f'{split_role} timeline')
        if sha256_file(path) != _require_sha256(metadata.get('sha256'), f'{split_role} timeline'):
            raise ValueError(f'timeline hash mismatch: {split_role}')
        parquet = pq.ParquetFile(path)
        if metadata.get('row_count') != parquet.metadata.num_rows:
            raise ValueError(f'timeline row_count mismatch: {split_role}')
        if metadata.get('columns') != list(parquet.schema_arrow.names):
            raise ValueError(f'timeline schema metadata mismatch: {split_role}')
        _validate_timeline_schema(parquet, split_role)
        feature_types = tuple(str(parquet.schema_arrow.field(feature).type) for feature in ALLOWED_FEATURE_COLUMNS)
        if expected_feature_types is None:
            expected_feature_types = feature_types
        elif feature_types != expected_feature_types:
            raise ValueError(f'feature type schema mismatch: {split_role}')
        role_people[split_role] = _stream_role_people(parquet, split_role)
        timeline_paths[split_role] = path
    counts = {role: len(people) for role, people in role_people.items()}
    if counts != EXPECTED_SPLIT_COUNTS:
        raise ValueError(f'person count mismatch: {counts}')
    roles = list(EXPECTED_SPLIT_COUNTS)
    if any(role_people[roles[left]] & role_people[roles[right]] for left in range(3) for right in range(left + 1, 3)):
        raise ValueError('person leakage across split roles')

    sequence_files = sequence_manifest.get('files')
    expected_files = {
        f'{role}_{length}'
        for role in EXPECTED_SPLIT_COUNTS
        for length in SEQUENCE_LENGTHS_SECONDS
    }
    if not isinstance(sequence_files, dict) or set(sequence_files) != expected_files:
        raise ValueError('sequence manifest must declare six exact role/length files')
    index_paths: dict[str, Path] = {}
    for name, metadata in sequence_files.items():
        split_role, length_text = name.rsplit('_', 1)
        if int(length_text) not in SEQUENCE_LENGTHS_SECONDS:
            raise ValueError(f'wrong sequence manifest length: {name}')
        path = _safe_child(sequence_root, metadata.get('path'), name)
        if sha256_file(path) != _require_sha256(metadata.get('sha256'), name):
            raise ValueError(f'sequence index hash mismatch: {name}')
        _verify_sequence_index(path, split_role, int(length_text), metadata)
        index_paths[name] = path
    return VerifiedSequenceInputs(source_dataset_hash, split_hash, index_paths, timeline_paths, normalization)


## 2. 지연 시퀀스 데이터셋
인덱스와 full timeline을 row-group 단위로만 읽고, Task 4의 train 통계로만 정규화합니다.

In [ ]:
class Goal15SequenceDataset(Dataset):
    def __init__(
        self,
        index_path: Path,
        timeline_path: Path,
        normalization: Mapping[str, Any],
        split_role: str,
    ) -> None:
        if split_role not in EXPECTED_SPLIT_COUNTS:
            raise ValueError(f'unknown split role: {split_role}')
        if normalization.get('fit_split_role') != 'train':
            raise ValueError('only train-fitted normalization is accepted')
        if list(normalization.get('features', {}).keys()) != sorted(ALLOWED_FEATURE_COLUMNS):
            if set(normalization.get('features', {})) != set(ALLOWED_FEATURE_COLUMNS):
                raise ValueError('normalization feature schema mismatch')
        self.split_role = split_role
        self.index_file = pq.ParquetFile(index_path)
        self.timeline_file = pq.ParquetFile(timeline_path)
        self.normalization = normalization
        self._index_ends = np.cumsum([
            self.index_file.metadata.row_group(index).num_rows
            for index in range(self.index_file.num_row_groups)
        ]).tolist()
        self._index_cache: tuple[int, pd.DataFrame] | None = None
        self._person_ranges = self._catalog_index_person_ranges()
        self._timeline_catalog = RowGroupIntervalIndex(self.timeline_file, split_role)
        self._row_group_cache = BoundedRowGroupCache(
            max_groups=ROW_GROUP_CACHE_MAX_GROUPS,
            max_bytes=ROW_GROUP_CACHE_MAX_BYTES,
        )
        optional_audit = [column for column in ('forecast_60s', 'phase') if column in self.timeline_file.schema_arrow.names]
        self._timeline_read_columns = list(dict.fromkeys([
            *TIMELINE_READ_COLUMNS, *optional_audit, *ALLOWED_FEATURE_COLUMNS,
        ]))

    def _catalog_index_person_ranges(self) -> dict[str, list[range]]:
        ranges: dict[str, list[range]] = {}
        global_offset = 0
        previous_person: str | None = None
        closed_people: set[str] = set()
        for row_group in range(self.index_file.num_row_groups):
            people = self.index_file.read_row_group(row_group, columns=['person_key']).column('person_key').to_pylist()
            start = 0
            while start < len(people):
                person = people[start]
                end = start + 1
                while end < len(people) and people[end] == person:
                    end += 1
                if not isinstance(person, str) or not person.strip():
                    raise ValueError('index person identity is invalid')
                if previous_person is not None and person != previous_person:
                    closed_people.add(previous_person)
                if person in closed_people:
                    raise ValueError('index person rows are not globally contiguous')
                ranges.setdefault(person, []).append(range(global_offset + start, global_offset + end))
                previous_person = person
                start = end
            global_offset += len(people)
        return ranges

    def person_ranges(self) -> Mapping[str, Sequence[range]]:
        return self._person_ranges

    def _catalog_timeline_row_groups(self) -> dict[str, list[tuple[int, pd.Timestamp, pd.Timestamp]]]:
        catalog: dict[str, list[tuple[int, pd.Timestamp, pd.Timestamp]]] = {}
        columns = ['dataset_id', 'canonical_time', 'split_role']
        for row_group in range(self.timeline_file.num_row_groups):
            frame = self.timeline_file.read_row_group(row_group, columns=columns).to_pandas()
            if frame.empty or not frame['split_role'].eq(self.split_role).all():
                raise ValueError('timeline row group role mismatch')
            if frame['dataset_id'].nunique(dropna=False) != 1:
                raise ValueError('timeline row group must bind one dataset_id')
            times = pd.to_datetime(frame['canonical_time'], utc=True, errors='raise')
            if times.duplicated().any() or not times.is_monotonic_increasing:
                raise ValueError('timeline row group time must be unique and ordered')
            dataset_id = str(frame['dataset_id'].iloc[0])
            catalog.setdefault(dataset_id, []).append((row_group, times.iloc[0], times.iloc[-1]))
        return catalog

    def __len__(self) -> int:
        return int(self.index_file.metadata.num_rows)

    def _index_row(self, item: int) -> pd.Series:
        if item < 0:
            item += len(self)
        if item < 0 or item >= len(self):
            raise IndexError(item)
        row_group = bisect_right(self._index_ends, item)
        start = 0 if row_group == 0 else self._index_ends[row_group - 1]
        if self._index_cache is None or self._index_cache[0] != row_group:
            self._index_cache = (row_group, self.index_file.read_row_group(row_group).to_pandas())
        return self._index_cache[1].iloc[item - start]

    def _window_frame(self, index_row: pd.Series) -> pd.DataFrame:
        dataset_id = str(index_row['dataset_id'])
        window_start = pd.Timestamp(index_row['window_start'])
        window_end = pd.Timestamp(index_row['window_end'])
        pieces: list[pd.DataFrame] = []
        for row_group in self._timeline_catalog.overlapping(dataset_id, window_start, window_end):
            frame = self._row_group_cache.get(
                row_group,
                lambda row_group=row_group: self.timeline_file.read_row_group(
                    row_group, columns=self._timeline_read_columns,
                ).to_pandas(),
            )
            times = pd.to_datetime(frame['canonical_time'], utc=True, errors='raise')
            pieces.append(frame.loc[times.between(window_start, window_end)])
        if not pieces:
            raise ValueError(f'window has no timeline rows: {index_row["window_id"]}')
        window = pd.concat(pieces, ignore_index=True).sort_values('canonical_time')
        expected_length = int(index_row['length_seconds'])
        if len(window) != expected_length:
            raise ValueError(f'window length mismatch: {index_row["window_id"]}')
        for key in ('person_key', 'run_id', 'dataset_id'):
            if not window[key].eq(index_row[key]).all():
                raise ValueError(f'window crosses {key}')
        times = pd.to_datetime(window['canonical_time'], utc=True, errors='raise')
        if times.iloc[0] != window_start or times.iloc[-1] != window_end:
            raise ValueError('window boundary mismatch')
        if not times.diff().iloc[1:].eq(pd.Timedelta(seconds=1)).all():
            raise ValueError('window is not causal contiguous 1 Hz')
        endpoint = window.iloc[-1]
        if times.iloc[-1] != pd.Timestamp(index_row['prediction_time']):
            raise ValueError('endpoint timestamp mismatch')
        if endpoint['context'] != index_row['context']:
            raise ValueError('endpoint context mismatch')
        for column in (PATTERN_TARGET, ONSET_EVENT_TARGET, 'hard_negative', STAGE_TARGET, *BEHAVIOR_CODES):
            if endpoint[column] != index_row[column]:
                raise ValueError(f'endpoint label mismatch: {column}')
        for column in ('forecast_60s', 'phase'):
            if column in index_row.index and column in endpoint.index and endpoint[column] != index_row[column]:
                raise ValueError(f'endpoint label mismatch: {column}')
        return window

    def __getitem__(self, item: int) -> dict[str, Any]:
        row = self._index_row(item)
        window = self._window_frame(row)
        matrix = window.loc[:, list(ALLOWED_FEATURE_COLUMNS)].to_numpy(dtype=np.float32)
        for column_index, feature in enumerate(ALLOWED_FEATURE_COLUMNS):
            statistic = self.normalization['features'][feature]
            matrix[:, column_index] = (matrix[:, column_index] - float(statistic['median'])) / float(statistic['iqr'])
        if not np.isfinite(matrix).all():
            raise ValueError('normalized sequence contains non-finite values')
        behaviors = np.asarray([row[code] for code in BEHAVIOR_CODES], dtype=np.float32)
        stage_index = STAGE_CODES.index(row[STAGE_TARGET]) if row[STAGE_TARGET] in STAGE_CODES else -1
        return {
            'features': torch.from_numpy(matrix),
            'mask': torch.ones(len(matrix), dtype=torch.bool),
            PATTERN_TARGET: torch.tensor(float(row[PATTERN_TARGET]), dtype=torch.float32),
            ONSET_EVENT_TARGET: torch.tensor(float(row[ONSET_EVENT_TARGET]), dtype=torch.float32),
            'hard_negative': torch.tensor(float(row['hard_negative']), dtype=torch.float32),
            STAGE_TARGET: torch.tensor(stage_index, dtype=torch.long),
            'behaviors': torch.from_numpy(behaviors),
            'dataset_id': str(row['dataset_id']),
            'run_id': str(row['run_id']),
            'person_key': str(row['person_key']),
            'canonical_time': str(pd.Timestamp(row['prediction_time'])),
            'split_role': self.split_role,
            'window_id': str(row['window_id']),
        }


## 3. Causal TCN과 조건부 손실
왼쪽 패딩만 쓰는 dilated residual backbone과 event, 5-stage, 10-behavior head를 정의합니다.

In [ ]:
class ChannelLayerNorm1d(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.normalization = nn.LayerNorm(channels)

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        return self.normalization(values.transpose(1, 2)).transpose(1, 2)


class CausalConvBlock(nn.Module):
    def __init__(self, hidden_size: int, kernel_size: int, dilation: int, dropout: float) -> None:
        super().__init__()
        self.left_padding = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(hidden_size, hidden_size, kernel_size, dilation=dilation, padding=0)
        self.conv2 = nn.Conv1d(hidden_size, hidden_size, kernel_size, dilation=dilation, padding=0)
        self.norm1 = ChannelLayerNorm1d(hidden_size)
        self.norm2 = ChannelLayerNorm1d(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def _causal_conv(self, values: torch.Tensor, convolution: nn.Conv1d) -> torch.Tensor:
        return convolution(F.pad(values, (self.left_padding, 0)))

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        residual = values
        values = self.dropout(F.gelu(self.norm1(self._causal_conv(values, self.conv1))))
        values = self.dropout(F.gelu(self.norm2(self._causal_conv(values, self.conv2))))
        return values + residual


class Goal15TCN(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 128, dropout: float = 0.1) -> None:
        super().__init__()
        self.input_projection = nn.Conv1d(input_size, hidden_size, kernel_size=1)
        self.blocks = nn.ModuleList([
            CausalConvBlock(hidden_size, kernel_size=3, dilation=dilation, dropout=dropout)
            for dilation in (1, 2, 4, 8, 16, 32)
        ])
        self.event_head = nn.Linear(hidden_size, 1)
        self.stage_head = nn.Linear(hidden_size, 5)
        self.behavior_head = nn.Linear(hidden_size, 10)

    def forward(self, features: torch.Tensor, mask: torch.Tensor) -> dict[str, torch.Tensor]:
        values = self.input_projection(features.transpose(1, 2))
        for block in self.blocks:
            values = block(values)
        time_mask = mask.unsqueeze(1).to(dtype=values.dtype)
        pooled = (values * time_mask).sum(dim=2) / time_mask.sum(dim=2).clamp_min(1.0)
        return {
            'event_logits': self.event_head(pooled).squeeze(-1),
            'stage_logits': self.stage_head(pooled),
            'behavior_logits': self.behavior_head(pooled),
        }


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def assert_parameter_budget(model: nn.Module) -> int:
    count = count_trainable_parameters(model)
    if not 500_000 <= count <= 5_000_000:
        raise ValueError(f'TCN parameter budget violation: {count:,}')
    return count


def assert_right_padding_invariance(model: Goal15TCN, input_size: int) -> None:
    model.eval()
    device = next(model.parameters()).device
    prefix = torch.randn(2, 300, input_size, device=device)
    short_mask = torch.ones(2, 300, dtype=torch.bool, device=device)
    right_padding = torch.randn(2, 37, input_size, device=device)
    padded = torch.cat([prefix, right_padding], dim=1)
    padded_mask = torch.cat([short_mask, torch.zeros(2, 37, dtype=torch.bool, device=device)], dim=1)
    with torch.no_grad():
        short_outputs = model(prefix, short_mask)
        padded_outputs = model(padded, padded_mask)
    for head in ('event_logits', 'stage_logits', 'behavior_logits'):
        if not torch.allclose(short_outputs[head], padded_outputs[head], atol=1e-6, rtol=1e-5):
            raise AssertionError(f'right padding changed valid-prefix logits: {head}')


def masked_multitask_loss(
    outputs: Mapping[str, torch.Tensor],
    batch: Mapping[str, torch.Tensor],
    *,
    stage_weight: float = 1.0,
    behavior_weight: float = 1.0,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    pattern = batch[PATTERN_TARGET].float()
    event_bce = F.binary_cross_entropy_with_logits(outputs['event_logits'], pattern)
    stage_mask = pattern.eq(1)
    stage_loss = (
        F.cross_entropy(outputs['stage_logits'][stage_mask], batch[STAGE_TARGET][stage_mask])
        if stage_mask.any()
        else outputs['stage_logits'].sum() * 0.0
    )
    behavior_labels = batch['behaviors'].float()
    behavior_positive = behavior_labels.eq(1).any(dim=1)
    hard_negative = batch['hard_negative'].eq(1)
    invalid_positive = behavior_positive & ~(stage_mask | hard_negative)
    if invalid_positive.any():
        raise ValueError('behavior-positive rows must be pattern or hard negative')
    behavior_mask = stage_mask | (hard_negative & behavior_positive)
    behavior_loss = (
        F.binary_cross_entropy_with_logits(
            outputs['behavior_logits'][behavior_mask], behavior_labels[behavior_mask]
        )
        if behavior_mask.any()
        else outputs['behavior_logits'].sum() * 0.0
    )
    total = event_bce + stage_weight * stage_loss + behavior_weight * behavior_loss
    return total, {'event_bce': event_bce, 'stage_loss': stage_loss, 'behavior_loss': behavior_loss}


## 4. T4 x2 DDP 안전 게이트와 학습 루프
GPU 검사는 DDP, W&B, Dataset/DataLoader, trainer보다 먼저 실행됩니다.

In [ ]:
def require_exactly_two_cuda_devices() -> None:
    count = torch.cuda.device_count()
    if count != 2:
        raise RuntimeError(f'T4 x2가 필요합니다. 감지된 CUDA 장치: {count}')


def set_deterministic_seed(seed: int, rank: int = 0) -> None:
    effective_seed = seed + rank
    random.seed(effective_seed)
    np.random.seed(effective_seed)
    torch.manual_seed(effective_seed)
    torch.cuda.manual_seed_all(effective_seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def setup_ddp() -> tuple[int, int, int]:
    required = ('LOCAL_RANK', 'RANK', 'WORLD_SIZE')
    missing = [name for name in required if name not in os.environ]
    if missing:
        raise RuntimeError(f'torchrun 환경이 필요합니다. 누락: {missing}')
    local_rank = int(os.environ['LOCAL_RANK'])
    rank = int(os.environ['RANK'])
    world_size = int(os.environ['WORLD_SIZE'])
    if world_size != 2 or local_rank not in (0, 1):
        raise RuntimeError('T4 x2는 torchrun --nproc_per_node=2로 실행해야 합니다.')
    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend='nccl', init_method='env://', rank=rank, world_size=world_size)
    return rank, local_rank, world_size


def cleanup_ddp() -> None:
    if dist.is_available() and dist.is_initialized():
        dist.barrier()
        dist.destroy_process_group()


def login_wandb_from_kaggle_secret() -> bool:
    try:
        from kaggle_secrets import UserSecretsClient
        import wandb

        key = UserSecretsClient().get_secret("WANDB_API_KEY")
        if not key:
            raise RuntimeError('Kaggle W&B secret is empty')
        return bool(wandb.login(key=key, verify=True))
    except Exception as exc:
        print(f'W&B 비활성화: {type(exc).__name__}')
        return False


def _move_batch(batch: Mapping[str, Any], device: torch.device) -> dict[str, Any]:
    return {
        key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


def train_one_epoch(
    model: DistributedDataParallel,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    device: torch.device,
    *,
    epoch: int,
    sampler: DistributedSampler,
) -> dict[str, float]:
    sampler.set_epoch(epoch)
    model.train()
    running = {'total': 0.0, 'event_bce': 0.0, 'stage_loss': 0.0, 'behavior_loss': 0.0}
    examples = 0
    for raw_batch in loader:
        batch = _move_batch(raw_batch, device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(batch['features'], batch['mask'])
            loss, components = masked_multitask_loss(outputs, batch)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        batch_size = int(batch['features'].shape[0])
        examples += batch_size
        running['total'] += float(loss.detach()) * batch_size
        for name, value in components.items():
            running[name] += float(value.detach()) * batch_size
    return {name: value / max(examples, 1) for name, value in running.items()}


## 5. 공통 평가와 validation champion
동일 prediction/metric schema를 사용하며 locked test는 champion 선택 후 별도 gate에서만 평가합니다.

In [ ]:
def expected_calibration_error(labels: np.ndarray, probability: np.ndarray, bins: int = 10) -> float:
    value = 0.0
    edges = np.linspace(0.0, 1.0, bins + 1)
    for lower, upper in zip(edges[:-1], edges[1:], strict=True):
        mask = (probability >= lower) & (probability < upper if upper < 1 else probability <= upper)
        if mask.any():
            value += float(mask.mean()) * abs(float(labels[mask].mean()) - float(probability[mask].mean()))
    return value


def _segments(values: np.ndarray) -> list[tuple[int, int]]:
    changes = np.diff(np.pad(values.astype(np.int8), (1, 1)))
    starts = np.flatnonzero(changes == 1)
    ends = np.flatnonzero(changes == -1) - 1
    return list(zip(starts.tolist(), ends.tolist(), strict=True))


def _event_summary(frame: pd.DataFrame) -> tuple[float, float, float]:
    truth_events = detected = false_alerts = 0
    duration_hours = len(frame) / 3600.0
    for _, person in frame.sort_values(['person_key', 'canonical_time']).groupby('person_key', sort=False):
        truth = person['label'].to_numpy(dtype=bool)
        predicted = person['probability'].ge(person['threshold']).to_numpy(dtype=bool)
        events = _segments(truth)
        truth_events += len(events)
        detected += sum(bool(predicted[start:end + 1].any()) for start, end in events)
        false_alerts += len(_segments(predicted & ~truth))
    event_recall = detected / truth_events if truth_events else 0.0
    event_precision = detected / (detected + false_alerts) if detected + false_alerts else 0.0
    event_f1 = 2 * event_precision * event_recall / (event_precision + event_recall) if event_precision + event_recall else 0.0
    return event_recall, false_alerts / duration_hours if duration_hours else 0.0, event_f1


def _binary_metric_rows(predictions: pd.DataFrame, model_name: str, target: str) -> list[dict[str, Any]]:
    labels = predictions['label'].to_numpy(dtype=np.int8)
    probability = predictions['probability'].to_numpy(dtype=np.float64)
    predicted = probability >= predictions['threshold'].to_numpy(dtype=np.float64)
    two_classes = len(np.unique(labels)) == 2
    aucpr = float(average_precision_score(labels, probability)) if two_classes else np.nan
    auroc = float(roc_auc_score(labels, probability)) if two_classes else np.nan
    person_scores = []
    for _, person in predictions.assign(predicted=predicted).groupby('person_key', sort=False):
        person_scores.append((
            f1_score(person['label'], person['predicted'], zero_division=0),
            recall_score(person['label'], person['predicted'], zero_division=0),
        ))
    person_macro_f1 = float(np.mean([score[0] for score in person_scores])) if person_scores else np.nan
    person_macro_recall = float(np.mean([score[1] for score in person_scores])) if person_scores else np.nan
    event_recall, false_alerts_per_hour, event_f1 = _event_summary(predictions) if target == PATTERN_TARGET else (np.nan, np.nan, np.nan)
    values = {
        'aucpr': aucpr,
        'auroc': auroc,
        'event_recall': event_recall,
        'event_f1': event_f1,
        'false_alerts_per_hour': false_alerts_per_hour,
        'row_f1': float(f1_score(labels, predicted, zero_division=0)),
        'row_recall': float(recall_score(labels, predicted, zero_division=0)),
        'brier_score': float(brier_score_loss(labels, probability)),
        'ece': expected_calibration_error(labels, probability),
        'person_macro_f1': person_macro_f1,
        'person_macro_recall': person_macro_recall,
    }
    return [
        {
            'model_family': 'deep_learning_tcn', 'model_name': model_name,
            'series_id': SERIES_ID, 'split_role': predictions['split_role'].iloc[0],
            'target': target, 'metric': metric, 'value': value,
            'support': len(labels), 'data_status': DATA_STATUS,
        }
        for metric, value in values.items()
    ]


def _aggregate_metric_row(
    model_name: str,
    split_role: str,
    target: str,
    metric: str,
    value: float,
    support: int,
) -> dict[str, Any]:
    return {
        'model_family': 'deep_learning_tcn', 'model_name': model_name,
        'series_id': SERIES_ID, 'split_role': split_role, 'target': target,
        'metric': metric, 'value': value, 'support': support, 'data_status': DATA_STATUS,
    }


def compute_metrics_from_predictions(
    predictions: pd.DataFrame,
    *,
    model_name: str,
) -> pd.DataFrame:
    missing = sorted(set(PREDICTION_COLUMNS).difference(predictions.columns))
    if missing or predictions.empty:
        raise ValueError(f'common predictions are empty or incomplete: {missing}')
    split_roles = set(predictions['split_role'])
    if len(split_roles) != 1:
        raise ValueError('metric input must contain exactly one split role')
    split_role = str(next(iter(split_roles)))
    metric_rows: list[dict[str, Any]] = []
    for target, target_rows in predictions.groupby('target', sort=True):
        metric_rows.extend(_binary_metric_rows(target_rows, model_name, target))

    decision_keys = ['dataset_id', 'run_id', 'person_key', 'canonical_time']
    stage_rows = predictions.loc[predictions['target'].str.startswith('stage::')]
    if not stage_rows.empty:
        stage_probability = stage_rows.pivot(index=decision_keys, columns='target', values='probability')
        stage_labels = stage_rows.loc[stage_rows['label'].eq(1)].set_index(decision_keys)['target']
        if len(stage_labels) != len(stage_probability) or not stage_labels.index.is_unique:
            raise ValueError('conditional stage truth must be exactly one class per pattern row')
        stage_truth = stage_labels.loc[stage_probability.index]
        stage_prediction = stage_probability.idxmax(axis=1)
        stage_macro_f1 = float(f1_score(stage_truth, stage_prediction, average='macro', zero_division=0))
        stage_balanced_accuracy = float(balanced_accuracy_score(stage_truth, stage_prediction))
        metric_rows.extend([
            _aggregate_metric_row(model_name, split_role, 'stage::all', 'stage_macro_f1', stage_macro_f1, len(stage_truth)),
            _aggregate_metric_row(model_name, split_role, 'stage::all', 'stage_balanced_accuracy', stage_balanced_accuracy, len(stage_truth)),
        ])

    behavior_rows = predictions.loc[predictions['target'].str.startswith('behavior::')]
    if not behavior_rows.empty:
        behavior_labels = behavior_rows.pivot(index=decision_keys, columns='target', values='label').astype(np.int8)
        behavior_probability = behavior_rows.pivot(index=decision_keys, columns='target', values='probability').loc[behavior_labels.index, behavior_labels.columns]
        flat_labels = behavior_labels.to_numpy().ravel()
        flat_probability = behavior_probability.to_numpy().ravel()
        behavior_micro_aucpr = float(average_precision_score(flat_labels, flat_probability)) if len(np.unique(flat_labels)) == 2 else np.nan
        per_behavior_aucpr = [
            average_precision_score(behavior_labels[column], behavior_probability[column])
            for column in behavior_labels
            if behavior_labels[column].nunique() == 2
        ]
        behavior_macro_aucpr = float(np.mean(per_behavior_aucpr)) if per_behavior_aucpr else np.nan
        metric_rows.extend([
            _aggregate_metric_row(model_name, split_role, 'behavior::all', 'behavior_micro_aucpr', behavior_micro_aucpr, len(behavior_labels)),
            _aggregate_metric_row(model_name, split_role, 'behavior::all', 'behavior_macro_aucpr', behavior_macro_aucpr, len(behavior_labels)),
        ])
    return pd.DataFrame(metric_rows, columns=METRIC_COLUMNS)


def select_validation_threshold(predictions: pd.DataFrame) -> float:
    pattern = predictions.loc[
        predictions['split_role'].eq('validation') & predictions['target'].eq(PATTERN_TARGET)
    ].copy()
    if pattern.empty or len(pattern) != len(predictions.loc[predictions['target'].eq(PATTERN_TARGET)]):
        raise ValueError('threshold selection accepts validation pattern rows only')
    candidates = np.unique(pattern['probability'].to_numpy(dtype=np.float64))
    scores: list[tuple[float, float, float, float]] = []
    for threshold in candidates:
        pattern['threshold'] = float(threshold)
        predicted = pattern['probability'].ge(threshold)
        event_recall, false_alerts_per_hour, _ = _event_summary(pattern)
        scores.append((
            float(f1_score(pattern['label'], predicted, zero_division=0)),
            event_recall,
            -false_alerts_per_hour,
            float(threshold),
        ))
    return max(scores)[3] if scores else 0.5


def evaluate_common_schema(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    *,
    split_role: str,
    model_name: str,
    thresholds: Mapping[str, float] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if split_role not in EXPECTED_SPLIT_COUNTS:
        raise ValueError(f'unknown split role: {split_role}')
    thresholds = dict(thresholds or {})
    rows: list[dict[str, Any]] = []
    model.eval()
    with torch.no_grad():
        for raw_batch in loader:
            batch = _move_batch(raw_batch, device)
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(batch['features'], batch['mask'])
            pattern_probability = torch.sigmoid(outputs['event_logits']).cpu().numpy()
            stage_probability = torch.softmax(outputs['stage_logits'], dim=1).cpu().numpy()
            behavior_probability = torch.sigmoid(outputs['behavior_logits']).cpu().numpy()
            pattern = batch[PATTERN_TARGET].cpu().numpy().astype(np.int8)
            hard_negative = batch['hard_negative'].cpu().numpy().astype(np.int8)
            behaviors = batch['behaviors'].cpu().numpy().astype(np.int8)
            stage_labels = batch[STAGE_TARGET].cpu().numpy().astype(np.int64)
            behavior_positive = behaviors.eq(1).any(axis=1) if hasattr(behaviors, 'eq') else (behaviors == 1).any(axis=1)
            behavior_mask = (pattern == 1) | ((hard_negative == 1) & behavior_positive)
            for item in range(len(pattern)):
                base = {
                    'model_family': 'deep_learning_tcn', 'model_name': model_name,
                    'series_id': SERIES_ID, 'dataset_id': raw_batch['dataset_id'][item],
                    'run_id': raw_batch['run_id'][item], 'person_key': raw_batch['person_key'][item],
                    'canonical_time': raw_batch['canonical_time'][item], 'split_role': split_role,
                }
                rows.append({**base, 'target': PATTERN_TARGET, 'label': int(pattern[item]), 'probability': float(pattern_probability[item]), 'threshold': float(thresholds.get(PATTERN_TARGET, 0.5))})
                if pattern[item] == 1:
                    for stage_index, stage in enumerate(STAGE_CODES):
                        rows.append({**base, 'target': f'stage::{stage}', 'label': int(stage_labels[item] == stage_index), 'probability': float(stage_probability[item, stage_index]), 'threshold': float(thresholds.get(f'stage::{stage}', 0.5))})
                if behavior_mask[item]:
                    for behavior_index, behavior in enumerate(BEHAVIOR_CODES):
                        rows.append({**base, 'target': f'behavior::{behavior}', 'label': int(behaviors[item, behavior_index]), 'probability': float(behavior_probability[item, behavior_index]), 'threshold': float(thresholds.get(f'behavior::{behavior}', 0.5))})
    predictions = pd.DataFrame(rows, columns=PREDICTION_COLUMNS)
    metrics = compute_metrics_from_predictions(predictions, model_name=model_name)
    return predictions, metrics


def select_validation_champion(metrics: pd.DataFrame) -> str:
    locked_test_rows = metrics.loc[metrics['split_role'].eq('locked_test')]
    validation = metrics.loc[metrics['split_role'].eq('validation') & metrics['target'].eq(PATTERN_TARGET)]
    if validation.empty:
        raise ValueError('validation metrics are required for champion selection')
    _ = locked_test_rows
    pivot = validation.pivot_table(index='model_name', columns='metric', values='value', aggfunc='first')
    required = {'aucpr', 'event_recall', 'false_alerts_per_hour', 'ece'}
    if not required.issubset(pivot.columns):
        raise ValueError('validation tie-break metrics are incomplete')
    ordered = pivot.reset_index().sort_values(
        ['aucpr', 'event_recall', 'false_alerts_per_hour', 'ece', 'model_name'],
        ascending=[False, False, True, True, True],
        kind='mergesort',
    )
    return str(ordered.iloc[0]['model_name'])


def write_validation_model_comparison(
    ml_metrics_path: Path,
    dl_metrics_path: Path,
    output_path: Path = DL_BENCHMARK_OUTPUT_ROOT / 'model_comparison_validation.parquet',
) -> Path:
    ml_metrics = pd.read_parquet(ml_metrics_path)
    dl_metrics = pd.read_parquet(dl_metrics_path)
    for name, frame in (('ML', ml_metrics), ('DL', dl_metrics)):
        if not frame['split_role'].eq('validation').all():
            raise ValueError(f'{name} comparison file must contain validation only; locked_test is forbidden')
    comparison = ml_metrics.merge(
        dl_metrics,
        on=['target', 'metric'],
        how='inner',
        suffixes=('_ml', '_dl'),
        validate='many_to_many',
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    comparison.to_parquet(output_path, index=False)
    return output_path


## 6. 실행 오케스트레이션
기본값에서는 아무 학습도 시작하지 않습니다. 실제 실행은 Kaggle T4 x2와 torchrun 2-process 환경을 명시적으로 준비한 뒤에만 가능합니다.

In [ ]:
def _legacy_disabled_frame_collective(frame: pd.DataFrame, world_size: int) -> pd.DataFrame:
    raise RuntimeError('DataFrame collectives are disabled; use prediction shards')


def run_dl_training(epochs: int = 20, batch_size: int = 64) -> None:
    require_exactly_two_cuda_devices()
    rank, local_rank, world_size = setup_ddp()
    device = torch.device('cuda', local_rank)
    try:
        set_deterministic_seed(SEED, rank)
        verified = verify_sequence_inputs()
        wandb_enabled = rank == 0 and login_wandb_from_kaggle_secret()
        if wandb_enabled:
            import wandb

            wandb.init(project=WANDB_PROJECT, group=WANDB_GROUP, tags=WANDB_TAGS, config={'epochs': epochs, 'batch_size': batch_size, 'data_status': DATA_STATUS})
        train_dataset = Goal15SequenceDataset(
            verified.index_paths['train_600'], verified.timeline_paths['train'],
            verified.normalization, 'train',
        )
        validation_dataset = Goal15SequenceDataset(
            verified.index_paths['validation_600'], verified.timeline_paths['validation'],
            verified.normalization, 'validation',
        )
        train_sampler = DistributedSampler(train_dataset, num_replicas=world_size, rank=rank, shuffle=True, seed=SEED, drop_last=False)
        validation_sampler = DistributedSampler(validation_dataset, num_replicas=world_size, rank=rank, shuffle=False, seed=SEED, drop_last=False)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler, num_workers=2, pin_memory=True)
        validation_loader = DataLoader(validation_dataset, batch_size=batch_size, sampler=validation_sampler, num_workers=2, pin_memory=True)
        model = Goal15TCN(input_size=len(ALLOWED_FEATURE_COLUMNS)).to(device)
        parameter_count = assert_parameter_budget(model)
        model = DistributedDataParallel(model, device_ids=[local_rank], output_device=local_rank)
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
        scaler = torch.amp.GradScaler('cuda')
        for epoch in range(epochs):
            train_sampler.set_epoch(epoch)
            losses = train_one_epoch(model, train_loader, optimizer, scaler, device, epoch=epoch, sampler=train_sampler)
            if rank == 0 and wandb_enabled:
                wandb.log({f'train/{name}': value for name, value in losses.items()}, step=epoch)
        local_predictions, _ = evaluate_common_schema(
            model, validation_loader, device, split_role='validation', model_name='causal_tcn',
        )
        validation_predictions = _legacy_disabled_frame_collective(local_predictions, world_size)
        validation_predictions = validation_predictions.drop_duplicates(
            subset=['dataset_id', 'run_id', 'person_key', 'canonical_time', 'target'],
            keep='first',
        ).reset_index(drop=True)
        selected_threshold: float | None = None
        if rank == 0:
            selected_threshold = select_validation_threshold(validation_predictions)
            validation_predictions.loc[validation_predictions['target'].eq(PATTERN_TARGET), 'threshold'] = selected_threshold
            validation_metrics = compute_metrics_from_predictions(validation_predictions, model_name='causal_tcn')
            champion = select_validation_champion(validation_metrics)
            DL_BENCHMARK_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
            validation_predictions.to_parquet(DL_BENCHMARK_OUTPUT_ROOT / 'validation_predictions.parquet', index=False)
            validation_metrics.to_parquet(DL_BENCHMARK_OUTPUT_ROOT / 'validation_metrics.parquet', index=False)
            (DL_BENCHMARK_OUTPUT_ROOT / 'validation_champion.json').write_text(json.dumps({'model_name': champion, 'parameter_count': parameter_count, 'source_dataset_hash': verified.source_dataset_hash, 'split_hash': verified.split_hash}, indent=2) + '\n')
            if wandb_enabled:
                wandb.log({'validation/champion': champion, 'model/parameter_count': parameter_count})
        threshold_holder = [selected_threshold]
        dist.broadcast_object_list(threshold_holder, src=0)
        selected_threshold = float(threshold_holder[0])
        dist.barrier()
        if RUN_LOCKED_TEST:
            locked_dataset = Goal15SequenceDataset(
                verified.index_paths['locked_test_600'], verified.timeline_paths['locked_test'],
                verified.normalization, 'locked_test',
            )
            locked_sampler = DistributedSampler(locked_dataset, num_replicas=world_size, rank=rank, shuffle=False, seed=SEED, drop_last=False)
            locked_loader = DataLoader(locked_dataset, batch_size=batch_size, sampler=locked_sampler, num_workers=2, pin_memory=True)
            locked_predictions, _ = evaluate_common_schema(
                model, locked_loader, device, split_role='locked_test', model_name='causal_tcn',
                thresholds={PATTERN_TARGET: selected_threshold},
            )
            locked_predictions = _legacy_disabled_frame_collective(locked_predictions, world_size)
            locked_predictions = locked_predictions.drop_duplicates(
                subset=['dataset_id', 'run_id', 'person_key', 'canonical_time', 'target'],
                keep='first',
            ).reset_index(drop=True)
            if rank == 0:
                locked_metrics = compute_metrics_from_predictions(locked_predictions, model_name='causal_tcn')
                locked_predictions.to_parquet(DL_BENCHMARK_OUTPUT_ROOT / 'locked_test_predictions.parquet', index=False)
                locked_metrics.to_parquet(DL_BENCHMARK_OUTPUT_ROOT / 'locked_test_metrics.parquet', index=False)
        if rank == 0 and wandb_enabled:
            wandb.finish()
    finally:
        cleanup_ddp()


if RUN_TRAINING:
    print('확장형 streaming/DDP 실행 정의를 계속 로드합니다.')
else:
    print('학습 비활성화: RUN_TRAINING=False. GPU, W&B, Dataset/DataLoader를 사용하지 않았습니다.')


## 7. Attached Dataset 재발견과 물리적 lineage 재검증
선언된 hash끼리 비교하지 않고 `/kaggle/input`의 실제 manifest, split, person Parquet inventory를 다시 계산합니다.

In [ ]:
import inspect
import subprocess
import tempfile
from datetime import timedelta
from torch.utils.data import Sampler

ATTACHED_INPUT_ROOT = Path('/kaggle/input')
SEQUENCE_LENGTH_CANDIDATE = 600
THRESHOLD_GRID_SIZE = 101
COMMON_THRESHOLD_GRID_SIZE = THRESHOLD_GRID_SIZE
SEQUENCE_LENGTH_POLICY = 'fixed 600 candidate; 300 retained for future comparison'
ROW_GROUP_CACHE_MAX_GROUPS = 4
ROW_GROUP_CACHE_MAX_BYTES = 512 * 1024 * 1024
RUN_VALIDATION_COMPARISON = False
RUN_NOISE_STRESS = True
RECOMPUTE_NORMALIZATION = True
PREDICTION_SHARD_COLUMNS = [*PREDICTION_COLUMNS, 'window_id', 'audit_event_binary']


@dataclass(frozen=True)
class AttachedGoal15Inputs:
    sequence_root: Path
    timeline_root: Path
    source_root: Path


def _manifest_candidates(root: Path, name: str) -> list[Path]:
    if not root.is_dir():
        raise FileNotFoundError(f'Kaggle attached input root not found: {root}')
    return sorted(path for path in root.rglob(name) if path.is_file())


def discover_attached_goal15_inputs(root: Path = ATTACHED_INPUT_ROOT) -> AttachedGoal15Inputs:
    sequence_manifests = _manifest_candidates(root, 'sequence_manifest.json')
    timeline_manifests = _manifest_candidates(root, 'view_manifest.json')
    source_manifests = _manifest_candidates(root, 'prepared__manifest.json')
    matches: list[AttachedGoal15Inputs] = []
    for sequence_path in sequence_manifests:
        sequence = json.loads(sequence_path.read_text())
        if sequence.get('series_id') != SERIES_ID or sequence.get('data_status') != DATA_STATUS:
            continue
        for timeline_path in timeline_manifests:
            timeline = json.loads(timeline_path.read_text())
            if any(timeline.get(field) != sequence.get(field) for field in ('series_id', 'data_status', 'source_dataset_hash', 'split_hash')):
                continue
            for source_path in source_manifests:
                try:
                    physical_source, physical_split, _, _ = recompute_physical_dataset_identity(source_path.parent)
                except (FileNotFoundError, ValueError, pa.ArrowInvalid):
                    continue
                if physical_source == sequence.get('source_dataset_hash') and physical_split == sequence.get('split_hash'):
                    matches.append(AttachedGoal15Inputs(sequence_path.parent, timeline_path.parent, source_path.parent))
    unique = list(dict.fromkeys(matches))
    if len(unique) != 1:
        raise ValueError(f'exactly one hash-matched attached Goal 1.5 input is required: {len(unique)}')
    return unique[0]


def _physical_split_assignments(source_root: Path) -> tuple[dict[str, str], str]:
    split_path = source_root / 'registry__splits.parquet'
    if not split_path.is_file():
        raise FileNotFoundError('physical split registry is missing')
    parquet = pq.ParquetFile(split_path)
    assignments: dict[str, str] = {}
    for row_group in range(parquet.num_row_groups):
        frame = parquet.read_row_group(row_group, columns=['person_key', 'split_role']).to_pandas()
        if frame[['person_key', 'split_role']].isna().any().any():
            raise ValueError('physical split contains null identity')
        for row in frame.itertuples(index=False):
            if row.split_role not in EXPECTED_SPLIT_COUNTS or not isinstance(row.person_key, str) or not row.person_key.strip():
                raise ValueError('physical split contains invalid identity')
            previous = assignments.setdefault(row.person_key, row.split_role)
            if previous != row.split_role:
                raise ValueError('physical person has multiple split roles')
    counts = {role: sum(value == role for value in assignments.values()) for role in EXPECTED_SPLIT_COUNTS}
    if counts != EXPECTED_SPLIT_COUNTS:
        raise ValueError(f'physical split count mismatch: {counts}')
    return assignments, sha256_file(split_path)


def recompute_physical_dataset_identity(
    source_root: Path,
) -> tuple[str, str, list[dict[str, Any]], str]:
    manifest_names = ('prepared__manifest.json', 'outcomes__manifest.json', 'registry__manifest.json')
    manifests: dict[str, dict[str, Any]] = {}
    manifest_hashes: dict[str, str] = {}
    for name in manifest_names:
        path = source_root / name
        if not path.is_file():
            raise FileNotFoundError(f'physical source manifest missing: {name}')
        payload = path.read_bytes()
        manifests[name] = json.loads(payload)
        manifest_hashes[name] = hashlib.sha256(payload).hexdigest()
    assignments, split_hash = _physical_split_assignments(source_root)
    people = manifests['prepared__manifest.json'].get('people')
    if not isinstance(people, list) or len(people) != sum(EXPECTED_SPLIT_COUNTS.values()):
        raise ValueError('physical prepared manifest must declare exactly 36 people')
    inventory: list[dict[str, Any]] = []
    seen_people: set[str] = set()
    seen_datasets: set[str] = set()
    for entry in sorted(people, key=lambda item: item.get('dataset_id', '')):
        required = ('dataset_id', 'person_key', 'run_id', 'person_id')
        if not isinstance(entry, dict) or any(not isinstance(entry.get(field), str) or not entry[field].strip() for field in required):
            raise ValueError('physical prepared inventory identity is invalid')
        dataset_id, person_key = entry['dataset_id'], entry['person_key']
        if dataset_id in seen_datasets or person_key in seen_people or person_key not in assignments:
            raise ValueError('physical prepared inventory identity is duplicated or unsplit')
        seen_datasets.add(dataset_id)
        seen_people.add(person_key)
        relative_path = f'prepared__people__{dataset_id}.parquet'
        path = source_root / relative_path
        if not path.is_file() or path.resolve().parent != source_root.resolve():
            raise ValueError('physical prepared source path is missing or escapes root')
        parquet = pq.ParquetFile(path)
        identity_values: set[tuple[str, str, str]] = set()
        for row_group in range(parquet.num_row_groups):
            frame = parquet.read_row_group(row_group, columns=['person_key', 'run_id', 'person_id']).to_pandas()
            identity_values.update(tuple(row) for row in frame.itertuples(index=False, name=None))
        expected_identity = {(person_key, entry['run_id'], entry['person_id'])}
        if identity_values != expected_identity:
            raise ValueError(f'physical person identity mismatch: {dataset_id}')
        schema_hash = hashlib.sha256(parquet.schema_arrow.remove_metadata().serialize().to_pybytes()).hexdigest()
        inventory.append({
            'dataset_id': dataset_id, 'person_key': person_key,
            'run_id': entry['run_id'], 'person_id': entry['person_id'],
            'split_role': assignments[person_key], 'path': relative_path,
            'sha256': sha256_file(path), 'row_count': parquet.metadata.num_rows,
            'schema_fingerprint': schema_hash,
        })
    if seen_people != set(assignments):
        raise ValueError('physical prepared inventory does not cover exact split membership')
    inventory_payload = json.dumps(inventory, sort_keys=True, separators=(',', ':'))
    inventory_hash = hashlib.sha256(inventory_payload.encode()).hexdigest()
    envelope = {
        'manifest_hashes': dict(sorted(manifest_hashes.items())),
        'prepared_source_inventory_hash': inventory_hash,
        'split_sha256': split_hash,
    }
    source_hash = hashlib.sha256(json.dumps(envelope, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    return source_hash, split_hash, inventory, inventory_hash


def verify_train_normalization_statistics(
    normalization: Mapping[str, Any],
    train_timeline_path: Path,
    *,
    recompute: bool = RECOMPUTE_NORMALIZATION,
) -> None:
    if normalization.get('fit_split_role') != 'train':
        raise ValueError('normalization support is not train-only')
    if not recompute:
        return
    parquet = pq.ParquetFile(train_timeline_path)
    with tempfile.TemporaryDirectory(prefix='goal15-normalization-audit-') as temporary:
        for feature in ALLOWED_FEATURE_COLUMNS:
            count = parquet.metadata.num_rows
            path = Path(temporary) / f'{hashlib.sha256(feature.encode()).hexdigest()}.mmap'
            values = np.memmap(path, mode='w+', dtype=np.float64, shape=(count,))
            offset = 0
            for row_group in range(parquet.num_row_groups):
                batch = parquet.read_row_group(row_group, columns=[feature]).column(feature).to_numpy(zero_copy_only=False)
                values[offset:offset + len(batch)] = np.asarray(batch, dtype=np.float64)
                offset += len(batch)
            median = float(np.median(values))
            lower, upper = np.quantile(values, [0.25, 0.75])
            iqr = float(upper - lower) if upper > lower else 1.0
            declared = normalization['features'][feature]
            if not np.isclose(median, declared['median']) or not np.isclose(iqr, declared['iqr']):
                raise ValueError(f'train normalization recomputation mismatch: {feature}')
            del values
            path.unlink(missing_ok=True)


_verify_declared_sequence_inputs = verify_sequence_inputs


def verify_sequence_inputs(
    sequence_root: Path | None = None,
    timeline_root: Path | None = None,
    source_root: Path | None = None,
) -> VerifiedSequenceInputs:
    if sequence_root is None or timeline_root is None or source_root is None:
        attached = discover_attached_goal15_inputs()
        sequence_root = sequence_root or attached.sequence_root
        timeline_root = timeline_root or attached.timeline_root
        source_root = source_root or attached.source_root
    declared = _verify_declared_sequence_inputs(sequence_root, timeline_root)
    physical_source, physical_split, physical_inventory, physical_inventory_hash = recompute_physical_dataset_identity(source_root)
    verify_declared_physical_inventory(source_root)
    if declared.source_dataset_hash != physical_source or declared.split_hash != physical_split:
        raise ValueError('declared Task4 identity does not match physical Dataset')
    timeline_manifest = json.loads((timeline_root / 'view_manifest.json').read_text())
    if timeline_manifest.get('source_content_inventory') != physical_inventory:
        raise ValueError('source_content_inventory does not match physical Dataset')
    if timeline_manifest.get('source_content_inventory_hash') != physical_inventory_hash:
        raise ValueError('source content inventory hash mismatch')
    physical_counts = {
        role: sum(item['split_role'] == role for item in physical_inventory)
        for role in EXPECTED_SPLIT_COUNTS
    }
    if physical_counts != EXPECTED_SPLIT_COUNTS:
        raise ValueError(f'physical inventory membership mismatch: {physical_counts}')
    verify_train_normalization_statistics(declared.normalization, declared.timeline_paths['train'])
    verify_exact_sequence_person_sets(source_root, declared.timeline_paths, declared.index_paths)
    return VerifiedSequenceInputs(
        declared.source_dataset_hash, declared.split_hash, declared.index_paths,
        declared.timeline_paths, declared.normalization, source_root,
    )


## 8. Fail-fast DDP와 train-only imbalance support
조건부 손실은 rank별 유효 행 수가 달라도 DDP 평균 gradient가 전역 유효 행 평균과 같도록 보정합니다.

In [ ]:
def setup_ddp() -> tuple[int, int, int]:
    required = ('LOCAL_RANK', 'RANK', 'WORLD_SIZE')
    missing = [name for name in required if name not in os.environ]
    if missing:
        raise RuntimeError(f'torchrun 환경이 필요합니다. 누락: {missing}')
    os.environ.setdefault('TORCH_NCCL_ASYNC_ERROR_HANDLING', '1')
    os.environ.setdefault('NCCL_ASYNC_ERROR_HANDLING', '1')
    local_rank = int(os.environ['LOCAL_RANK'])
    rank = int(os.environ['RANK'])
    world_size = int(os.environ['WORLD_SIZE'])
    if world_size != 2 or local_rank not in (0, 1):
        raise RuntimeError('T4 x2는 torchrun --nproc_per_node=2로 실행해야 합니다.')
    torch.cuda.set_device(local_rank)
    dist.init_process_group(
        backend='nccl', init_method='env://', rank=rank, world_size=world_size,
        timeout=timedelta(minutes=10),
    )
    return rank, local_rank, world_size


def cleanup_ddp() -> None:
    if dist.is_available() and dist.is_initialized():
        dist.destroy_process_group()


def safe_wandb_call(operation: Any, *args: Any, **kwargs: Any) -> Any | None:
    try:
        return operation(*args, **kwargs)
    except Exception as exc:
        print(f'선택 telemetry 실패: {type(exc).__name__}')
        return None


def derive_train_loss_weights(train_index_path: Path) -> dict[str, Any]:
    parquet = pq.ParquetFile(train_index_path)
    event_positive = event_total = 0
    stage_counts = {stage: 0 for stage in STAGE_CODES}
    behavior_positive = {code: 0 for code in BEHAVIOR_CODES}
    behavior_decision_count = 0
    columns = [PATTERN_TARGET, 'hard_negative', STAGE_TARGET, *BEHAVIOR_CODES]
    for row_group in range(parquet.num_row_groups):
        frame = parquet.read_row_group(row_group, columns=columns).to_pandas()
        event_positive += int(frame[PATTERN_TARGET].sum())
        event_total += len(frame)
        pattern_rows = frame[PATTERN_TARGET].eq(1)
        for stage in STAGE_CODES:
            stage_counts[stage] += int((pattern_rows & frame[STAGE_TARGET].eq(stage)).sum())
        positive_behavior = frame.loc[:, list(BEHAVIOR_CODES)].eq(1).any(axis=1)
        decision = pattern_rows | (frame['hard_negative'].eq(1) & positive_behavior)
        behavior_decision_count += int(decision.sum())
        for code in BEHAVIOR_CODES:
            behavior_positive[code] += int(frame.loc[decision, code].sum())
    if event_total < 1 or event_positive < 1 or event_positive >= event_total:
        raise ValueError('train pattern support requires both classes')
    stage_total = sum(stage_counts.values())
    if any(count < 1 for count in stage_counts.values()):
        raise ValueError('train stage support is incomplete')
    if behavior_decision_count < 1:
        raise ValueError('train behavior decision support is empty')
    return {
        'fit_split_role': 'train',
        'event_support': {'positive': event_positive, 'negative': event_total - event_positive},
        'event_pos_weight': (event_total - event_positive) / event_positive,
        'stage_support': stage_counts,
        'stage_class_weight': {stage: stage_total / (len(STAGE_CODES) * count) for stage, count in stage_counts.items()},
        'behavior_support': {
            code: {'positive': behavior_positive[code], 'negative': behavior_decision_count - behavior_positive[code]}
            for code in BEHAVIOR_CODES
        },
        'behavior_pos_weight': {
            code: (behavior_decision_count - count) / count if count else 1.0
            for code, count in behavior_positive.items()
        },
    }


def write_train_loss_support(output_root: Path, support: Mapping[str, Any]) -> tuple[Path, str]:
    if support.get('fit_split_role') != 'train':
        raise ValueError('loss support must be train-only')
    output_root.mkdir(parents=True, exist_ok=True)
    path = output_root / 'train_loss_support.json'
    path.write_text(json.dumps(support, indent=2, sort_keys=True) + '\n')
    return path, sha256_file(path)


def global_valid_count(local_count: torch.Tensor) -> torch.Tensor:
    count = local_count.detach().to(dtype=torch.float64).clone()
    if dist.is_available() and dist.is_initialized():
        dist.all_reduce(count, op=dist.ReduceOp.SUM)
    return count


def _ddp_global_mean(
    local_sum: torch.Tensor, local_count: torch.Tensor, world_size: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    global_count = global_valid_count(local_count)
    if global_count.item() <= 0:
        return local_sum * 0.0, global_count
    return local_sum * world_size / global_count.to(local_sum.dtype), global_count


def masked_multitask_loss(
    outputs: Mapping[str, torch.Tensor],
    batch: Mapping[str, torch.Tensor],
    *,
    loss_weights: Mapping[str, Any],
    stage_weight: float = 1.0,
    behavior_weight: float = 1.0,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    pattern = batch[PATTERN_TARGET].float()
    world_size = dist.get_world_size() if dist.is_available() and dist.is_initialized() else 1
    event_pos_weight = torch.tensor(loss_weights['event_pos_weight'], device=pattern.device, dtype=pattern.dtype)
    event_elements = F.binary_cross_entropy_with_logits(outputs['event_logits'], pattern, pos_weight=event_pos_weight, reduction='none')
    event_bce, global_event_count = _ddp_global_mean(event_elements.sum(), torch.tensor(event_elements.numel(), device=pattern.device), world_size)
    stage_mask = pattern.eq(1)
    stage_class_weight = torch.tensor([loss_weights['stage_class_weight'][stage] for stage in STAGE_CODES], device=pattern.device, dtype=outputs['stage_logits'].dtype)
    stage_elements = F.cross_entropy(outputs['stage_logits'][stage_mask], batch[STAGE_TARGET][stage_mask], weight=stage_class_weight, reduction='none')
    stage_loss, global_stage_count = _ddp_global_mean(stage_elements.sum(), stage_mask.sum(), world_size)
    behavior_labels = batch['behaviors'].float()
    behavior_positive = behavior_labels.eq(1).any(dim=1)
    hard_negative = batch['hard_negative'].eq(1)
    invalid_positive = behavior_positive & ~(stage_mask | hard_negative)
    if invalid_positive.any():
        raise ValueError('behavior-positive rows must be pattern or hard negative')
    behavior_mask = stage_mask | (hard_negative & behavior_positive)
    behavior_pos_weight = torch.tensor([loss_weights['behavior_pos_weight'][code] for code in BEHAVIOR_CODES], device=pattern.device, dtype=outputs['behavior_logits'].dtype)
    behavior_elements = F.binary_cross_entropy_with_logits(outputs['behavior_logits'][behavior_mask], behavior_labels[behavior_mask], pos_weight=behavior_pos_weight, reduction='none')
    behavior_local_count = behavior_mask.sum() * len(BEHAVIOR_CODES)
    behavior_loss, global_behavior_count = _ddp_global_mean(behavior_elements.sum(), behavior_local_count, world_size)
    total = event_bce + stage_weight * stage_loss + behavior_weight * behavior_loss
    return total, {
        'event_bce': event_bce.detach(), 'stage_loss': stage_loss.detach(),
        'behavior_loss': behavior_loss.detach(), 'global_event_count': global_event_count,
        'global_stage_count': global_stage_count, 'global_behavior_count': global_behavior_count,
        'event_local_sum': event_elements.detach().sum(),
        'stage_local_sum': stage_elements.detach().sum(),
        'behavior_local_sum': behavior_elements.detach().sum(),
        'event_local_count': torch.tensor(event_elements.numel(), device=pattern.device),
        'stage_local_count': stage_mask.detach().sum(),
        'behavior_local_count': behavior_local_count.detach(),
    }


def reduce_training_statistics(
    totals: Mapping[str, float], counts: Mapping[str, float], device: torch.device,
) -> dict[str, float]:
    names = sorted(totals)
    values = torch.tensor(
        [totals[name] for name in names] + [counts[name] for name in names],
        device=device, dtype=torch.float64,
    )
    if dist.is_available() and dist.is_initialized():
        dist.all_reduce(values, op=dist.ReduceOp.SUM)
    count_offset = len(names)
    means = {
        name: float(values[index].item()) / max(float(values[count_offset + index].item()), 1.0)
        for index, name in enumerate(names)
    }
    means['total'] = sum(means.values())
    return means


def train_one_epoch(
    model: DistributedDataParallel, loader: DataLoader,
    optimizer: torch.optim.Optimizer, scaler: torch.amp.GradScaler,
    device: torch.device, *, epoch: int, sampler: DistributedSampler,
    loss_weights: Mapping[str, Any],
) -> dict[str, float]:
    sampler.set_epoch(epoch)
    model.train()
    running = {'event_bce': 0.0, 'stage_loss': 0.0, 'behavior_loss': 0.0}
    valid_counts = {'event_bce': 0.0, 'stage_loss': 0.0, 'behavior_loss': 0.0}
    for raw_batch in loader:
        batch = _move_batch(raw_batch, device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(batch['features'], batch['mask'])
            loss, components = masked_multitask_loss(outputs, batch, loss_weights=loss_weights)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        for name in ('event_bce', 'stage_loss', 'behavior_loss'):
            prefix = name.removesuffix('_bce').removesuffix('_loss')
            running[name] += float(components[f'{prefix}_local_sum'])
            valid_counts[name] += float(components[f'{prefix}_local_count'])
    return reduce_training_statistics(running, valid_counts, device)


## 9. Person-sharded streaming 평가와 완전한 validation 지표
예측은 rank별 Parquet shard로 기록하고 rank 0은 person 단위로 두 번 순회합니다. 전체 DataFrame은 어느 rank에도 복제하지 않습니다.

In [ ]:
class PersonShardEvalSampler(Sampler[int]):
    def __init__(self, dataset: Goal15SequenceDataset, *, rank: int, num_replicas: int) -> None:
        if num_replicas < 1 or rank < 0 or rank >= num_replicas:
            raise ValueError('invalid evaluation shard rank')
        self.ranges: list[range] = []
        for person in sorted(dataset.person_ranges()):
            shard = int(hashlib.sha256(person.encode()).hexdigest(), 16) % num_replicas
            if shard != rank:
                continue
            self.ranges.extend(dataset.person_ranges()[person])

    def __iter__(self):
        return (index for person_range in self.ranges for index in person_range)

    def __len__(self) -> int:
        return sum(len(person_range) for person_range in self.ranges)


def prediction_shard_arrow_schema() -> pa.Schema:
    return pa.schema([
        pa.field('model_family', pa.string(), nullable=False),
        pa.field('model_name', pa.string(), nullable=False),
        pa.field('series_id', pa.string(), nullable=False),
        pa.field('dataset_id', pa.string(), nullable=False),
        pa.field('run_id', pa.string(), nullable=False),
        pa.field('person_key', pa.string(), nullable=False),
        pa.field('canonical_time', pa.string(), nullable=False),
        pa.field('split_role', pa.string(), nullable=False),
        pa.field('target', pa.string(), nullable=False),
        pa.field('label', pa.int8(), nullable=False),
        pa.field('probability', pa.float64(), nullable=False),
        pa.field('threshold', pa.float64(), nullable=False),
        pa.field('window_id', pa.string(), nullable=False),
        pa.field('audit_event_binary', pa.int8(), nullable=False),
    ])


def _bounded_prediction_rows(
    raw_batch: Mapping[str, Any], outputs: Mapping[str, torch.Tensor],
    split_role: str, model_name: str, thresholds: Mapping[str, float],
) -> list[dict[str, Any]]:
    pattern_probability = torch.sigmoid(outputs['event_logits']).float().cpu().numpy()
    stage_probability = torch.softmax(outputs['stage_logits'], dim=1).float().cpu().numpy()
    behavior_probability = torch.sigmoid(outputs['behavior_logits']).float().cpu().numpy()
    pattern = raw_batch[PATTERN_TARGET].cpu().numpy().astype(np.int8)
    onset_event = raw_batch[ONSET_EVENT_TARGET].cpu().numpy().astype(np.int8)
    hard_negative = raw_batch['hard_negative'].cpu().numpy().astype(np.int8)
    behaviors = raw_batch['behaviors'].cpu().numpy().astype(np.int8)
    stages = raw_batch[STAGE_TARGET].cpu().numpy().astype(np.int64)
    behavior_mask = (pattern == 1) | ((hard_negative == 1) & (behaviors == 1).any(axis=1))
    rows: list[dict[str, Any]] = []
    for item in range(len(pattern)):
        base = {
            'model_family': 'deep_learning_tcn', 'model_name': model_name,
            'series_id': SERIES_ID, 'dataset_id': raw_batch['dataset_id'][item],
            'run_id': raw_batch['run_id'][item], 'person_key': raw_batch['person_key'][item],
            'canonical_time': raw_batch['canonical_time'][item], 'split_role': split_role,
            'window_id': raw_batch['window_id'][item], 'audit_event_binary': int(onset_event[item]),
        }
        rows.append({**base, 'target': PATTERN_TARGET, 'label': int(pattern[item]), 'probability': float(pattern_probability[item]), 'threshold': float(thresholds.get(PATTERN_TARGET, 0.5))})
        if pattern[item] == 1:
            for stage_index, stage in enumerate(STAGE_CODES):
                rows.append({**base, 'target': f'stage::{stage}', 'label': int(stages[item] == stage_index), 'probability': float(stage_probability[item, stage_index]), 'threshold': float(thresholds.get(f'stage::{stage}', 0.5))})
        if behavior_mask[item]:
            for behavior_index, behavior in enumerate(BEHAVIOR_CODES):
                rows.append({**base, 'target': f'behavior::{behavior}', 'label': int(behaviors[item, behavior_index]), 'probability': float(behavior_probability[item, behavior_index]), 'threshold': float(thresholds.get(f'behavior::{behavior}', 0.5))})
    return rows


def write_prediction_shard_manifest(
    shard_path: Path, *, split_role: str, rank: int, window_count: int,
    row_count: int, source_dataset_hash: str, split_hash: str,
) -> Path:
    manifest_path = shard_path.with_suffix('.manifest.json')
    manifest_path.write_text(json.dumps({
        'schema_version': 'goal1.5/dl-prediction-shard/v1',
        'series_id': SERIES_ID, 'data_status': DATA_STATUS,
        'split_role': split_role, 'rank': rank, 'path': shard_path.name,
        'sha256': sha256_file(shard_path), 'row_count': row_count,
        'window_count': window_count, 'source_dataset_hash': source_dataset_hash,
        'split_hash': split_hash, 'columns': PREDICTION_SHARD_COLUMNS,
    }, indent=2, sort_keys=True) + '\n')
    return manifest_path


def evaluate_to_prediction_shard(
    model: nn.Module, loader: DataLoader, device: torch.device,
    *, split_role: str, model_name: str, rank: int, output_path: Path,
    source_dataset_hash: str, split_hash: str,
    thresholds: Mapping[str, float] | None = None,
    stress_condition: str | None = None,
) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    schema = prediction_shard_arrow_schema()
    writer = pq.ParquetWriter(output_path, schema, compression='zstd')
    row_count = window_count = 0
    model.eval()
    try:
        with torch.no_grad():
            for raw_batch in loader:
                if stress_condition is not None:
                    raw_batch = deterministic_stress_batch(raw_batch, stress_condition)
                batch = _move_batch(raw_batch, device)
                with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                    outputs = model(batch['features'], batch['mask'])
                rows = _bounded_prediction_rows(raw_batch, outputs, split_role, model_name, dict(thresholds or {}))
                table = pa.Table.from_pylist(rows, schema=schema) if rows else pa.Table.from_batches([], schema=schema)
                writer.write_table(table)
                row_count += len(rows)
                window_count += len(raw_batch['window_id'])
    finally:
        writer.close()
    return write_prediction_shard_manifest(
        output_path, split_role=split_role, rank=rank, window_count=window_count,
        row_count=row_count, source_dataset_hash=source_dataset_hash, split_hash=split_hash,
    )


class StreamingMetricAccumulator:
    '''Bounded global_aucpr/global_auroc/global_row_f1/global_ece owner.'''
    def __init__(self, *, bins: int = THRESHOLD_GRID_SIZE) -> None:
        self.bins = bins
        self.histograms: dict[str, dict[str, np.ndarray]] = {}
        self.person_summaries: list[dict[str, float | str]] = []

    def update_histograms(self, frame: pd.DataFrame) -> None:
        for target, rows in frame.groupby('target', sort=False):
            bucket = np.minimum((rows['probability'].to_numpy() * (self.bins - 1)).astype(np.int64), self.bins - 1)
            state = self.histograms.setdefault(target, {'positive': np.zeros(self.bins), 'negative': np.zeros(self.bins)})
            labels = rows['label'].to_numpy(dtype=np.int8)
            np.add.at(state['positive'], bucket[labels == 1], 1)
            np.add.at(state['negative'], bucket[labels == 0], 1)

    def bounded_threshold(self, target: str = PATTERN_TARGET) -> float:
        state = self.histograms[target]
        true_positive = np.cumsum(state['positive'][::-1])
        false_positive = np.cumsum(state['negative'][::-1])
        false_negative = state['positive'].sum() - true_positive
        denominator = 2 * true_positive + false_positive + false_negative
        f1 = np.divide(2 * true_positive, denominator, out=np.zeros_like(denominator), where=denominator > 0)
        reverse_index = int(np.flatnonzero(f1 == f1.max())[-1])
        return float((self.bins - 1 - reverse_index) / (self.bins - 1))


def select_validation_threshold(predictions: pd.DataFrame | StreamingMetricAccumulator) -> float:
    if isinstance(predictions, StreamingMetricAccumulator):
        return predictions.bounded_threshold(PATTERN_TARGET)
    pattern = predictions.loc[predictions['split_role'].eq('validation') & predictions['target'].eq(PATTERN_TARGET)]
    if pattern.empty:
        raise ValueError('validation pattern rows are required')
    grid = np.linspace(0.0, 1.0, THRESHOLD_GRID_SIZE)
    labels = pattern['label'].to_numpy(dtype=np.int8)
    probability = pattern['probability'].to_numpy(dtype=np.float64)
    scores = [f1_score(labels, probability >= threshold, zero_division=0) for threshold in grid]
    return float(grid[int(np.flatnonzero(scores == np.max(scores))[-1])])


def _iter_shard_people(shard_path: Path):
    parquet = pq.ParquetFile(shard_path)
    carry = pd.DataFrame()
    for row_group in range(parquet.num_row_groups):
        frame = parquet.read_row_group(row_group).to_pandas()
        if not carry.empty:
            frame = pd.concat([carry, frame], ignore_index=True)
            carry = pd.DataFrame()
        people = list(frame['person_key'].drop_duplicates())
        for person in people[:-1]:
            yield frame.loc[frame['person_key'].eq(person)].copy()
        if people:
            carry = frame.loc[frame['person_key'].eq(people[-1])].copy()
    if not carry.empty:
        yield carry


def bootstrap_people_ci(
    person_values: Sequence[float], *, iterations: int = 1000, seed: int = SEED,
) -> dict[str, float]:
    values = np.asarray(person_values, dtype=np.float64)
    if not len(values):
        return {'estimate': np.nan, 'lower': np.nan, 'upper': np.nan}
    generator = np.random.default_rng(seed)
    estimates = np.asarray([generator.choice(values, size=len(values), replace=True).mean() for _ in range(iterations)])
    return {'estimate': float(values.mean()), 'lower': float(np.quantile(estimates, 0.025)), 'upper': float(np.quantile(estimates, 0.975))}


def compute_forecast_lead_times(pattern_rows: pd.DataFrame) -> list[float]:
    ordered = pattern_rows.sort_values('canonical_time').reset_index(drop=True)
    labels = ordered['audit_event_binary'].to_numpy(dtype=np.int8)
    alerts = ordered['probability'].ge(ordered['threshold']).to_numpy()
    times = pd.to_datetime(ordered['canonical_time'], utc=True)
    onset_positions = np.flatnonzero((labels == 1) & (np.r_[0, labels[:-1]] == 0))
    lead_times: list[float] = []
    for onset in onset_positions:
        candidates = np.flatnonzero(alerts[:onset] & ordered['label'].to_numpy(dtype=bool)[:onset])
        if len(candidates):
            lead_times.append(float((times.iloc[onset] - times.iloc[candidates[-1]]).total_seconds()))
    return lead_times


def compute_stage_confusion_rows(person_rows: pd.DataFrame) -> pd.DataFrame:
    rows = person_rows.loc[person_rows['target'].str.startswith('stage::')]
    if rows.empty:
        return pd.DataFrame(columns=['actual', 'predicted', 'count'])
    keys = ['window_id', 'person_key', 'canonical_time']
    probability = rows.pivot(index=keys, columns='target', values='probability')
    actual = rows.loc[rows['label'].eq(1)].set_index(keys)['target'].loc[probability.index]
    predicted = probability.idxmax(axis=1)
    return pd.DataFrame({'actual': actual.to_numpy(), 'predicted': predicted.to_numpy()}).value_counts().rename('count').reset_index()


def _validate_person_prediction_rows(frame: pd.DataFrame) -> None:
    if frame['person_key'].nunique() != 1 or frame['split_role'].nunique() != 1:
        raise ValueError('prediction stream must yield one person/role at a time')
    pattern = frame.loc[frame['target'].eq(PATTERN_TARGET)].sort_values('canonical_time')
    if pattern['window_id'].duplicated().any() or not pd.to_datetime(pattern['canonical_time'], utc=True).is_monotonic_increasing:
        raise ValueError('prediction window coverage is duplicate or unordered')


def aggregate_prediction_shards(
    manifest_paths: Sequence[Path], *, expected_window_count: int,
    source_dataset_hash: str, split_hash: str, model_name: str,
) -> tuple[pd.DataFrame, float, pd.DataFrame]:
    shard_paths: list[Path] = []
    declared_windows = 0
    for manifest_path in sorted(manifest_paths):
        manifest = json.loads(manifest_path.read_text())
        shard_path = manifest_path.parent / manifest['path']
        if manifest.get('source_dataset_hash') != source_dataset_hash or manifest.get('split_hash') != split_hash:
            raise ValueError('prediction shard identity mismatch')
        if manifest.get('columns') != PREDICTION_SHARD_COLUMNS or sha256_file(shard_path) != manifest.get('sha256'):
            raise ValueError('prediction shard schema/hash mismatch')
        if pq.ParquetFile(shard_path).metadata.num_rows != manifest.get('row_count'):
            raise ValueError('prediction shard row count mismatch')
        declared_windows += int(manifest['window_count'])
        shard_paths.append(shard_path)
    if declared_windows != expected_window_count:
        raise ValueError('prediction shard window coverage mismatch')
    accumulator = StreamingMetricAccumulator()
    seen_people: set[str] = set()
    for shard_path in shard_paths:
        for person in _iter_shard_people(shard_path):
            _validate_person_prediction_rows(person)
            person_key = str(person['person_key'].iloc[0])
            if person_key in seen_people:
                raise ValueError('person prediction appears in multiple shards')
            seen_people.add(person_key)
            accumulator.update_histograms(person)
    threshold = select_validation_threshold(accumulator)
    metric_frames: list[pd.DataFrame] = []
    confusion_frames: list[pd.DataFrame] = []
    person_f1: list[float] = []
    lead_times: list[float] = []
    for shard_path in shard_paths:
        for person in _iter_shard_people(shard_path):
            person.loc[person['target'].eq(PATTERN_TARGET), 'threshold'] = threshold
            metric_frames.append(compute_metrics_from_predictions(person.loc[:, PREDICTION_COLUMNS], model_name=model_name))
            confusion_frames.append(compute_stage_confusion_rows(person))
            pattern = person.loc[person['target'].eq(PATTERN_TARGET)]
            person_f1.append(float(f1_score(pattern['label'], pattern['probability'].ge(threshold), zero_division=0)))
            lead_times.extend(compute_forecast_lead_times(pattern))
    metrics = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame(columns=METRIC_COLUMNS)
    summary = metrics.groupby(['model_family', 'model_name', 'series_id', 'split_role', 'target', 'metric', 'data_status'], as_index=False).agg(value=('value', 'mean'), support=('support', 'sum'))
    ci = bootstrap_people_ci(person_f1)
    for metric, value in (('person_bootstrap_f1_lower', ci['lower']), ('person_bootstrap_f1_upper', ci['upper']), ('forecast_lead_mean', float(np.mean(lead_times)) if lead_times else np.nan), ('forecast_lead_median', float(np.median(lead_times)) if lead_times else np.nan)):
        summary.loc[len(summary)] = ['deep_learning_tcn', model_name, SERIES_ID, summary['split_role'].iloc[0], PATTERN_TARGET, metric, value, len(person_f1), DATA_STATUS]
    confusion = pd.concat(confusion_frames, ignore_index=True).groupby(['actual', 'predicted'], as_index=False)['count'].sum() if confusion_frames else pd.DataFrame(columns=['actual', 'predicted', 'count'])
    return summary.loc[:, METRIC_COLUMNS], threshold, confusion


## 10. 상세 지표, deterministic stress, identity-safe 비교
Stage confusion/per-stage recall, behavior support/AUROC/F1, lead time, person bootstrap과 Goal 1.5 stress degradation을 validation에만 기록합니다.

In [ ]:
_base_compute_metrics_from_predictions = compute_metrics_from_predictions


def compute_metrics_from_predictions(
    predictions: pd.DataFrame, *, model_name: str,
) -> pd.DataFrame:
    base = _base_compute_metrics_from_predictions(predictions, model_name=model_name)
    split_role = str(predictions['split_role'].iloc[0])
    additions: list[dict[str, Any]] = []
    for target, rows in predictions.groupby('target', sort=True):
        additions.append(_aggregate_metric_row(model_name, split_role, target, 'positive_support', float(rows['label'].sum()), len(rows)))
    stage_rows = predictions.loc[predictions['target'].str.startswith('stage::')]
    confusion_matrix = compute_stage_confusion_rows(predictions)
    if not stage_rows.empty and not confusion_matrix.empty:
        for stage in (f'stage::{code}' for code in STAGE_CODES):
            relevant = confusion_matrix.loc[confusion_matrix['actual'].eq(stage)]
            support = int(relevant['count'].sum())
            correct = int(relevant.loc[relevant['predicted'].eq(stage), 'count'].sum())
            per_stage_recall = correct / support if support else np.nan
            additions.append(_aggregate_metric_row(model_name, split_role, stage, 'per_stage_recall', per_stage_recall, support))
        additions.append(_aggregate_metric_row(model_name, split_role, 'stage::all', 'confusion_matrix_cells', float(len(confusion_matrix)), int(confusion_matrix['count'].sum())))
    behavior_rows = predictions.loc[predictions['target'].str.startswith('behavior::')]
    if not behavior_rows.empty:
        labels = behavior_rows['label'].to_numpy(dtype=np.int8)
        probability = behavior_rows['probability'].to_numpy(dtype=np.float64)
        predicted = probability >= behavior_rows['threshold'].to_numpy(dtype=np.float64)
        behavior_micro_auroc = float(roc_auc_score(labels, probability)) if len(np.unique(labels)) == 2 else np.nan
        behavior_micro_f1 = float(f1_score(labels, predicted, zero_division=0))
        additions.extend([
            _aggregate_metric_row(model_name, split_role, 'behavior::all', 'behavior_micro_auroc', behavior_micro_auroc, len(labels)),
            _aggregate_metric_row(model_name, split_role, 'behavior::all', 'behavior_micro_f1', behavior_micro_f1, len(labels)),
        ])
    pattern = predictions.loc[predictions['target'].eq(PATTERN_TARGET)].copy()
    person_values = [
        f1_score(person['label'], person['probability'].ge(person['threshold']), zero_division=0)
        for _, person in pattern.groupby('person_key', sort=False)
    ]
    person_ci = bootstrap_people_ci(person_values)
    for metric, value in (('person_bootstrap_f1_lower', person_ci['lower']), ('person_bootstrap_f1_upper', person_ci['upper'])):
        additions.append(_aggregate_metric_row(model_name, split_role, PATTERN_TARGET, metric, value, len(person_values)))
    if 'audit_event_binary' in pattern:
        forecast_lead = compute_forecast_lead_times(pattern)
        additions.extend([
            _aggregate_metric_row(model_name, split_role, PATTERN_TARGET, 'forecast_lead_mean', float(np.mean(forecast_lead)) if forecast_lead else np.nan, len(forecast_lead)),
            _aggregate_metric_row(model_name, split_role, PATTERN_TARGET, 'forecast_lead_median', float(np.median(forecast_lead)) if forecast_lead else np.nan, len(forecast_lead)),
        ])
    return pd.concat([base, pd.DataFrame(additions, columns=METRIC_COLUMNS)], ignore_index=True)


STRESS_CONDITIONS = (
    'gaussian_0.05', 'gaussian_0.10', 'gaussian_0.20',
    'block_missing_5', 'block_missing_30', 'block_missing_120',
    'time_shift_-5', 'time_shift_-1', 'time_shift_1', 'time_shift_5',
    'latent_axis_dropout',
)


def deterministic_stress_batch(batch: Mapping[str, Any], condition: str) -> dict[str, Any]:
    if condition not in STRESS_CONDITIONS:
        raise ValueError(f'unknown Goal 1.5 stress condition: {condition}')
    stressed = dict(batch)
    features = batch['features'].clone()
    mask = batch['mask'].clone()
    for item, window_id in enumerate(batch['window_id']):
        seed = int(hashlib.sha256(f'{window_id}|{condition}'.encode()).hexdigest()[:16], 16)
        generator = torch.Generator(device=features.device).manual_seed(seed)
        if condition.startswith('gaussian_'):
            scale = float(condition.rsplit('_', 1)[1])
            features[item] += torch.randn(features[item].shape, generator=generator, device=features.device, dtype=features.dtype) * scale
        elif condition.startswith('block_missing_'):
            length = int(condition.rsplit('_', 1)[1])
            start = seed % max(int(mask[item].sum()) - length + 1, 1)
            features[item, start:start + length] = 0
            mask[item, start:start + length] = False
        elif condition.startswith('time_shift_'):
            shift = int(condition.removeprefix('time_shift_'))
            features[item] = torch.roll(features[item], shifts=shift, dims=0)
        else:
            axis = seed % features.shape[-1]
            features[item, :, axis] = 0
    stressed['features'] = features
    stressed['mask'] = mask
    return stressed


def evaluate_noise_stress_to_shards(
    model: nn.Module, loader: DataLoader, device: torch.device,
    *, split_role: str, model_name: str, rank: int, output_root: Path,
    source_dataset_hash: str, split_hash: str, threshold: float,
) -> dict[str, Path]:
    manifests: dict[str, Path] = {}
    for condition in STRESS_CONDITIONS:
        path = output_root / condition / f'rank-{rank}.parquet'
        manifests[condition] = evaluate_to_prediction_shard(
            model, loader, device, split_role=split_role, model_name=model_name,
            rank=rank, output_path=path, source_dataset_hash=source_dataset_hash,
            split_hash=split_hash, thresholds={PATTERN_TARGET: threshold},
            stress_condition=condition,
        )
    return manifests


def compute_noise_degradation(clean_metrics: pd.DataFrame, stressed_metrics: pd.DataFrame, condition: str) -> pd.DataFrame:
    clean = clean_metrics.loc[clean_metrics['split_role'].eq('validation')]
    stressed = stressed_metrics.loc[stressed_metrics['split_role'].eq('validation')]
    keys = ['target', 'metric']
    if clean.duplicated(keys).any() or stressed.duplicated(keys).any():
        raise ValueError('noise degradation requires one metric row per target/metric')
    result = clean.merge(stressed, on=keys, validate='one_to_one', suffixes=('_clean', '_stress'))
    result['stress_condition'] = condition
    denominator = result['value_clean'].abs().replace(0, np.nan)
    result['degradation_rate'] = (result['value_clean'] - result['value_stress']) / denominator
    return result


def verify_metric_identity(manifest_path: Path, metrics_path: Path) -> dict[str, Any]:
    manifest = json.loads(manifest_path.read_text())
    required = ('source_dataset_hash', 'split_hash', 'label_hash', 'feature_hash')
    for field in required:
        _require_sha256(manifest.get(field), field)
    if manifest.get('split_role') != 'validation' or manifest.get('locked_test_included') is not False:
        raise ValueError('comparison identity must be validation-only; locked_test is forbidden')
    if sha256_file(metrics_path) != _require_sha256(manifest.get('metrics_sha256'), 'metrics'):
        raise ValueError('comparison metric hash mismatch')
    return manifest


def write_validation_model_comparison(
    ml_metrics_path: Path, dl_metrics_path: Path,
    ml_manifest_path: Path, dl_manifest_path: Path,
    output_path: Path = DL_BENCHMARK_OUTPUT_ROOT / 'model_comparison_validation.parquet',
) -> Path:
    ml_identity = verify_metric_identity(ml_manifest_path, ml_metrics_path)
    dl_identity = verify_metric_identity(dl_manifest_path, dl_metrics_path)
    identity_fields = ('source_dataset_hash', 'split_hash', 'label_hash', 'feature_hash')
    if any(ml_identity[field] != dl_identity[field] for field in identity_fields):
        raise ValueError('ML/DL validation comparison identity mismatch')
    ml_metrics = pd.read_parquet(ml_metrics_path)
    dl_metrics = pd.read_parquet(dl_metrics_path)
    keys = ['target', 'metric']
    for name, frame in (('ML', ml_metrics), ('DL', dl_metrics)):
        if not frame['split_role'].eq('validation').all() or frame.duplicated(keys).any():
            raise ValueError(f'{name} comparison must be validation-only and one row per key; locked_test is forbidden')
    comparison = ml_metrics.merge(dl_metrics, on=keys, how='inner', suffixes=('_ml', '_dl'), validate='one_to_one')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    comparison.to_parquet(output_path, index=False)
    return output_path


## 11. Guarded Kaggle torchrun launcher와 streaming training worker
노트북 process는 GPU를 먼저 확인하고 `/kaggle/working`에 임시 self-contained worker를 만든 뒤 정확히 두 process만 실행합니다.

In [ ]:
def _python_literal(value: Any) -> str:
    if isinstance(value, Path):
        return f'Path({str(value)!r})'
    return repr(value)


def build_torchrun_worker_source(expected_source_hash: str, expected_split_hash: str) -> str:
    _require_sha256(expected_source_hash, 'GOAL15_EXPECTED_SOURCE_HASH')
    _require_sha256(expected_split_hash, 'GOAL15_EXPECTED_SPLIT_HASH')
    header = '''from __future__ import annotations
import hashlib, inspect, json, os, random, subprocess, tempfile, time
from collections import OrderedDict
from bisect import bisect_right
from dataclasses import dataclass
from datetime import timedelta
from pathlib import Path
from typing import Any, Callable, Iterator, Mapping, Sequence
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, balanced_accuracy_score, brier_score_loss, f1_score, recall_score, roc_auc_score
from torch.nn.parallel import DistributedDataParallel
from torch.utils.data import DataLoader, Dataset, DistributedSampler, Sampler
'''
    constant_names = (
        'SERIES_ID', 'EXPECTED_SPLIT_COUNTS', 'DATA_STATUS', 'REAL_ACCURACY_STATUS',
        'DEVICE_SYNCHRONIZATION_STATUS', 'SEQUENCE_OUTPUT_ROOT', 'ML_VIEW_ROOT',
        'DL_BENCHMARK_OUTPUT_ROOT', 'ML_BENCHMARK_OUTPUT_ROOT', 'SEQUENCE_LENGTHS_SECONDS',
        'PATTERN_TARGET', 'ONSET_EVENT_TARGET', 'STAGE_TARGET', 'STAGE_CODES',
        'BEHAVIOR_CODES', 'CAUSAL_FACTORS', 'ROLLING_STATISTICS',
        'ROLLING_WINDOWS_SECONDS', 'TIME_FEATURE_COLUMNS', 'CONTEXT_FEATURE_COLUMNS',
        'APPROVED_CONTEXTS', 'ALLOWED_FEATURE_COLUMNS', 'PREDICTION_COLUMNS',
        'METRIC_COLUMNS', 'SEQUENCE_INDEX_COLUMNS', 'SEED', 'WANDB_PROJECT',
        'WANDB_GROUP', 'WANDB_TAGS', 'ATTACHED_INPUT_ROOT', 'SEQUENCE_LENGTH_CANDIDATE',
        'THRESHOLD_GRID_SIZE', 'COMMON_THRESHOLD_GRID_SIZE',
        'SEQUENCE_LENGTH_POLICY', 'ROW_GROUP_CACHE_MAX_GROUPS',
        'ROW_GROUP_CACHE_MAX_BYTES', 'TIMELINE_READ_COLUMNS',
        'ROUND3_METRIC_COLUMNS', 'METRIC_UNIQUE_KEYS',
        'RUN_TRAINING', 'RUN_LOCKED_TEST',
        'RUN_VALIDATION_COMPARISON', 'RUN_NOISE_STRESS',
        'RECOMPUTE_NORMALIZATION', 'PREDICTION_SHARD_COLUMNS', 'STRESS_CONDITIONS',
    )
    constants = '\n'.join(f'{name} = {_python_literal(globals()[name])}' for name in constant_names)
    definition_objects = [
        value for name, value in globals().items()
        if (inspect.isfunction(value) or inspect.isclass(value))
        and getattr(value, '__module__', None) == __name__
        and name not in {
            'build_torchrun_worker_source', 'write_guarded_torchrun_worker',
            'launch_dual_t4_torchrun', '_verify_declared_sequence_inputs',
            '_base_compute_metrics_from_predictions',
            'StreamingEvaluationState', 'Round4StreamingEvaluationState',
            'finalize_streaming_metrics', 'round4_finalize_streaming_metrics',
        }
    ]
    ordinary_definitions = '\n\n'.join(inspect.getsource(value) for value in definition_objects)
    streaming_definitions = '\n\n'.join([
        inspect.getsource(Round4StreamingEvaluationState),
        'Round4StreamingEvaluationState = StreamingEvaluationState',
        inspect.getsource(StreamingEvaluationState),
        inspect.getsource(round4_finalize_streaming_metrics),
        'round4_finalize_streaming_metrics = finalize_streaming_metrics',
        inspect.getsource(finalize_streaming_metrics),
    ])
    definitions = ordinary_definitions + '\n\n' + streaming_definitions
    declared_base = inspect.getsource(_verify_declared_sequence_inputs).replace('def verify_sequence_inputs(', 'def _verify_declared_sequence_inputs(', 1)
    metric_base = inspect.getsource(_base_compute_metrics_from_predictions).replace('def compute_metrics_from_predictions(', 'def _base_compute_metrics_from_predictions(', 1)
    trailer = f'''\nos.environ["GOAL15_EXPECTED_SOURCE_HASH"] = {expected_source_hash!r}
os.environ["GOAL15_EXPECTED_SPLIT_HASH"] = {expected_split_hash!r}
run_dl_training()\n'''
    return header + '\n' + constants + '\n\n' + declared_base + '\n\n' + metric_base + '\n\n' + definitions + trailer


def write_guarded_torchrun_worker(source: str) -> Path:
    path = Path('/kaggle/working/goal15_tcn_worker.py')
    path.write_text(source)
    return path


def launch_dual_t4_torchrun() -> None:
    require_exactly_two_cuda_devices()
    verified = verify_sequence_inputs()
    worker_source = build_torchrun_worker_source(verified.source_dataset_hash, verified.split_hash)
    worker_path = write_guarded_torchrun_worker(worker_source)
    if worker_path.parent != Path('/kaggle/working'):
        raise RuntimeError('runtime worker must stay under /kaggle/working')
    environment = os.environ.copy()
    environment['GOAL15_EXPECTED_SOURCE_HASH'] = verified.source_dataset_hash
    environment['GOAL15_EXPECTED_SPLIT_HASH'] = verified.split_hash
    command = [
        'python', '-m', 'torch.distributed.run', '--standalone',
        '--nproc_per_node=2', str(worker_path),
    ]
    subprocess.run(command, check=True, env=environment)


def _collect_small_manifests(local_manifest: Path, rank: int, world_size: int) -> list[Path]:
    gathered: list[str | None] | None = [None] * world_size if rank == 0 else None
    dist.gather_object(str(local_manifest), gathered, dst=0)
    return [Path(value) for value in gathered if value is not None] if gathered is not None else []


def _metric_identity_hashes() -> tuple[str, str]:
    label_payload = json.dumps({'pattern': PATTERN_TARGET, 'onset_audit': ONSET_EVENT_TARGET, 'stages': STAGE_CODES, 'behaviors': BEHAVIOR_CODES}, sort_keys=True)
    feature_payload = json.dumps(list(ALLOWED_FEATURE_COLUMNS), separators=(',', ':'))
    return hashlib.sha256(label_payload.encode()).hexdigest(), hashlib.sha256(feature_payload.encode()).hexdigest()


def write_metrics_identity_manifest(
    metrics_path: Path, *, source_dataset_hash: str, split_hash: str, split_role: str,
) -> Path:
    label_hash, feature_hash = _metric_identity_hashes()
    path = metrics_path.with_suffix('.manifest.json')
    path.write_text(json.dumps({
        'source_dataset_hash': source_dataset_hash, 'split_hash': split_hash,
        'label_hash': label_hash, 'feature_hash': feature_hash,
        'split_role': split_role, 'locked_test_included': split_role == 'locked_test',
        'metrics_sha256': sha256_file(metrics_path),
    }, indent=2, sort_keys=True) + '\n')
    return path


def export_model_state(model: DistributedDataParallel, output_root: Path) -> tuple[Path, str]:
    from safetensors.torch import save_file

    output_root.mkdir(parents=True, exist_ok=True)
    path = output_root / 'causal_tcn.safetensors'
    state = {name: tensor.detach().cpu().contiguous() for name, tensor in model.module.state_dict().items()}
    save_file(state, str(path))
    return path, sha256_file(path)


def run_dl_training(epochs: int = 20, batch_size: int = 64) -> None:
    require_exactly_two_cuda_devices()
    if 'LOCAL_RANK' not in os.environ:
        launch_dual_t4_torchrun()
        return
    rank, local_rank, world_size = setup_ddp()
    device = torch.device('cuda', local_rank)
    try:
        set_deterministic_seed(SEED, rank)
        verified = verify_sequence_inputs()
        expected_source = os.environ.get('GOAL15_EXPECTED_SOURCE_HASH')
        expected_split = os.environ.get('GOAL15_EXPECTED_SPLIT_HASH')
        if verified.source_dataset_hash != expected_source or verified.split_hash != expected_split:
            raise RuntimeError('worker physical identity differs from guarded launcher')
        loss_weights = derive_train_loss_weights(verified.index_paths[f'train_{SEQUENCE_LENGTH_CANDIDATE}'])
        if rank == 0:
            write_train_loss_support(DL_BENCHMARK_OUTPUT_ROOT, loss_weights)
        wandb_enabled = rank == 0 and login_wandb_from_kaggle_secret()
        wandb_module: Any | None = None
        if wandb_enabled:
            import wandb

            wandb_module = wandb
            safe_wandb_call(wandb.init, project=WANDB_PROJECT, group=WANDB_GROUP, tags=WANDB_TAGS, config={'epochs': epochs, 'batch_size': batch_size, 'sequence_length': SEQUENCE_LENGTH_CANDIDATE, 'data_status': DATA_STATUS})
        train_dataset = Goal15SequenceDataset(verified.index_paths[f'train_{SEQUENCE_LENGTH_CANDIDATE}'], verified.timeline_paths['train'], verified.normalization, 'train')
        validation_dataset = Goal15SequenceDataset(verified.index_paths[f'validation_{SEQUENCE_LENGTH_CANDIDATE}'], verified.timeline_paths['validation'], verified.normalization, 'validation')
        train_sampler = DistributedSampler(train_dataset, num_replicas=world_size, rank=rank, shuffle=True, seed=SEED, drop_last=False)
        validation_sampler = PersonShardEvalSampler(validation_dataset, rank=rank, num_replicas=world_size)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler, num_workers=2, pin_memory=True)
        validation_loader = DataLoader(validation_dataset, batch_size=batch_size, sampler=validation_sampler, num_workers=2, pin_memory=True)
        model = Goal15TCN(input_size=len(ALLOWED_FEATURE_COLUMNS)).to(device)
        parameter_count = assert_parameter_budget(model)
        assert_right_padding_invariance(model, len(ALLOWED_FEATURE_COLUMNS))
        model = DistributedDataParallel(model, device_ids=[local_rank], output_device=local_rank)
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
        scaler = torch.amp.GradScaler('cuda')
        for epoch in range(epochs):
            losses = train_one_epoch(model, train_loader, optimizer, scaler, device, epoch=epoch, sampler=train_sampler, loss_weights=loss_weights)
            if rank == 0 and wandb_module is not None:
                safe_wandb_call(wandb_module.log, {f'train/{name}': value for name, value in losses.items()}, step=epoch)
        validation_root = DL_BENCHMARK_OUTPUT_ROOT / 'validation' / 'clean'
        local_manifest = evaluate_to_prediction_shard(model, validation_loader, device, split_role='validation', model_name='causal_tcn', rank=rank, output_path=validation_root / f'rank-{rank}.parquet', source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash)
        manifest_paths = _collect_small_manifests(local_manifest, rank, world_size)
        selected_threshold: float | None = None
        if rank == 0:
            validation_metrics, selected_threshold, confusion = aggregate_prediction_shards(manifest_paths, expected_window_count=len(validation_dataset), source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash, model_name='causal_tcn')
            DL_BENCHMARK_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
            metrics_path = DL_BENCHMARK_OUTPUT_ROOT / 'validation_metrics.parquet'
            validation_metrics.to_parquet(metrics_path, index=False)
            confusion.to_parquet(DL_BENCHMARK_OUTPUT_ROOT / 'validation_stage_confusion.parquet', index=False)
            metrics_manifest = write_metrics_identity_manifest(metrics_path, source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash, split_role='validation')
            export_model_state(model, DL_BENCHMARK_OUTPUT_ROOT)
            if RUN_VALIDATION_COMPARISON:
                write_validation_model_comparison(ML_BENCHMARK_OUTPUT_ROOT / 'validation_metrics.parquet', metrics_path, ML_BENCHMARK_OUTPUT_ROOT / 'validation_metrics.manifest.json', metrics_manifest)
        threshold_holder = [selected_threshold]
        dist.broadcast_object_list(threshold_holder, src=0)
        selected_threshold = float(threshold_holder[0])
        if RUN_NOISE_STRESS:
            stress_local = evaluate_noise_stress_to_shards(model, validation_loader, device, split_role='validation', model_name='causal_tcn', rank=rank, output_root=DL_BENCHMARK_OUTPUT_ROOT / 'validation' / 'stress', source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash, threshold=selected_threshold)
            for condition, stress_manifest in stress_local.items():
                stress_paths = _collect_small_manifests(stress_manifest, rank, world_size)
                if rank == 0:
                    stress_metrics, _, _ = aggregate_prediction_shards(stress_paths, expected_window_count=len(validation_dataset), source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash, model_name='causal_tcn')
                    degradation = compute_noise_degradation(validation_metrics, stress_metrics, condition)
                    degradation.to_parquet(DL_BENCHMARK_OUTPUT_ROOT / f'validation_noise_{condition}.parquet', index=False)
        if RUN_LOCKED_TEST:
            locked_dataset = Goal15SequenceDataset(verified.index_paths[f'locked_test_{SEQUENCE_LENGTH_CANDIDATE}'], verified.timeline_paths['locked_test'], verified.normalization, 'locked_test')
            locked_sampler = PersonShardEvalSampler(locked_dataset, rank=rank, num_replicas=world_size)
            locked_loader = DataLoader(locked_dataset, batch_size=batch_size, sampler=locked_sampler, num_workers=2, pin_memory=True)
            locked_manifest = evaluate_to_prediction_shard(model, locked_loader, device, split_role='locked_test', model_name='causal_tcn', rank=rank, output_path=DL_BENCHMARK_OUTPUT_ROOT / 'locked_test' / f'rank-{rank}.parquet', source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash, thresholds={PATTERN_TARGET: selected_threshold})
            locked_paths = _collect_small_manifests(locked_manifest, rank, world_size)
            if rank == 0:
                locked_metrics, _, locked_confusion = aggregate_prediction_shards(locked_paths, expected_window_count=len(locked_dataset), source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash, model_name='causal_tcn')
                locked_metrics_path = DL_BENCHMARK_OUTPUT_ROOT / 'locked_test_metrics.parquet'
                locked_metrics.to_parquet(locked_metrics_path, index=False)
                locked_confusion.to_parquet(DL_BENCHMARK_OUTPUT_ROOT / 'locked_test_stage_confusion.parquet', index=False)
                write_metrics_identity_manifest(locked_metrics_path, source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash, split_role='locked_test')
        if rank == 0 and wandb_module is not None:
            safe_wandb_call(wandb_module.log, {'model/parameter_count': parameter_count})
            safe_wandb_call(wandb_module.finish)
    finally:
        cleanup_ddp()


if RUN_TRAINING:
    print('Round3 검증 정의를 계속 로드합니다.')
else:
    print('학습 비활성화: guarded torchrun worker를 생성하거나 실행하지 않았습니다.')


## 12. Round3 공통-grid 지표·threshold 소유권
Clean validation만 event-level threshold를 선택합니다. Stress와 locked-test는 해시로 고정된 clean threshold만 소비하며, 전역 비선형 지표와 person-macro 지표를 분리합니다.

In [ ]:
from collections import OrderedDict
from typing import Callable, Iterator

ROUND3_METRIC_COLUMNS = [*METRIC_COLUMNS, 'stress_condition']
METRIC_UNIQUE_KEYS = ['target', 'metric', 'split_role', 'stress_condition']


def assert_unique_metric_rows(metrics: pd.DataFrame) -> None:
    if 'stress_condition' not in metrics.columns:
        raise ValueError('stress_condition is required in metric ownership keys')
    missing = sorted(set(METRIC_UNIQUE_KEYS).difference(metrics.columns))
    if missing:
        raise ValueError(f'metric uniqueness columns missing: {missing}')
    if metrics.duplicated(METRIC_UNIQUE_KEYS).any():
        duplicate = metrics.loc[metrics.duplicated(METRIC_UNIQUE_KEYS, keep=False), METRIC_UNIQUE_KEYS]
        raise ValueError(f'duplicate metric owner rows: {duplicate.to_dict("records")[:5]}')


def _histogram_binary_metrics(
    positive: np.ndarray, negative: np.ndarray, probability_sum: np.ndarray,
    brier_sum: float, threshold_index: int,
) -> dict[str, float]:
    positive = positive.astype(np.float64, copy=False)
    negative = negative.astype(np.float64, copy=False)
    total_positive = float(positive.sum())
    total_negative = float(negative.sum())
    total = total_positive + total_negative
    tp = np.cumsum(positive[::-1])
    fp = np.cumsum(negative[::-1])
    recall = np.divide(tp, total_positive, out=np.zeros_like(tp), where=total_positive > 0)
    precision = np.divide(tp, tp + fp, out=np.ones_like(tp), where=(tp + fp) > 0)
    recall_delta = np.diff(np.r_[0.0, recall])
    global_aucpr = float(np.sum(recall_delta * precision)) if total_positive and total_negative else np.nan
    fpr = np.divide(fp, total_negative, out=np.zeros_like(fp), where=total_negative > 0)
    global_auroc = float(np.trapezoid(np.r_[0.0, recall], np.r_[0.0, fpr])) if total_positive and total_negative else np.nan
    reverse_index = len(positive) - 1 - threshold_index
    selected_tp = float(tp[reverse_index])
    selected_fp = float(fp[reverse_index])
    selected_fn = total_positive - selected_tp
    denominator = 2 * selected_tp + selected_fp + selected_fn
    global_row_f1 = 2 * selected_tp / denominator if denominator else 0.0
    count = positive + negative
    calibration = np.divide(probability_sum, count, out=np.zeros_like(count), where=count > 0)
    observed = np.divide(positive, count, out=np.zeros_like(count), where=count > 0)
    global_ece = float(np.sum(count * np.abs(observed - calibration)) / total) if total else np.nan
    return {
        'global_aucpr': global_aucpr, 'global_auroc': global_auroc,
        'global_row_f1': global_row_f1,
        'global_brier_score': float(brier_sum / total) if total else np.nan,
        'global_ece': global_ece,
    }


class StreamingMetricAccumulator:
    '''Bounded global_aucpr/global_auroc/global_row_f1/global_ece owner.'''
    def __init__(self, *, bins: int = COMMON_THRESHOLD_GRID_SIZE) -> None:
        self.bins = bins
        self.histograms: dict[str, dict[str, Any]] = {}

    def update_histograms(self, frame: pd.DataFrame) -> None:
        for target, rows in frame.groupby('target', sort=False):
            probability = rows['probability'].to_numpy(dtype=np.float64)
            if ((probability < 0) | (probability > 1) | ~np.isfinite(probability)).any():
                raise ValueError('prediction probability must be finite in [0, 1]')
            bucket = np.minimum((probability * (self.bins - 1)).astype(np.int64), self.bins - 1)
            state = self.histograms.setdefault(target, {
                'positive': np.zeros(self.bins), 'negative': np.zeros(self.bins),
                'probability_sum': np.zeros(self.bins), 'brier_sum': 0.0,
            })
            labels = rows['label'].to_numpy(dtype=np.int8)
            np.add.at(state['positive'], bucket[labels == 1], 1)
            np.add.at(state['negative'], bucket[labels == 0], 1)
            np.add.at(state['probability_sum'], bucket, probability)
            state['brier_sum'] += float(np.square(probability - labels).sum())

    def merge(self, other: 'StreamingMetricAccumulator') -> None:
        if self.bins != other.bins:
            raise ValueError('histogram grid mismatch')
        for target, incoming in other.histograms.items():
            state = self.histograms.setdefault(target, {
                'positive': np.zeros(self.bins), 'negative': np.zeros(self.bins),
                'probability_sum': np.zeros(self.bins), 'brier_sum': 0.0,
            })
            for name in ('positive', 'negative', 'probability_sum'):
                state[name] += incoming[name]
            state['brier_sum'] += incoming['brier_sum']

    def global_binary_metrics(self, target: str, threshold: float) -> dict[str, float]:
        state = self.histograms[target]
        threshold_index = int(round(threshold * (self.bins - 1)))
        return _histogram_binary_metrics(
            state['positive'], state['negative'], state['probability_sum'],
            float(state['brier_sum']), threshold_index,
        )


def _range_add(values: np.ndarray, lower: int, upper: int) -> None:
    if lower > upper:
        return
    values[lower] += 1
    if upper + 1 < len(values):
        values[upper + 1] -= 1


def person_event_threshold_statistics(
    pattern_rows: pd.DataFrame, *, bins: int = COMMON_THRESHOLD_GRID_SIZE,
) -> dict[str, Any]:
    ordered = pattern_rows.sort_values('canonical_time', kind='mergesort')
    truth = ordered['label'].to_numpy(dtype=np.int8).astype(bool)
    probability = ordered['probability'].to_numpy(dtype=np.float64)
    bucket = np.minimum((probability * (bins - 1)).astype(np.int64), bins - 1)
    detected_difference = np.zeros(bins, dtype=np.int64)
    truth_segments = _segments(truth)
    for start, end in truth_segments:
        _range_add(detected_difference, 0, int(bucket[start:end + 1].max()))
    false_difference = np.zeros(bins, dtype=np.int64)
    for index in np.flatnonzero(~truth):
        current = int(bucket[index])
        if index == 0 or truth[index - 1]:
            _range_add(false_difference, 0, current)
        else:
            previous = int(bucket[index - 1])
            if current > previous:
                _range_add(false_difference, previous + 1, current)
    times = pd.to_datetime(ordered['canonical_time'], utc=True)
    duration_hours = max(float((times.iloc[-1] - times.iloc[0]).total_seconds()) / 3600, 1 / 3600) if len(times) else 0.0
    return {
        'detected': np.cumsum(detected_difference),
        'false_alerts': np.cumsum(false_difference),
        'truth_events': len(truth_segments), 'duration_hours': duration_hours,
    }


def select_clean_validation_threshold(
    person_sufficient_statistics: Sequence[Mapping[str, Any]],
) -> tuple[float, dict[str, float]]:
    if not person_sufficient_statistics:
        raise ValueError('select_clean_validation requires person statistics')
    detected = sum((item['event']['detected'] for item in person_sufficient_statistics), np.zeros(COMMON_THRESHOLD_GRID_SIZE))
    false_alerts = sum((item['event']['false_alerts'] for item in person_sufficient_statistics), np.zeros(COMMON_THRESHOLD_GRID_SIZE))
    truth_events = sum(int(item['event']['truth_events']) for item in person_sufficient_statistics)
    duration_hours = sum(float(item['event']['duration_hours']) for item in person_sufficient_statistics)
    event_recall = np.divide(detected, truth_events, out=np.zeros_like(detected), where=truth_events > 0)
    event_level_f1 = np.divide(
        2 * detected, truth_events + detected + false_alerts,
        out=np.zeros_like(detected), where=(truth_events + detected + false_alerts) > 0,
    )
    false_alerts_per_hour = false_alerts / max(duration_hours, 1 / 3600)
    grid = np.linspace(0.0, 1.0, COMMON_THRESHOLD_GRID_SIZE)
    best = max(
        range(COMMON_THRESHOLD_GRID_SIZE),
        key=lambda index: (
            float(event_level_f1[index]), float(event_recall[index]),
            -float(false_alerts_per_hour[index]), float(grid[index]),
        ),
    )
    return float(grid[best]), {
        'event_level_f1': float(event_level_f1[best]),
        'event_recall': float(event_recall[best]),
        'false_alerts_per_hour': float(false_alerts_per_hour[best]),
    }


def write_clean_validation_threshold(
    output_root: Path, *, threshold: float, source_dataset_hash: str,
    split_hash: str, model_name: str,
) -> tuple[Path, str]:
    output_root.mkdir(parents=True, exist_ok=True)
    path = output_root / 'clean_validation_threshold.json'
    path.write_text(json.dumps({
        'schema_version': 'goal1.5/clean-validation-threshold/v1',
        'threshold_mode': 'select_clean_validation', 'threshold': threshold,
        'grid_size': COMMON_THRESHOLD_GRID_SIZE, 'selector_primary': 'event_level_f1',
        'selector_ties': ['event_recall', 'false_alerts_per_hour', 'threshold'],
        'source_dataset_hash': source_dataset_hash, 'split_hash': split_hash,
        'model_name': model_name, 'target': PATTERN_TARGET,
    }, indent=2, sort_keys=True) + '\n')
    return path, sha256_file(path)


def load_fixed_threshold(
    threshold_artifact: Path, *, threshold_artifact_hash: str,
    source_dataset_hash: str, split_hash: str, model_name: str,
) -> float:
    if sha256_file(threshold_artifact) != _require_sha256(threshold_artifact_hash, 'threshold_artifact_hash'):
        raise ValueError('fixed_threshold artifact hash mismatch')
    payload = json.loads(threshold_artifact.read_text())
    expected = {
        'threshold_mode': 'select_clean_validation',
        'source_dataset_hash': source_dataset_hash, 'split_hash': split_hash,
        'model_name': model_name, 'target': PATTERN_TARGET,
    }
    if any(payload.get(key) != value for key, value in expected.items()):
        raise ValueError('fixed_threshold identity mismatch')
    threshold = float(payload['threshold'])
    if not 0 <= threshold <= 1:
        raise ValueError('fixed_threshold is outside [0, 1]')
    return threshold


def _person_sufficient_statistics(person: pd.DataFrame) -> dict[str, Any]:
    pattern = person.loc[person['target'].eq(PATTERN_TARGET)].copy()
    accumulator = StreamingMetricAccumulator()
    accumulator.update_histograms(person)
    return {
        'person_key': str(pattern['person_key'].iloc[0]),
        'histograms': accumulator.histograms,
        'event': person_event_threshold_statistics(pattern),
    }


def _event_metrics_from_statistics(
    person_sufficient_statistics: Sequence[Mapping[str, Any]], threshold: float,
) -> dict[str, float]:
    index = int(round(threshold * (COMMON_THRESHOLD_GRID_SIZE - 1)))
    detected = sum(float(item['event']['detected'][index]) for item in person_sufficient_statistics)
    false_alerts = sum(float(item['event']['false_alerts'][index]) for item in person_sufficient_statistics)
    truth_events = sum(int(item['event']['truth_events']) for item in person_sufficient_statistics)
    duration_hours = sum(float(item['event']['duration_hours']) for item in person_sufficient_statistics)
    recall = detected / truth_events if truth_events else 0.0
    event_f1 = 2 * detected / (truth_events + detected + false_alerts) if (truth_events + detected + false_alerts) else 0.0
    return {
        'global_event_level_f1': event_f1, 'global_event_recall': recall,
        'global_false_alerts_per_hour': false_alerts / max(duration_hours, 1 / 3600),
    }


def bootstrap_person_metrics(
    person_sufficient_statistics: Sequence[Mapping[str, Any]], *, threshold: float,
    iterations: int = 1000, seed: int = SEED,
) -> dict[str, float]:
    if len(person_sufficient_statistics) < 2:
        raise ValueError('person_sufficient_statistics needs at least two people')
    ci_metric_names = (
        'global_aucpr_ci_lower', 'global_event_recall_ci_lower',
        'global_false_alerts_per_hour_ci_upper', 'person_macro_f1_ci_lower',
    )
    _ = ci_metric_names
    generator = np.random.default_rng(seed)
    samples: dict[str, list[float]] = {
        'global_aucpr': [], 'global_event_recall': [],
        'global_false_alerts_per_hour': [], 'person_macro_f1': [],
    }
    threshold_index = int(round(threshold * (COMMON_THRESHOLD_GRID_SIZE - 1)))
    for selected in generator.integers(0, len(person_sufficient_statistics), size=(iterations, len(person_sufficient_statistics))):
        chosen = [person_sufficient_statistics[int(index)] for index in selected]
        accumulator = StreamingMetricAccumulator()
        person_f1: list[float] = []
        for item in chosen:
            local = StreamingMetricAccumulator()
            local.histograms = item['histograms']
            accumulator.merge(local)
            state = item['histograms'][PATTERN_TARGET]
            person_f1.append(_histogram_binary_metrics(
                state['positive'], state['negative'], state['probability_sum'],
                float(state['brier_sum']), threshold_index,
            )['global_row_f1'])
        samples['global_aucpr'].append(accumulator.global_binary_metrics(PATTERN_TARGET, threshold)['global_aucpr'])
        events = _event_metrics_from_statistics(chosen, threshold)
        samples['global_event_recall'].append(events['global_event_recall'])
        samples['global_false_alerts_per_hour'].append(events['global_false_alerts_per_hour'])
        samples['person_macro_f1'].append(float(np.mean(person_f1)))
    result: dict[str, float] = {}
    for metric, values in samples.items():
        lower, upper = np.nanquantile(np.asarray(values, dtype=np.float64), [0.025, 0.975])
        result[f'{metric}_ci_lower'] = float(lower)
        result[f'{metric}_ci_upper'] = float(upper)
    return result


In [ ]:
def validate_person_shard_semantics(
    person_rows: pd.DataFrame, expected_person_windows: pd.DataFrame,
) -> None:
    expected_person_windows = expected_person_windows.copy()
    if 'canonical_time' not in expected_person_windows and 'prediction_time' in expected_person_windows:
        expected_person_windows['canonical_time'] = pd.to_datetime(
            expected_person_windows['prediction_time'], utc=True, errors='raise',
        ).astype(str)
    person_rows = person_rows.copy()
    person_rows['canonical_time'] = pd.to_datetime(
        person_rows['canonical_time'], utc=True, errors='raise',
    ).astype(str)
    identity = ['dataset_id', 'run_id', 'person_key', 'window_id', 'canonical_time', 'split_role']
    missing = sorted(set(identity + [PATTERN_TARGET, 'hard_negative', *BEHAVIOR_CODES]).difference(expected_person_windows.columns))
    if missing:
        raise ValueError(f'expected person window columns missing: {missing}')
    if expected_person_windows['person_key'].nunique() != 1 or person_rows['person_key'].nunique() != 1:
        raise ValueError('expected_person_windows and shard must each contain one person')
    expected = expected_person_windows.sort_values('canonical_time', kind='mergesort').reset_index(drop=True)
    allowed_targets = {
        PATTERN_TARGET, *(f'stage::{code}' for code in STAGE_CODES),
        *(f'behavior::{code}' for code in BEHAVIOR_CODES),
    }
    extras = set(person_rows['target']) - allowed_targets
    if extras:
        raise ValueError(f'unexpected shard targets: {sorted(extras)}')
    if expected[identity].duplicated().any() or not pd.to_datetime(expected['canonical_time'], utc=True).is_monotonic_increasing:
        raise ValueError('expected index identity is duplicate or not in canonical order')
    pattern = person_rows.loc[person_rows['target'].eq(PATTERN_TARGET)].sort_values('canonical_time', kind='mergesort').reset_index(drop=True)
    if len(pattern) != len(expected) or pattern[identity].to_dict('records') != expected[identity].to_dict('records'):
        raise ValueError('one pattern row per expected window is required with exact identity')
    if not np.array_equal(pattern['label'].to_numpy(dtype=np.int8), expected[PATTERN_TARGET].to_numpy(dtype=np.int8)):
        raise ValueError('pattern labels differ from expected index rows')
    if not np.array_equal(pattern['audit_event_binary'].to_numpy(dtype=np.int8), expected[ONSET_EVENT_TARGET].to_numpy(dtype=np.int8)):
        raise ValueError('audit event labels differ from expected index rows')
    stage_expected = expected.loc[expected[PATTERN_TARGET].eq(1), identity]
    stage = person_rows.loc[person_rows['target'].str.startswith('stage::')]
    if set(stage['target'].unique()) != {f'stage::{code}' for code in set(STAGE_CODES)}:
        raise ValueError('stage targets must be exactly set(STAGE_CODES)')
    if stage.duplicated([*identity, 'target']).any():
        raise ValueError('stage target rows are duplicated')
    stage_counts = stage.groupby(identity, sort=False)['target'].agg(lambda values: set(values))
    if len(stage_counts) != len(stage_expected) or any(values != {f'stage::{code}' for code in STAGE_CODES} for values in stage_counts):
        raise ValueError('five unique stage targets are required only for pattern windows')
    if set(stage_counts.index) != set(map(tuple, stage_expected.to_numpy())):
        raise ValueError('stage conditional window identities mismatch')
    stage_truth = stage.loc[stage['label'].eq(1), [*identity, 'target']]
    expected_stage_truth = expected.loc[
        expected[PATTERN_TARGET].eq(1), identity + [STAGE_TARGET],
    ].copy()
    expected_stage_truth['target'] = 'stage::' + expected_stage_truth[STAGE_TARGET].astype(str)
    if stage_truth.sort_values(identity).reset_index(drop=True).to_dict('records') != expected_stage_truth.loc[:, [*identity, 'target']].sort_values(identity).reset_index(drop=True).to_dict('records'):
        raise ValueError('stage truth labels differ from expected index rows')
    behavior_positive = expected.loc[:, list(BEHAVIOR_CODES)].eq(1).any(axis=1)
    decision_mask = expected[PATTERN_TARGET].eq(1) | (expected['hard_negative'].eq(1) & behavior_positive)
    behavior_expected = expected.loc[decision_mask, identity]
    behavior = person_rows.loc[person_rows['target'].str.startswith('behavior::')]
    if set(behavior['target'].unique()) != {f'behavior::{code}' for code in set(BEHAVIOR_CODES)}:
        raise ValueError('behavior targets must be exactly set(BEHAVIOR_CODES)')
    if behavior.duplicated([*identity, 'target']).any():
        raise ValueError('behavior target rows are duplicated')
    behavior_counts = behavior.groupby(identity, sort=False)['target'].agg(lambda values: set(values))
    if len(behavior_counts) != len(behavior_expected) or any(values != {f'behavior::{code}' for code in BEHAVIOR_CODES} for values in behavior_counts):
        raise ValueError('ten unique behavior targets are required only for decision_mask windows')
    if set(behavior_counts.index) != set(map(tuple, behavior_expected.to_numpy())):
        raise ValueError('behavior conditional window identities mismatch')
    behavior_labels = behavior.pivot(index=identity, columns='target', values='label')
    expected_behavior = expected.loc[decision_mask].set_index(identity)
    for code in BEHAVIOR_CODES:
        target = f'behavior::{code}'
        if not np.array_equal(
            behavior_labels.loc[expected_behavior.index, target].to_numpy(dtype=np.int8),
            expected_behavior[code].to_numpy(dtype=np.int8),
        ):
            raise ValueError(f'behavior truth label mismatch: {code}')


def compute_metrics_from_predictions(
    predictions: pd.DataFrame, *, model_name: str,
) -> pd.DataFrame:
    missing = sorted(set(PREDICTION_COLUMNS).difference(predictions.columns))
    if missing or predictions.empty:
        raise ValueError(f'common predictions are empty or incomplete: {missing}')
    accumulator = StreamingMetricAccumulator()
    accumulator.update_histograms(predictions)
    rows: list[dict[str, Any]] = []
    split_role = str(predictions['split_role'].iloc[0])
    for target, state in accumulator.histograms.items():
        threshold = float(predictions.loc[predictions['target'].eq(target), 'threshold'].iloc[0])
        for metric, value in accumulator.global_binary_metrics(target, threshold).items():
            rows.append({
                'model_family': 'deep_learning_tcn', 'model_name': model_name,
                'series_id': SERIES_ID, 'split_role': split_role, 'target': target,
                'metric': metric, 'value': value,
                'support': int(state['positive'].sum() + state['negative'].sum()),
                'data_status': DATA_STATUS, 'stress_condition': 'clean',
            })
    result = pd.DataFrame(rows, columns=ROUND3_METRIC_COLUMNS)
    assert_unique_metric_rows(result)
    return result


def _round3_metric_row(
    model_name: str, split_role: str, target: str, metric: str, value: float,
    support: int, stress_condition: str,
) -> dict[str, Any]:
    return {
        'model_family': 'deep_learning_tcn', 'model_name': model_name,
        'series_id': SERIES_ID, 'split_role': split_role, 'target': target,
        'metric': metric, 'value': value, 'support': support,
        'data_status': DATA_STATUS, 'stress_condition': stress_condition,
    }


def aggregate_prediction_shards(
    manifest_paths: Sequence[Path], *, expected_window_count: int,
    source_dataset_hash: str, split_hash: str, model_name: str,
    expected_person_windows: Callable[[str], pd.DataFrame],
    threshold_mode: str,
    threshold_artifact: Path | None = None,
    threshold_artifact_hash: str | None = None,
    stress_condition: str = 'clean',
) -> tuple[pd.DataFrame, float, pd.DataFrame]:
    if threshold_mode not in {'select_clean_validation', 'fixed_threshold'}:
        raise ValueError('unknown threshold_mode')
    required_stream_columns = {'window_id', 'audit_event_binary'}
    shard_paths: list[Path] = []
    declared_windows = 0
    split_roles: set[str] = set()
    for manifest_path in sorted(manifest_paths):
        manifest = json.loads(manifest_path.read_text())
        shard_path = manifest_path.parent / manifest['path']
        if manifest.get('source_dataset_hash') != source_dataset_hash or manifest.get('split_hash') != split_hash:
            raise ValueError('prediction shard identity mismatch')
        if manifest.get('columns') != PREDICTION_SHARD_COLUMNS or sha256_file(shard_path) != manifest.get('sha256'):
            raise ValueError('prediction shard schema/hash mismatch')
        if pq.ParquetFile(shard_path).metadata.num_rows != manifest.get('row_count'):
            raise ValueError('prediction shard row count mismatch')
        declared_windows += int(manifest['window_count'])
        split_roles.add(str(manifest['split_role']))
        shard_paths.append(shard_path)
    if declared_windows != expected_window_count or len(split_roles) != 1:
        raise ValueError('prediction shard window coverage/role mismatch')
    split_role = next(iter(split_roles))
    if threshold_mode == 'select_clean_validation' and (split_role != 'validation' or stress_condition != 'clean'):
        raise ValueError('select_clean_validation is allowed only for clean validation')
    accumulator = StreamingMetricAccumulator()
    seen_people: set[str] = set()
    person_sufficient_statistics: list[dict[str, Any]] = []
    confusion_frames: list[pd.DataFrame] = []
    for shard_path in shard_paths:
        for person in _iter_shard_people(shard_path):
            if not required_stream_columns.issubset(person.columns):
                raise ValueError('window_id/audit_event_binary lost before metric ownership')
            _validate_person_prediction_rows(person)
            person_key = str(person['person_key'].iloc[0])
            if person_key in seen_people:
                raise ValueError('person prediction appears in multiple shards')
            seen_people.add(person_key)
            validate_person_shard_semantics(person, expected_person_windows(person_key))
            accumulator.update_histograms(person)
            person_sufficient_statistics.append(_person_sufficient_statistics(person))
            confusion_frames.append(compute_stage_confusion_rows(person))
    if threshold_mode == 'select_clean_validation':
        threshold, _ = select_clean_validation_threshold(person_sufficient_statistics)
    else:
        if threshold_artifact is None or threshold_artifact_hash is None:
            raise ValueError('fixed_threshold requires persisted threshold_artifact and hash')
        threshold = load_fixed_threshold(
            threshold_artifact, threshold_artifact_hash=threshold_artifact_hash,
            source_dataset_hash=source_dataset_hash, split_hash=split_hash,
            model_name=model_name,
        )
    threshold_index = int(round(threshold * (COMMON_THRESHOLD_GRID_SIZE - 1)))
    rows: list[dict[str, Any]] = []
    for target, state in sorted(accumulator.histograms.items()):
        target_threshold = threshold if target == PATTERN_TARGET else 0.5
        support = int(state['positive'].sum() + state['negative'].sum())
        for metric, value in accumulator.global_binary_metrics(target, target_threshold).items():
            rows.append(_round3_metric_row(model_name, split_role, target, metric, value, support, stress_condition))
        rows.append(_round3_metric_row(
            model_name, split_role, target, 'global_positive_support',
            float(state['positive'].sum()), support, stress_condition,
        ))
    behavior_accumulator = StreamingMetricAccumulator()
    behavior_state = {
        'positive': np.zeros(COMMON_THRESHOLD_GRID_SIZE),
        'negative': np.zeros(COMMON_THRESHOLD_GRID_SIZE),
        'probability_sum': np.zeros(COMMON_THRESHOLD_GRID_SIZE), 'brier_sum': 0.0,
    }
    for target, state in accumulator.histograms.items():
        if not target.startswith('behavior::'):
            continue
        for name in ('positive', 'negative', 'probability_sum'):
            behavior_state[name] += state[name]
        behavior_state['brier_sum'] += state['brier_sum']
    behavior_accumulator.histograms['behavior::all'] = behavior_state
    if behavior_state['positive'].sum() + behavior_state['negative'].sum():
        behavior_metrics = behavior_accumulator.global_binary_metrics('behavior::all', 0.5)
        behavior_names = {
            'global_aucpr': 'behavior_micro_aucpr',
            'global_auroc': 'behavior_micro_auroc',
            'global_row_f1': 'behavior_micro_f1',
        }
        for source_name, output_name in behavior_names.items():
            rows.append(_round3_metric_row(
                model_name, split_role, 'behavior::all', output_name,
                behavior_metrics[source_name], int(behavior_state['positive'].sum() + behavior_state['negative'].sum()), stress_condition,
            ))
    event_metrics = _event_metrics_from_statistics(person_sufficient_statistics, threshold)
    for metric, value in event_metrics.items():
        rows.append(_round3_metric_row(model_name, split_role, PATTERN_TARGET, metric, value, expected_window_count, stress_condition))
    person_f1 = []
    for item in person_sufficient_statistics:
        state = item['histograms'][PATTERN_TARGET]
        person_f1.append(_histogram_binary_metrics(
            state['positive'], state['negative'], state['probability_sum'],
            float(state['brier_sum']), threshold_index,
        )['global_row_f1'])
    rows.append(_round3_metric_row(model_name, split_role, PATTERN_TARGET, 'person_macro_f1', float(np.mean(person_f1)), len(person_f1), stress_condition))
    lead_times: list[float] = []
    for shard_path in shard_paths:
        for person in _iter_shard_people(shard_path):
            pattern = person.loc[person['target'].eq(PATTERN_TARGET)].copy()
            pattern['threshold'] = threshold
            lead_times.extend(compute_forecast_lead_times(pattern))
    rows.append(_round3_metric_row(
        model_name, split_role, PATTERN_TARGET, 'forecast_lead_mean',
        float(np.mean(lead_times)) if lead_times else np.nan, len(lead_times), stress_condition,
    ))
    rows.append(_round3_metric_row(
        model_name, split_role, PATTERN_TARGET, 'forecast_lead_median',
        float(np.median(lead_times)) if lead_times else np.nan, len(lead_times), stress_condition,
    ))
    confusion = pd.concat(confusion_frames, ignore_index=True).groupby(['actual', 'predicted'], as_index=False)['count'].sum() if confusion_frames else pd.DataFrame(columns=['actual', 'predicted', 'count'])
    if not confusion.empty:
        labels = [f'stage::{code}' for code in STAGE_CODES]
        matrix = confusion.pivot(index='actual', columns='predicted', values='count').reindex(index=labels, columns=labels, fill_value=0).fillna(0)
        recalls = np.divide(np.diag(matrix), matrix.sum(axis=1), out=np.zeros(len(labels)), where=matrix.sum(axis=1).to_numpy() > 0)
        precision = np.divide(np.diag(matrix), matrix.sum(axis=0), out=np.zeros(len(labels)), where=matrix.sum(axis=0).to_numpy() > 0)
        f1 = np.divide(2 * precision * recalls, precision + recalls, out=np.zeros(len(labels)), where=(precision + recalls) > 0)
        rows.append(_round3_metric_row(model_name, split_role, 'stage::all', 'stage_macro_f1', float(np.mean(f1)), int(matrix.to_numpy().sum()), stress_condition))
        rows.append(_round3_metric_row(model_name, split_role, 'stage::all', 'stage_balanced_accuracy', float(np.mean(recalls)), int(matrix.to_numpy().sum()), stress_condition))
        for code, recall in zip(STAGE_CODES, recalls, strict=True):
            rows.append(_round3_metric_row(model_name, split_role, f'stage::{code}', 'global_stage_recall', float(recall), int(matrix.loc[f'stage::{code}'].sum()), stress_condition))
    bootstrap = bootstrap_person_metrics(person_sufficient_statistics, threshold=threshold)
    for metric, value in bootstrap.items():
        rows.append(_round3_metric_row(model_name, split_role, PATTERN_TARGET, metric, value, len(person_sufficient_statistics), stress_condition))
    metrics = pd.DataFrame(rows, columns=ROUND3_METRIC_COLUMNS)
    assert_unique_metric_rows(metrics)
    return metrics, threshold, confusion


## 13. Bounded row-group IO와 person-local block sampler
600초 후보는 timeline row-group metadata interval index와 bounded LRU를 사용합니다. 훈련은 사람별 연속 block을 rank에 정확히 한 번 배정하고 epoch마다 block 순서만 바꿉니다.

In [ ]:
TIMELINE_READ_COLUMNS = (
    'person_key', 'run_id', 'dataset_id', 'canonical_time', 'context',
    PATTERN_TARGET, ONSET_EVENT_TARGET, 'hard_negative', STAGE_TARGET,
    *BEHAVIOR_CODES,
)


class BoundedRowGroupCache:
    def __init__(self, *, max_groups: int, max_bytes: int) -> None:
        if max_groups < 1 or max_bytes < 1:
            raise ValueError('row-group cache bounds must be positive')
        self.max_groups = max_groups
        self.max_bytes = max_bytes
        self._values: OrderedDict[Any, tuple[pd.DataFrame, int]] = OrderedDict()
        self.current_bytes = 0

    @property
    def current_groups(self) -> int:
        return len(self._values)

    def get(self, key: Any, loader: Callable[[], pd.DataFrame]) -> pd.DataFrame:
        if key in self._values:
            frame, size = self._values.pop(key)
            self._values[key] = (frame, size)
            return frame
        frame = loader()
        if not isinstance(frame, pd.DataFrame):
            raise TypeError('row-group loader must return a DataFrame')
        size = int(frame.memory_usage(index=True, deep=True).sum())
        if size > self.max_bytes:
            return frame
        while self._values and (
            len(self._values) >= self.max_groups or self.current_bytes + size > self.max_bytes
        ):
            _, (_, evicted_size) = self._values.popitem(last=False)
            self.current_bytes -= evicted_size
        self._values[key] = (frame, size)
        self.current_bytes += size
        return frame


class RowGroupIntervalIndex:
    def __init__(self, parquet: pq.ParquetFile, split_role: str) -> None:
        schema_names = list(parquet.schema_arrow.names)
        required = {'dataset_id', 'canonical_time', 'split_role'}
        if not required.issubset(schema_names):
            raise ValueError('timeline row-group interval metadata columns missing')
        positions = {name: schema_names.index(name) for name in required}
        self.intervals: dict[str, list[tuple[pd.Timestamp, pd.Timestamp, int]]] = {}
        for row_group in range(parquet.num_row_groups):
            metadata = parquet.metadata.row_group(row_group)
            statistics = {
                name: metadata.column(position).statistics
                for name, position in positions.items()
            }
            if any(value is None or not value.has_min_max for value in statistics.values()):
                raise ValueError('timeline row-group statistics are required for bounded IO')
            if statistics['split_role'].min != split_role or statistics['split_role'].max != split_role:
                raise ValueError('timeline row-group role metadata mismatch')
            if statistics['dataset_id'].min != statistics['dataset_id'].max:
                raise ValueError('timeline row-group metadata must bind one dataset')
            dataset_id = str(statistics['dataset_id'].min)
            first = pd.Timestamp(statistics['canonical_time'].min)
            last = pd.Timestamp(statistics['canonical_time'].max)
            self.intervals.setdefault(dataset_id, []).append((first, last, row_group))
        for dataset_id in self.intervals:
            self.intervals[dataset_id].sort(key=lambda value: value[0])

    def overlapping(
        self, dataset_id: str, window_start: pd.Timestamp, window_end: pd.Timestamp,
    ) -> Iterator[int]:
        for first, last, row_group in self.intervals.get(dataset_id, []):
            if first > window_end:
                break
            if last >= window_start:
                yield row_group


class PersonBlockBatchSampler(Sampler[list[int]]):
    '''Rank-exclusive person blocks; every yielded batch is canonical_time contiguous.'''
    def __init__(
        self, dataset: Goal15SequenceDataset, *, batch_size: int, rank: int,
        num_replicas: int, seed: int, block_windows: int = 256,
    ) -> None:
        if batch_size < 1 or block_windows < batch_size:
            raise ValueError('block_windows must be at least batch_size')
        if num_replicas < 1 or rank not in range(num_replicas):
            raise ValueError('invalid block sampler rank')
        self.batch_size = batch_size
        self.seed = seed
        self.epoch = 0
        all_blocks: list[tuple[str, int, range]] = []
        for person, ranges in sorted(dataset.person_ranges().items()):
            block_number = 0
            for person_range in ranges:
                for start in range(person_range.start, person_range.stop, batch_size):
                    block = range(start, min(start + batch_size, person_range.stop))
                    all_blocks.append((person, block_number, block))
                    block_number += 1
        while len(all_blocks) % num_replicas:
            split_index = next((index for index, item in enumerate(all_blocks) if len(item[2]) >= 2), None)
            if split_index is None:
                raise ValueError('exact rank partition cannot balance singleton person blocks')
            person, block_number, block = all_blocks.pop(split_index)
            midpoint = block.start + len(block) // 2
            all_blocks.extend([
                (person, block_number * 2, range(block.start, midpoint)),
                (person, block_number * 2 + 1, range(midpoint, block.stop)),
            ])
        assigned: list[list[tuple[str, int, range]]] = [[] for _ in range(num_replicas)]
        ordered_blocks = sorted(
            all_blocks,
            key=lambda item: hashlib.sha256(f'{item[0]}|{item[1]}|{item[2].start}|{seed}'.encode()).hexdigest(),
        )
        for position, block in enumerate(ordered_blocks):
            owner = position % num_replicas
            assigned[owner].append(block)
        self.blocks = assigned[rank]
        _ = block_windows

    def set_epoch(self, epoch: int) -> None:
        self.epoch = int(epoch)

    def __iter__(self):
        order = np.arange(len(self.blocks))
        np.random.default_rng(self.seed + self.epoch).shuffle(order)
        for position in order:
            _, _, block = self.blocks[int(position)]
            yield list(block)

    def __len__(self) -> int:
        return len(self.blocks)


## 14. Fail-closed physical audit와 ML champion 비교
Task4 원본의 prepared/outcome/registry/personal-baseline 파일을 물리적으로 다시 읽고, timeline 및 300/600 index의 사람 집합을 원본 24/6/6 split과 정확히 비교합니다.

In [ ]:
def _parquet_physical_record(path: Path, category: str) -> dict[str, Any]:
    parquet = pq.ParquetFile(path)
    if parquet.metadata.num_rows < 1:
        raise ValueError(f'{category} physical parquet is empty: {path.name}')
    schema = hashlib.sha256(parquet.schema_arrow.remove_metadata().serialize().to_pybytes()).hexdigest()
    return {
        'category': category, 'path': path.name, 'sha256': sha256_file(path),
        'row_count': int(parquet.metadata.num_rows), 'schema_fingerprint': schema,
    }


def verify_declared_physical_inventory(source_root: Path) -> list[dict[str, Any]]:
    prepared = json.loads((source_root / 'prepared__manifest.json').read_text())
    outcomes = json.loads((source_root / 'outcomes__manifest.json').read_text())
    registry = json.loads((source_root / 'registry__manifest.json').read_text())
    inventory: list[dict[str, Any]] = []
    for person in prepared.get('people', []):
        path = source_root / f'prepared__people__{person["dataset_id"]}.parquet'
        inventory.append(_parquet_physical_record(path, 'prepared'))
    baseline_path = source_root / 'prepared__personal_baseline.parquet'
    baseline = _parquet_physical_record(baseline_path, 'personal_baseline')
    if baseline['sha256'] != _require_sha256(prepared.get('personal_baseline_sha256'), 'personal_baseline'):
        raise ValueError('personal_baseline physical hash mismatch')
    inventory.append(baseline)
    outcome_files = outcomes.get('files')
    if not isinstance(outcome_files, dict) or not outcome_files:
        raise ValueError('outcome manifest files are missing')
    for name, declaration in sorted(outcome_files.items()):
        path = source_root / f'outcomes__{name}'
        record = _parquet_physical_record(path, 'outcome')
        declared_hash = declaration.get('sha256') if isinstance(declaration, dict) else declaration
        if record['sha256'] != _require_sha256(declared_hash, f'outcome {name}'):
            raise ValueError(f'outcome physical hash mismatch: {name}')
        if isinstance(declaration, dict):
            for field in ('row_count', 'schema_fingerprint'):
                if field in declaration and declaration[field] != record[field]:
                    raise ValueError(f'outcome physical {field} mismatch: {name}')
        inventory.append(record)
    split_path = source_root / 'registry__splits.parquet'
    split_record = _parquet_physical_record(split_path, 'registry')
    if split_record['sha256'] != _require_sha256(registry.get('splits_sha256'), 'registry splits'):
        raise ValueError('registry split physical hash mismatch')
    inventory.append(split_record)
    records_path = source_root / 'registry__registry.jsonl'
    records_hash = sha256_file(records_path)
    if records_hash != _require_sha256(registry.get('records_sha256'), 'registry records'):
        raise ValueError('registry records physical hash mismatch')
    record_count = 0
    schema_keys: set[str] | None = None
    with records_path.open() as handle:
        for line in handle:
            payload = json.loads(line)
            keys = set(payload)
            if schema_keys is None:
                schema_keys = keys
            elif keys != schema_keys:
                raise ValueError('registry jsonl schema is inconsistent')
            record_count += 1
    if record_count < 1:
        raise ValueError('registry jsonl is empty')
    inventory.append({
        'category': 'registry', 'path': records_path.name, 'sha256': records_hash,
        'row_count': record_count,
        'schema_fingerprint': hashlib.sha256(json.dumps(sorted(schema_keys or [])).encode()).hexdigest(),
    })
    return inventory


def _physical_person_set(path: Path) -> set[str]:
    parquet = pq.ParquetFile(path)
    people: set[str] = set()
    for row_group in range(parquet.num_row_groups):
        people.update(str(value) for value in parquet.read_row_group(row_group, columns=['person_key']).column('person_key').to_pylist())
    if not people or '' in people:
        raise ValueError(f'physical person set is empty/invalid: {path.name}')
    return people


def verify_exact_sequence_person_sets(
    source_root: Path, timeline_paths: Mapping[str, Path],
    index_paths: Mapping[str, Path],
) -> None:
    assignments, _ = _physical_split_assignments(source_root)
    expected = {
        role: {person for person, assigned in assignments.items() if assigned == role}
        for role in EXPECTED_SPLIT_COUNTS
    }
    if {role: len(people) for role, people in expected.items()} != EXPECTED_SPLIT_COUNTS:
        raise ValueError('physical split is not exact 24/6/6')
    for role, path in timeline_paths.items():
        if _physical_person_set(path) != expected[role]:
            raise ValueError(f'timeline exact person set mismatch: {role}')
    required_indexes = {
        'train_300', 'train_600', 'validation_300', 'validation_600',
        'locked_test_300', 'locked_test_600',
    }
    if set(index_paths) != required_indexes:
        raise ValueError('all six 300/600 sequence index files are required')
    for key, path in index_paths.items():
        role = key.rsplit('_', 1)[0]
        if _physical_person_set(path) != expected[role]:
            raise ValueError(f'sequence exact person set mismatch: {key}')


def discover_ml_champion_artifact(
    *, source_dataset_hash: str, split_hash: str,
    root: Path = ATTACHED_INPUT_ROOT,
) -> tuple[Path, Path, dict[str, Any]]:
    label_hash, feature_hash = _metric_identity_hashes()
    matches: list[tuple[Path, Path, dict[str, Any]]] = []
    for manifest_path in _manifest_candidates(root, 'validation_champion_metrics.manifest.json'):
        manifest = json.loads(manifest_path.read_text())
        expected = {
            'source_dataset_hash': source_dataset_hash, 'split_hash': split_hash,
            'label_schema_hash': label_hash, 'feature_schema_hash': feature_hash,
            'split_role': 'validation', 'target': PATTERN_TARGET,
        }
        if any(manifest.get(key) != value for key, value in expected.items()):
            continue
        file_name = manifest.get('file')
        if not isinstance(file_name, str) or Path(file_name).name != file_name:
            continue
        metrics_path = manifest_path.parent / file_name
        if not metrics_path.is_file() or sha256_file(metrics_path) != manifest.get('file_sha256'):
            continue
        frame = pd.read_parquet(metrics_path)
        if frame.empty or not frame['split_role'].eq('validation').all() or 'locked_test' in set(frame['split_role']):
            continue
        if frame.duplicated(['target', 'metric']).any():
            continue
        if frame['model_name'].nunique() != 1 or frame['model_family'].nunique() != 1:
            continue
        if frame['model_name'].iloc[0] != manifest.get('model_name') or frame['model_family'].iloc[0] != manifest.get('model_family'):
            continue
        threshold = manifest.get('threshold')
        if not isinstance(threshold, (int, float)) or not 0 <= float(threshold) <= 1:
            continue
        matches.append((metrics_path, manifest_path, manifest))
    if len(matches) != 1:
        raise ValueError(f'exactly one physical ML validation champion artifact is required: {len(matches)}')
    return matches[0]


def write_metrics_identity_manifest(
    metrics_path: Path, *, source_dataset_hash: str, split_hash: str, split_role: str,
) -> Path:
    label_schema_hash, feature_schema_hash = _metric_identity_hashes()
    path = metrics_path.with_suffix('.manifest.json')
    path.write_text(json.dumps({
        'source_dataset_hash': source_dataset_hash, 'split_hash': split_hash,
        'label_schema_hash': label_schema_hash, 'feature_schema_hash': feature_schema_hash,
        'split_role': split_role, 'locked_test_included': split_role == 'locked_test',
        'file_sha256': sha256_file(metrics_path), 'file': metrics_path.name,
    }, indent=2, sort_keys=True) + '\n')
    return path


def write_validation_model_comparison(
    ml_metrics_path: Path, dl_metrics_path: Path,
    ml_manifest_path: Path, dl_manifest_path: Path,
) -> tuple[Path, str]:
    ml_manifest = json.loads(ml_manifest_path.read_text())
    dl_manifest = json.loads(dl_manifest_path.read_text())
    identity = ('source_dataset_hash', 'split_hash', 'label_schema_hash', 'feature_schema_hash')
    if any(ml_manifest.get(key) != dl_manifest.get(key) for key in identity):
        raise ValueError('ML/DL comparison identity mismatch')
    if sha256_file(ml_metrics_path) != ml_manifest.get('file_sha256') or sha256_file(dl_metrics_path) != dl_manifest.get('file_sha256'):
        raise ValueError('ML/DL comparison physical file hash mismatch')
    ml = pd.read_parquet(ml_metrics_path)
    dl = pd.read_parquet(dl_metrics_path)
    keys = ['target', 'metric']
    for frame in (ml, dl):
        if frame.empty or not frame['split_role'].eq('validation').all() or frame.duplicated(keys).any():
            raise ValueError('comparison requires unique validation-only metric keys; no locked_test')
    common = sorted(set(map(tuple, ml[keys].to_numpy())) & set(map(tuple, dl[keys].to_numpy())))
    if not common:
        raise ValueError('ML/DL comparison has no common-grid metric keys')
    common_frame = pd.DataFrame(common, columns=keys)
    left = common_frame.merge(ml, on=keys, how='left', validate='one_to_one')
    comparison = left.merge(dl, on=keys, how='left', suffixes=('_ml', '_dl'), validate='one_to_one')
    if 'locked_test' in set(comparison.get('split_role_ml', [])) | set(comparison.get('split_role_dl', [])):
        raise ValueError('locked_test is forbidden from validation comparison')
    output_path = DL_BENCHMARK_OUTPUT_ROOT / 'validation_model_comparison.parquet'
    comparison.to_parquet(output_path, index=False)
    return output_path, sha256_file(output_path)


In [ ]:
def evaluate_noise_stress_to_shards(
    model: nn.Module, loader: DataLoader, device: torch.device,
    *, split_role: str, model_name: str, rank: int, output_root: Path,
    source_dataset_hash: str, split_hash: str,
    threshold_artifact: Path, threshold_artifact_hash: str,
) -> dict[str, Path]:
    if split_role != 'validation':
        raise ValueError('noise stress is validation-only')
    threshold = load_fixed_threshold(
        threshold_artifact, threshold_artifact_hash=threshold_artifact_hash,
        source_dataset_hash=source_dataset_hash, split_hash=split_hash,
        model_name=model_name,
    )
    manifests: dict[str, Path] = {}
    for condition in STRESS_CONDITIONS:
        manifests[condition] = evaluate_to_prediction_shard(
            model, loader, device, split_role=split_role, model_name=model_name,
            rank=rank, output_path=output_root / condition / f'rank-{rank}.parquet',
            source_dataset_hash=source_dataset_hash, split_hash=split_hash,
            thresholds={PATTERN_TARGET: threshold}, stress_condition=condition,
        )
    return manifests


def run_dl_training(epochs: int = 20, batch_size: int = 64) -> None:
    require_exactly_two_cuda_devices()
    if 'LOCAL_RANK' not in os.environ:
        launch_dual_t4_torchrun()
        return
    rank, local_rank, world_size = setup_ddp()
    device = torch.device('cuda', local_rank)
    try:
        set_deterministic_seed(SEED, rank)
        verified = verify_sequence_inputs()
        if verified.source_dataset_hash != os.environ.get('GOAL15_EXPECTED_SOURCE_HASH') or verified.split_hash != os.environ.get('GOAL15_EXPECTED_SPLIT_HASH'):
            raise RuntimeError('worker physical identity differs from guarded launcher')
        loss_weights = derive_train_loss_weights(verified.index_paths[f'train_{SEQUENCE_LENGTH_CANDIDATE}'])
        if rank == 0:
            write_train_loss_support(DL_BENCHMARK_OUTPUT_ROOT, loss_weights)
        wandb_module: Any | None = None
        if rank == 0 and login_wandb_from_kaggle_secret():
            import wandb
            wandb_module = wandb
            safe_wandb_call(wandb.init, project=WANDB_PROJECT, group=WANDB_GROUP, tags=WANDB_TAGS, config={
                'epochs': epochs, 'batch_size': batch_size,
                'sequence_length': SEQUENCE_LENGTH_CANDIDATE,
                'sequence_length_policy': SEQUENCE_LENGTH_POLICY,
                'data_status': DATA_STATUS,
            })
        train_dataset = Goal15SequenceDataset(
            verified.index_paths[f'train_{SEQUENCE_LENGTH_CANDIDATE}'],
            verified.timeline_paths['train'], verified.normalization, 'train',
        )
        validation_dataset = Goal15SequenceDataset(
            verified.index_paths[f'validation_{SEQUENCE_LENGTH_CANDIDATE}'],
            verified.timeline_paths['validation'], verified.normalization, 'validation',
        )
        train_batch_sampler = PersonBlockBatchSampler(
            train_dataset, batch_size=batch_size, rank=rank,
            num_replicas=world_size, seed=SEED,
        )
        validation_sampler = PersonShardEvalSampler(validation_dataset, rank=rank, num_replicas=world_size)
        train_loader = DataLoader(train_dataset, batch_sampler=train_batch_sampler, num_workers=2, pin_memory=True)
        validation_loader = DataLoader(validation_dataset, batch_size=batch_size, sampler=validation_sampler, num_workers=2, pin_memory=True)
        model = Goal15TCN(input_size=len(ALLOWED_FEATURE_COLUMNS)).to(device)
        parameter_count = assert_parameter_budget(model)
        assert_right_padding_invariance(model, len(ALLOWED_FEATURE_COLUMNS))
        model = DistributedDataParallel(model, device_ids=[local_rank], output_device=local_rank)
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
        scaler = torch.amp.GradScaler('cuda')
        for epoch in range(epochs):
            losses = train_one_epoch(
                model, train_loader, optimizer, scaler, device,
                epoch=epoch, sampler=train_batch_sampler, loss_weights=loss_weights,
            )
            if rank == 0 and wandb_module is not None:
                safe_wandb_call(wandb_module.log, {f'train/{name}': value for name, value in losses.items()}, step=epoch)
        validation_root = DL_BENCHMARK_OUTPUT_ROOT / 'validation' / 'clean'
        local_manifest = evaluate_to_prediction_shard(
            model, validation_loader, device, split_role='validation', model_name='causal_tcn',
            rank=rank, output_path=validation_root / f'rank-{rank}.parquet',
            source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
        )
        manifest_paths = _collect_small_manifests(local_manifest, rank, world_size)
        threshold_payload: list[Any] = [None, None, None]
        validation_metrics: pd.DataFrame | None = None
        if rank == 0:
            validation_metrics, selected_threshold, confusion = aggregate_prediction_shards(
                manifest_paths, expected_window_count=len(validation_dataset),
                source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
                model_name='causal_tcn',
                expected_person_windows=validation_dataset.expected_windows_for_person,
                threshold_mode='select_clean_validation', stress_condition='clean',
            )
            DL_BENCHMARK_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
            metrics_path = DL_BENCHMARK_OUTPUT_ROOT / 'validation_metrics.parquet'
            assert_unique_metric_rows(validation_metrics)
            validation_metrics.to_parquet(metrics_path, index=False)
            confusion.to_parquet(DL_BENCHMARK_OUTPUT_ROOT / 'validation_stage_confusion.parquet', index=False)
            metrics_manifest = write_metrics_identity_manifest(
                metrics_path, source_dataset_hash=verified.source_dataset_hash,
                split_hash=verified.split_hash, split_role='validation',
            )
            threshold_artifact, threshold_artifact_hash = write_clean_validation_threshold(
                DL_BENCHMARK_OUTPUT_ROOT, threshold=selected_threshold,
                source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
                model_name='causal_tcn',
            )
            threshold_payload = [str(threshold_artifact), threshold_artifact_hash, selected_threshold]
            export_model_state(model, DL_BENCHMARK_OUTPUT_ROOT)
            if RUN_VALIDATION_COMPARISON:
                ml_metrics, ml_manifest, _ = discover_ml_champion_artifact(
                    source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
                )
                write_validation_model_comparison(ml_metrics, metrics_path, ml_manifest, metrics_manifest)
        dist.broadcast_object_list(threshold_payload, src=0)
        threshold_artifact = Path(str(threshold_payload[0]))
        threshold_artifact_hash = str(threshold_payload[1])
        selected_threshold = float(threshold_payload[2])
        if RUN_NOISE_STRESS:
            stress_local = evaluate_noise_stress_to_shards(
                model, validation_loader, device, split_role='validation', model_name='causal_tcn',
                rank=rank, output_root=DL_BENCHMARK_OUTPUT_ROOT / 'validation' / 'stress',
                source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
                threshold_artifact=threshold_artifact,
                threshold_artifact_hash=threshold_artifact_hash,
            )
            for condition, stress_manifest in stress_local.items():
                stress_paths = _collect_small_manifests(stress_manifest, rank, world_size)
                if rank == 0:
                    stress_metrics, _, _ = aggregate_prediction_shards(
                        stress_paths, expected_window_count=len(validation_dataset),
                        source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
                        model_name='causal_tcn',
                        expected_person_windows=validation_dataset.expected_windows_for_person,
                        threshold_mode='fixed_threshold', threshold_artifact=threshold_artifact,
                        threshold_artifact_hash=threshold_artifact_hash,
                        stress_condition=condition,
                    )
                    assert_unique_metric_rows(stress_metrics)
                    stress_metrics.to_parquet(DL_BENCHMARK_OUTPUT_ROOT / f'validation_noise_{condition}.parquet', index=False)
        if RUN_LOCKED_TEST:
            locked_dataset = Goal15SequenceDataset(
                verified.index_paths[f'locked_test_{SEQUENCE_LENGTH_CANDIDATE}'],
                verified.timeline_paths['locked_test'], verified.normalization, 'locked_test',
            )
            locked_sampler = PersonShardEvalSampler(locked_dataset, rank=rank, num_replicas=world_size)
            locked_loader = DataLoader(locked_dataset, batch_size=batch_size, sampler=locked_sampler, num_workers=2, pin_memory=True)
            locked_manifest = evaluate_to_prediction_shard(
                model, locked_loader, device, split_role='locked_test', model_name='causal_tcn',
                rank=rank, output_path=DL_BENCHMARK_OUTPUT_ROOT / 'locked_test' / f'rank-{rank}.parquet',
                source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
                thresholds={PATTERN_TARGET: selected_threshold},
            )
            locked_paths = _collect_small_manifests(locked_manifest, rank, world_size)
            if rank == 0:
                locked_metrics, _, locked_confusion = aggregate_prediction_shards(
                    locked_paths, expected_window_count=len(locked_dataset),
                    source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
                    model_name='causal_tcn', expected_person_windows=locked_dataset.expected_windows_for_person,
                    threshold_mode='fixed_threshold', threshold_artifact=threshold_artifact,
                    threshold_artifact_hash=threshold_artifact_hash, stress_condition='clean',
                )
                locked_path = DL_BENCHMARK_OUTPUT_ROOT / 'locked_test_metrics.parquet'
                assert_unique_metric_rows(locked_metrics)
                locked_metrics.to_parquet(locked_path, index=False)
                locked_confusion.to_parquet(DL_BENCHMARK_OUTPUT_ROOT / 'locked_test_stage_confusion.parquet', index=False)
                write_metrics_identity_manifest(
                    locked_path, source_dataset_hash=verified.source_dataset_hash,
                    split_hash=verified.split_hash, split_role='locked_test',
                )
        if rank == 0 and wandb_module is not None:
            safe_wandb_call(wandb_module.log, {'model/parameter_count': parameter_count})
            safe_wandb_call(wandb_module.finish)
    finally:
        cleanup_ddp()


if RUN_TRAINING:
    print('Round4 bounded lockstep 정의를 계속 로드합니다.')
else:
    print('학습 비활성화: Round3 runner/GPU/W&B/locked-test를 실행하지 않았습니다.')


## 15. Round4 true bounded window lockstep
Prediction과 600초 expected index를 window 단위로 병합 순회합니다. Carry는 불완전 window 하나(최대 16행), event state는 101개 고정 threshold 배열과 사람별 compact sufficient statistics만 유지합니다.

In [ ]:
def iter_window_major_prediction_groups(
    shard_path, *, batch_rows: int = 8192, max_window_rows: int = 16,
):
    parquet = pq.ParquetFile(shard_path)
    current_key = None
    carry: list[dict[str, Any]] = []
    previous_key = None
    for batch in parquet.iter_batches(batch_size=batch_rows):
        for row in batch.to_pylist():
            key = (
                str(row['person_key']), str(row['run_id']), str(row['dataset_id']),
                pd.Timestamp(row['canonical_time']).value, str(row['window_id']),
            )
            if previous_key is not None and key < previous_key:
                raise ValueError('prediction shard is not canonical window-major ordered')
            if current_key is not None and key != current_key:
                yield pd.DataFrame.from_records(carry)
                carry = []
            current_key = key
            previous_key = key
            carry.append(row)
            if len(carry) > max_window_rows:
                raise ValueError('prediction carry exceeds one incomplete window group')
    if carry:
        yield pd.DataFrame.from_records(carry)


def iter_expected_sequence_windows(
    index_path: Path, *, batch_rows: int = 8192,
):
    columns = list(SEQUENCE_INDEX_COLUMNS)
    parquet = pq.ParquetFile(index_path)
    previous_key = None
    for batch in parquet.iter_batches(batch_size=batch_rows, columns=columns):
        for row in batch.to_pylist():
            row['canonical_time'] = pd.Timestamp(row['prediction_time']).isoformat()
            key = (
                str(row['person_key']), str(row['run_id']), str(row['dataset_id']),
                pd.Timestamp(row['canonical_time']).value, str(row['window_id']),
            )
            if previous_key is not None and key <= previous_key:
                raise ValueError('expected sequence index is not canonical/unique')
            previous_key = key
            yield row


def _window_group_key(group: pd.DataFrame) -> tuple[str, str, str, int, str]:
    if group.empty:
        raise ValueError('window group is empty')
    return (
        str(group['person_key'].iloc[0]), str(group['run_id'].iloc[0]),
        str(group['dataset_id'].iloc[0]),
        pd.Timestamp(group['canonical_time'].iloc[0]).value,
        str(group['window_id'].iloc[0]),
    )


def merge_window_major_shards(shard_paths: Sequence[Path]):
    import heapq

    iterators = [iter(iter_window_major_prediction_groups(path)) for path in shard_paths]
    heap: list[tuple[tuple[str, str, str, int, str], int, pd.DataFrame]] = []
    for index, iterator in enumerate(iterators):
        group = next(iterator, None)
        if group is not None:
            heapq.heappush(heap, (_window_group_key(group), index, group))
    previous = None
    while heap:
        key, index, group = heapq.heappop(heap)
        if previous is not None and key <= previous:
            raise ValueError('merged prediction shards overlap or are unordered')
        previous = key
        yield group
        following = next(iterators[index], None)
        if following is not None:
            heapq.heappush(heap, (_window_group_key(following), index, following))


class StreamingEventGridState:
    def __init__(self, *, bins: int = 101) -> None:
        if bins < 2:
            raise ValueError('event grid requires at least two bins')
        self.bins = bins
        self.grid = np.linspace(0.0, 1.0, bins)
        self.detected = np.zeros(bins, dtype=np.int64)
        self.false_alerts = np.zeros(bins, dtype=np.int64)
        self.truth_events = 0
        self.truth_active = False
        self.truth_event_max_bucket = -1
        self.alert_active = np.zeros(bins, dtype=bool)
        self.alert_truth_overlap = np.zeros(bins, dtype=bool)

    def _close_truth_event(self) -> None:
        if not self.truth_active:
            return
        self.truth_events += 1
        self.detected[: self.truth_event_max_bucket + 1] += 1
        self.truth_active = False
        self.truth_event_max_bucket = -1

    def update_chunk(self, truth: np.ndarray, probability: np.ndarray) -> None:
        labels = np.asarray(truth, dtype=np.int8).astype(bool)
        scores = np.asarray(probability, dtype=np.float64)
        if len(labels) != len(scores) or ((scores < 0) | (scores > 1)).any():
            raise ValueError('event chunk labels/probabilities are invalid')
        buckets = np.minimum((scores * (self.bins - 1)).astype(np.int64), self.bins - 1)
        for label, score, bucket in zip(labels, scores, buckets, strict=True):
            if label:
                if not self.truth_active:
                    self.truth_active = True
                    self.truth_event_max_bucket = int(bucket)
                else:
                    self.truth_event_max_bucket = max(self.truth_event_max_bucket, int(bucket))
            else:
                self._close_truth_event()
            active = score >= self.grid
            ending = self.alert_active & ~active
            self.false_alerts += ending & ~self.alert_truth_overlap
            starting = active & ~self.alert_active
            self.alert_truth_overlap[starting] = bool(label)
            continuing = active & self.alert_active
            if label:
                self.alert_truth_overlap[continuing] = True
            self.alert_truth_overlap[~active] = False
            self.alert_active = active

    def finalize_person(self, *, duration_hours: float) -> dict[str, Any]:
        self._close_truth_event()
        self.false_alerts += self.alert_active & ~self.alert_truth_overlap
        self.alert_active[:] = False
        self.alert_truth_overlap[:] = False
        return {
            'detected': self.detected.copy(),
            'false_alerts': self.false_alerts.copy(),
            'truth_events': int(self.truth_events),
            'duration_hours': float(duration_hours),
        }


def validate_window_group_semantics(group: pd.DataFrame, expected: Mapping[str, Any]) -> None:
    identity = ('person_key', 'run_id', 'dataset_id', 'window_id', 'split_role')
    if any(group[field].nunique(dropna=False) != 1 or str(group[field].iloc[0]) != str(expected[field]) for field in identity):
        raise ValueError('window group exact identity mismatch')
    if pd.Timestamp(group['canonical_time'].iloc[0]) != pd.Timestamp(expected['prediction_time']):
        raise ValueError('window group endpoint time mismatch')
    if group.duplicated('target').any():
        raise ValueError('window target is duplicated')
    pattern = group.loc[group['target'].eq(PATTERN_TARGET)]
    if len(pattern) != 1 or int(pattern['label'].iloc[0]) != int(expected[PATTERN_TARGET]):
        raise ValueError('window requires exactly one pattern target')
    if int(pattern['audit_event_binary'].iloc[0]) != int(expected[ONSET_EVENT_TARGET]):
        raise ValueError('window audit event mismatch')
    expected_targets = {PATTERN_TARGET}
    if int(expected[PATTERN_TARGET]) == 1:
        expected_targets.update(f'stage::{code}' for code in STAGE_CODES)
    behavior_positive = any(int(expected[code]) == 1 for code in BEHAVIOR_CODES)
    decision_mask = int(expected[PATTERN_TARGET]) == 1 or (int(expected['hard_negative']) == 1 and behavior_positive)
    if decision_mask:
        expected_targets.update(f'behavior::{code}' for code in BEHAVIOR_CODES)
    if set(group['target']) != expected_targets:
        raise ValueError('window conditional target set is missing or has extras')
    if int(expected[PATTERN_TARGET]) == 1:
        stage_truth = group.loc[group['target'].str.startswith('stage::') & group['label'].eq(1), 'target'].tolist()
        if stage_truth != [f'stage::{expected[STAGE_TARGET]}']:
            raise ValueError('window stage truth mismatch')
    if decision_mask:
        for code in BEHAVIOR_CODES:
            row = group.loc[group['target'].eq(f'behavior::{code}')]
            if len(row) != 1 or int(row['label'].iloc[0]) != int(expected[code]):
                raise ValueError(f'window behavior truth mismatch: {code}')


In [ ]:
class StreamingEvaluationState:
    def __init__(self, *, bins: int = COMMON_THRESHOLD_GRID_SIZE) -> None:
        self.bins = bins
        self.accumulator = StreamingMetricAccumulator(bins=bins)
        self.person_sufficient_statistics: list[dict[str, Any]] = []
        self.current_person: str | None = None
        self.person_event = StreamingEventGridState(bins=bins)
        self.person_pattern = {
            'positive': np.zeros(bins), 'negative': np.zeros(bins),
            'probability_sum': np.zeros(bins), 'brier_sum': 0.0,
        }
        self.person_first_time: pd.Timestamp | None = None
        self.person_last_time: pd.Timestamp | None = None
        self.stage_confusion = np.zeros((len(STAGE_CODES), len(STAGE_CODES)), dtype=np.int64)
        self.window_count = 0

    def _finish_person(self) -> None:
        if self.current_person is None:
            return
        duration = 1 / 3600
        if self.person_first_time is not None and self.person_last_time is not None:
            duration = max((self.person_last_time - self.person_first_time).total_seconds() / 3600, 1 / 3600)
        self.person_sufficient_statistics.append({
            'person_key': self.current_person,
            'histograms': {PATTERN_TARGET: {
                name: value.copy() if isinstance(value, np.ndarray) else float(value)
                for name, value in self.person_pattern.items()
            }},
            'event': self.person_event.finalize_person(duration_hours=duration),
        })
        self.person_event = StreamingEventGridState(bins=self.bins)
        self.person_pattern = {
            'positive': np.zeros(self.bins), 'negative': np.zeros(self.bins),
            'probability_sum': np.zeros(self.bins), 'brier_sum': 0.0,
        }
        self.person_first_time = self.person_last_time = None

    def update_window(self, group: pd.DataFrame) -> None:
        person = str(group['person_key'].iloc[0])
        if self.current_person is not None and person != self.current_person:
            if person <= self.current_person:
                raise ValueError('streaming people are not canonical unique')
            self._finish_person()
        self.current_person = person
        timestamp = pd.Timestamp(group['canonical_time'].iloc[0])
        self.person_first_time = timestamp if self.person_first_time is None else self.person_first_time
        self.person_last_time = timestamp
        self.accumulator.update_histograms(group)
        pattern = group.loc[group['target'].eq(PATTERN_TARGET)].iloc[0]
        label = int(pattern['label'])
        probability = float(pattern['probability'])
        bucket = min(int(probability * (self.bins - 1)), self.bins - 1)
        name = 'positive' if label else 'negative'
        self.person_pattern[name][bucket] += 1
        self.person_pattern['probability_sum'][bucket] += probability
        self.person_pattern['brier_sum'] += (probability - label) ** 2
        self.person_event.update_chunk(np.asarray([label]), np.asarray([probability]))
        stage = group.loc[group['target'].str.startswith('stage::')]
        if not stage.empty:
            actual = str(stage.loc[stage['label'].eq(1), 'target'].iloc[0]).removeprefix('stage::')
            predicted = str(stage.loc[stage['probability'].idxmax(), 'target']).removeprefix('stage::')
            self.stage_confusion[STAGE_CODES.index(actual), STAGE_CODES.index(predicted)] += 1
        self.window_count += 1

    def finish(self) -> None:
        self._finish_person()
        self.current_person = None


def stream_lockstep_evaluation(
    shard_paths: Sequence[Path], index_path: Path,
) -> tuple[StreamingEvaluationState, str]:
    predictions = iter(merge_window_major_shards(shard_paths))
    expected_rows = iter(iter_expected_sequence_windows(index_path))
    state = StreamingEvaluationState()
    coverage = hashlib.sha256()
    while True:
        group = next(predictions, None)
        expected = next(expected_rows, None)
        if group is None or expected is None:
            if group is not None or expected is not None:
                raise ValueError('prediction/expected endpoint coverage length mismatch')
            break
        validate_window_group_semantics(group, expected)
        identity = '|'.join([
            str(expected['person_key']), str(expected['run_id']), str(expected['dataset_id']),
            pd.Timestamp(expected['prediction_time']).isoformat(), str(expected['window_id']),
        ])
        coverage.update((identity + '\n').encode())
        state.update_window(group)
    state.finish()
    return state, coverage.hexdigest()


def finalize_streaming_metrics(
    state: StreamingEvaluationState, *, model_name: str, split_role: str,
    threshold_mode: str, stress_condition: str,
    source_dataset_hash: str, split_hash: str,
    threshold_artifact: Path | None = None,
    threshold_artifact_hash: str | None = None,
) -> tuple[pd.DataFrame, float]:
    if threshold_mode == 'select_clean_validation':
        if split_role != 'validation' or stress_condition != 'clean':
            raise ValueError('clean threshold selection is validation clean only')
        threshold, _ = select_clean_validation_threshold(state.person_sufficient_statistics)
    elif threshold_mode == 'fixed_threshold':
        if threshold_artifact is None or threshold_artifact_hash is None:
            raise ValueError('fixed_threshold needs threshold identity')
        threshold = load_fixed_threshold(
            threshold_artifact, threshold_artifact_hash=threshold_artifact_hash,
            source_dataset_hash=source_dataset_hash, split_hash=split_hash,
            model_name=model_name,
        )
    else:
        raise ValueError('unknown threshold mode')
    rows: list[dict[str, Any]] = []
    behavior_code_metrics: dict[str, dict[str, float]] = {}
    for target, histogram in sorted(state.accumulator.histograms.items()):
        target_threshold = threshold if target == PATTERN_TARGET else 0.5
        values = state.accumulator.global_binary_metrics(target, target_threshold)
        support = int(histogram['positive'].sum() + histogram['negative'].sum())
        for metric, value in values.items():
            output_metric = metric
            if target.startswith('behavior::'):
                output_metric = {
                    'global_aucpr': 'behavior_code_aucpr',
                    'global_auroc': 'behavior_code_auroc',
                    'global_row_f1': 'behavior_code_f1',
                }.get(metric, metric)
                behavior_code_metrics.setdefault(target, {})[output_metric] = value
            rows.append(_round3_metric_row(model_name, split_role, target, output_metric, value, support, stress_condition))
        if target.startswith('behavior::'):
            rows.append(_round3_metric_row(
                model_name, split_role, target, 'behavior_positive_support',
                float(histogram['positive'].sum()), support, stress_condition,
            ))
    missing_behaviors = [f'behavior::{code}' for code in BEHAVIOR_CODES if f'behavior::{code}' not in behavior_code_metrics]
    if missing_behaviors:
        raise ValueError(f'behavior code metric support missing: {missing_behaviors}')
    for source_metric, output_metric in (
        ('behavior_code_aucpr', 'behavior_macro_aucpr'),
        ('behavior_code_auroc', 'behavior_macro_auroc'),
        ('behavior_code_f1', 'behavior_macro_f1'),
    ):
        values = [behavior_code_metrics[f'behavior::{code}'][source_metric] for code in BEHAVIOR_CODES]
        rows.append(_round3_metric_row(
            model_name, split_role, 'behavior::all', output_metric,
            float(np.nanmean(values)), len(BEHAVIOR_CODES), stress_condition,
        ))
    combined = {
        'positive': np.zeros(COMMON_THRESHOLD_GRID_SIZE),
        'negative': np.zeros(COMMON_THRESHOLD_GRID_SIZE),
        'probability_sum': np.zeros(COMMON_THRESHOLD_GRID_SIZE), 'brier_sum': 0.0,
    }
    for code in BEHAVIOR_CODES:
        histogram = state.accumulator.histograms[f'behavior::{code}']
        for name in ('positive', 'negative', 'probability_sum'):
            combined[name] += histogram[name]
        combined['brier_sum'] += histogram['brier_sum']
    micro = _histogram_binary_metrics(
        combined['positive'], combined['negative'], combined['probability_sum'],
        float(combined['brier_sum']), (COMMON_THRESHOLD_GRID_SIZE - 1) // 2,
    )
    for source, output in (
        ('global_aucpr', 'behavior_micro_aucpr'),
        ('global_auroc', 'behavior_micro_auroc'),
        ('global_row_f1', 'behavior_micro_f1'),
    ):
        rows.append(_round3_metric_row(
            model_name, split_role, 'behavior::all', output, micro[source],
            int(combined['positive'].sum() + combined['negative'].sum()), stress_condition,
        ))
    event_values = _event_metrics_from_statistics(state.person_sufficient_statistics, threshold)
    for metric, value in event_values.items():
        rows.append(_round3_metric_row(model_name, split_role, PATTERN_TARGET, metric, value, state.window_count, stress_condition))
    threshold_index = int(round(threshold * (COMMON_THRESHOLD_GRID_SIZE - 1)))
    person_f1 = []
    for item in state.person_sufficient_statistics:
        histogram = item['histograms'][PATTERN_TARGET]
        person_f1.append(_histogram_binary_metrics(
            histogram['positive'], histogram['negative'], histogram['probability_sum'],
            float(histogram['brier_sum']), threshold_index,
        )['global_row_f1'])
    rows.append(_round3_metric_row(model_name, split_role, PATTERN_TARGET, 'person_macro_f1', float(np.mean(person_f1)), len(person_f1), stress_condition))
    for metric, value in bootstrap_person_metrics(state.person_sufficient_statistics, threshold=threshold).items():
        rows.append(_round3_metric_row(model_name, split_role, PATTERN_TARGET, metric, value, len(person_f1), stress_condition))
    confusion = state.stage_confusion
    recalls = np.divide(np.diag(confusion), confusion.sum(axis=1), out=np.zeros(len(STAGE_CODES), dtype=float), where=confusion.sum(axis=1) > 0)
    precision = np.divide(np.diag(confusion), confusion.sum(axis=0), out=np.zeros(len(STAGE_CODES), dtype=float), where=confusion.sum(axis=0) > 0)
    f1 = np.divide(2 * precision * recalls, precision + recalls, out=np.zeros(len(STAGE_CODES)), where=(precision + recalls) > 0)
    rows.append(_round3_metric_row(model_name, split_role, 'stage::all', 'stage_macro_f1', float(np.mean(f1)), int(confusion.sum()), stress_condition))
    rows.append(_round3_metric_row(model_name, split_role, 'stage::all', 'stage_balanced_accuracy', float(np.mean(recalls)), int(confusion.sum()), stress_condition))
    metrics = pd.DataFrame(rows, columns=ROUND3_METRIC_COLUMNS)
    assert_unique_metric_rows(metrics)
    return metrics, threshold


In [ ]:
def latent_factor_feature_indices(
    factor: str, columns: Sequence[str],
) -> list[int]:
    prefix = f'{factor}__'
    return [index for index, column in enumerate(columns) if column.startswith(prefix)]


def deterministic_stress_batch(batch: Mapping[str, Any], condition: str) -> dict[str, Any]:
    if condition not in STRESS_CONDITIONS:
        raise ValueError(f'unknown Goal 1.5 stress condition: {condition}')
    stressed = dict(batch)
    features = batch['features'].clone()
    mask = batch['mask'].clone()
    for item, window_id in enumerate(batch['window_id']):
        seed = int(hashlib.sha256(f'{window_id}|{condition}'.encode()).hexdigest()[:16], 16)
        generator = torch.Generator(device=features.device).manual_seed(seed)
        if condition.startswith('gaussian_'):
            scale = float(condition.rsplit('_', 1)[1])
            features[item] += torch.randn(
                features[item].shape, generator=generator, device=features.device,
                dtype=features.dtype,
            ) * scale
        elif condition.startswith('block_missing_'):
            length = min(int(condition.rsplit('_', 1)[1]), int(mask[item].sum()))
            start = seed % max(int(mask[item].sum()) - length + 1, 1)
            features[item, start:start + length] = 0
            mask[item, start:start + length] = False
        elif condition.startswith('time_shift_'):
            shift = int(condition.removeprefix('time_shift_'))
            original = features[item].clone()
            features[item].zero_()
            if shift > 0:
                features[item, shift:] = original[:-shift]
                invalid_edge = slice(0, shift)
            else:
                amount = abs(shift)
                features[item, :-amount] = original[amount:]
                invalid_edge = slice(-amount, None)
            mask[item, invalid_edge] = False
        else:
            factor = CAUSAL_FACTORS[seed % len(CAUSAL_FACTORS)]
            indices = latent_factor_feature_indices(factor, ALLOWED_FEATURE_COLUMNS)
            if not indices:
                raise ValueError(f'latent factor has no derived features: {factor}')
            features[item, :, indices] = 0
    stressed['features'] = features
    stressed['mask'] = mask
    return stressed


In [ ]:
class PersonBlockBatchSampler(Sampler[list[int]]):
    '''Assign each person to one rank; preserve canonical_time contiguous blocks.'''
    def __init__(
        self, dataset: Goal15SequenceDataset, *, batch_size: int, rank: int,
        num_replicas: int, seed: int, block_windows: int = 256,
    ) -> None:
        if batch_size < 1 or block_windows < batch_size:
            raise ValueError('block_windows must be at least batch_size')
        if num_replicas < 1 or rank not in range(num_replicas):
            raise ValueError('invalid block sampler rank')
        self.seed = seed
        self.epoch = 0
        person_blocks: dict[str, list[tuple[str, int, range]]] = {}
        for person, ranges in sorted(dataset.person_ranges().items()):
            blocks: list[tuple[str, int, range]] = []
            block_number = 0
            for person_range in ranges:
                for start in range(person_range.start, person_range.stop, block_windows):
                    blocks.append((
                        person, block_number,
                        range(start, min(start + block_windows, person_range.stop)),
                    ))
                    block_number += 1
            person_blocks[person] = blocks
        rank_people: list[list[str]] = [[] for _ in range(num_replicas)]
        rank_windows = [0] * num_replicas
        for person in sorted(person_blocks, key=lambda key: (-sum(len(block) for _, _, block in person_blocks[key]), key)):
            owner = min(range(num_replicas), key=lambda candidate: (rank_windows[candidate], candidate))
            rank_people[owner].append(person)
            rank_windows[owner] += sum(len(block) for _, _, block in person_blocks[person])
        rank_blocks: list[list[tuple[str, int, list[range]]]] = [[] for _ in range(num_replicas)]
        for owner, people in enumerate(rank_people):
            for person in people:
                for block_person, block_number, block in person_blocks[person]:
                    batches_per_block = [
                        range(start, min(start + batch_size, block.stop))
                        for start in range(block.start, block.stop, batch_size)
                    ]
                    rank_blocks[owner].append((block_person, block_number, batches_per_block))
        batch_counts = [sum(len(batches) for _, _, batches in blocks) for blocks in rank_blocks]
        while min(batch_counts) != max(batch_counts):
            owner = min(range(num_replicas), key=lambda candidate: (batch_counts[candidate], candidate))
            split_done = False
            for block_index, (person, block_number, batches) in enumerate(rank_blocks[owner]):
                batch_index = next((index for index, batch in enumerate(batches) if len(batch) >= 2), None)
                if batch_index is None:
                    continue
                batch = batches[batch_index]
                midpoint = batch.start + len(batch) // 2
                replacement = [*batches[:batch_index], range(batch.start, midpoint), range(midpoint, batch.stop), *batches[batch_index + 1:]]
                rank_blocks[owner][block_index] = (person, block_number, replacement)
                batch_counts[owner] += 1
                split_done = True
                break
            if not split_done:
                raise ValueError(f'cannot balance exact rank-person DDP steps: {batch_counts}')
        self.blocks = rank_blocks[rank]

    def set_epoch(self, epoch: int) -> None:
        self.epoch = int(epoch)

    def __iter__(self):
        order = np.arange(len(self.blocks))
        np.random.default_rng(self.seed + self.epoch).shuffle(order)
        for position in order:
            _, _, batches_per_block = self.blocks[int(position)]
            for batch in batches_per_block:
                yield list(batch)

    def __len__(self) -> int:
        return sum(len(batches_per_block) for _, _, batches_per_block in self.blocks)


## 16. DDP probability production과 rank0 CPU postprocess 분리
두 rank는 threshold 없는 clean/stress/optional locked probability shard와 done marker를 모두 기록한 뒤 process group을 종료합니다. 이후 rank0만 bounded lockstep CPU 집계와 export를 수행합니다.

In [ ]:
import time


def write_rank_done_marker(
    output_root: Path, *, rank: int, source_dataset_hash: str,
    split_hash: str, probability_manifests: Mapping[str, Any],
) -> Path:
    marker_root = output_root / 'rank_done'
    marker_root.mkdir(parents=True, exist_ok=True)
    path = marker_root / f'rank-{rank}.json'
    temporary = marker_root / f'rank-{rank}.tmp'
    temporary.write_text(json.dumps({
        'schema_version': 'goal1.5/dl-rank-probability-done/v1',
        'rank': rank, 'source_dataset_hash': source_dataset_hash,
        'split_hash': split_hash, 'probability_manifests': probability_manifests,
    }, indent=2, sort_keys=True) + '\n')
    temporary.replace(path)
    return path


def wait_for_rank_done_markers(
    output_root: Path, *, world_size: int, source_dataset_hash: str,
    split_hash: str, timeout_seconds: float = 120.0,
) -> list[dict[str, Any]]:
    deadline = time.monotonic() + timeout_seconds
    paths = [output_root / 'rank_done' / f'rank-{rank}.json' for rank in range(world_size)]
    while not all(path.is_file() for path in paths):
        if time.monotonic() >= deadline:
            raise TimeoutError('required rank probability done marker is missing')
        time.sleep(0.25)
    payloads = [json.loads(path.read_text()) for path in paths]
    for rank, payload in enumerate(payloads):
        if payload.get('rank') != rank or payload.get('source_dataset_hash') != source_dataset_hash or payload.get('split_hash') != split_hash:
            raise ValueError('rank done marker identity mismatch')
    return payloads


def write_endpoint_metrics_manifest(
    metrics_path: Path, *, source_dataset_hash: str, split_hash: str,
    endpoint_coverage_hash: str, split_role: str,
) -> Path:
    label_schema_hash, feature_schema_hash = _metric_identity_hashes()
    path = metrics_path.with_suffix('.manifest.json')
    path.write_text(json.dumps({
        'schema_version': 'goal1.5/dl600-endpoint-metrics/v1',
        'source_dataset_hash': source_dataset_hash, 'split_hash': split_hash,
        'label_schema_hash': label_schema_hash, 'feature_schema_hash': feature_schema_hash,
        'endpoint_coverage_hash': endpoint_coverage_hash,
        'sequence_length_seconds': SEQUENCE_LENGTH_CANDIDATE,
        'split_role': split_role, 'file': metrics_path.name,
        'file_sha256': sha256_file(metrics_path),
    }, indent=2, sort_keys=True) + '\n')
    return path


def _marker_manifest_paths(
    payloads: Sequence[Mapping[str, Any]], role: str,
    condition: str | None = None,
) -> list[Path]:
    paths: list[Path] = []
    for payload in payloads:
        manifests = payload['probability_manifests']
        value = manifests[role] if condition is None else manifests[role][condition]
        path = Path(value)
        manifest = json.loads(path.read_text())
        shard = path.parent / manifest['path']
        if sha256_file(shard) != manifest.get('sha256'):
            raise ValueError('rank marker prediction shard hash mismatch')
        paths.append(shard)
    return paths


def run_rank0_postprocess(
    verified: VerifiedSequenceInputs, *, world_size: int,
) -> None:
    payloads = wait_for_rank_done_markers(
        DL_BENCHMARK_OUTPUT_ROOT, world_size=world_size,
        source_dataset_hash=verified.source_dataset_hash,
        split_hash=verified.split_hash,
    )
    clean_paths = _marker_manifest_paths(payloads, 'clean')
    clean_state, endpoint_coverage_hash = stream_lockstep_evaluation(
        clean_paths, verified.index_paths[f'validation_{SEQUENCE_LENGTH_CANDIDATE}'],
    )
    clean_metrics, threshold = finalize_streaming_metrics(
        clean_state, model_name='causal_tcn', split_role='validation',
        threshold_mode='select_clean_validation', stress_condition='clean',
        source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
    )
    assert_unique_metric_rows(clean_metrics)
    metrics_path = DL_BENCHMARK_OUTPUT_ROOT / 'validation_metrics.parquet'
    clean_metrics.to_parquet(metrics_path, index=False)
    metrics_manifest = write_endpoint_metrics_manifest(
        metrics_path, source_dataset_hash=verified.source_dataset_hash,
        split_hash=verified.split_hash, endpoint_coverage_hash=endpoint_coverage_hash,
        split_role='validation',
    )
    threshold_artifact, threshold_artifact_hash = write_clean_validation_threshold(
        DL_BENCHMARK_OUTPUT_ROOT, threshold=threshold,
        source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
        model_name='causal_tcn',
    )
    for condition in STRESS_CONDITIONS:
        stress_paths = _marker_manifest_paths(payloads, 'stress', condition)
        stress_state, stress_coverage = stream_lockstep_evaluation(
            stress_paths, verified.index_paths[f'validation_{SEQUENCE_LENGTH_CANDIDATE}'],
        )
        if stress_coverage != endpoint_coverage_hash:
            raise ValueError('stress endpoint coverage differs from clean validation')
        stress_metrics, _ = finalize_streaming_metrics(
            stress_state, model_name='causal_tcn', split_role='validation',
            threshold_mode='fixed_threshold', stress_condition=condition,
            source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
            threshold_artifact=threshold_artifact,
            threshold_artifact_hash=threshold_artifact_hash,
        )
        assert_unique_metric_rows(stress_metrics)
        stress_path = DL_BENCHMARK_OUTPUT_ROOT / f'validation_noise_{condition}.parquet'
        stress_metrics.to_parquet(stress_path, index=False)
        degradation = compute_noise_degradation(clean_metrics, stress_metrics, condition)
        degradation.to_parquet(
            DL_BENCHMARK_OUTPUT_ROOT / f'validation_noise_degradation_{condition}.parquet',
            index=False,
        )
    if RUN_VALIDATION_COMPARISON:
        compare_ml_dl_on_dl600_endpoints(
            verified=verified, dl_shard_paths=clean_paths,
            dl_metrics_path=metrics_path, dl_manifest_path=metrics_manifest,
        )
    if RUN_LOCKED_TEST:
        locked_paths = _marker_manifest_paths(payloads, 'locked_test')
        locked_state, locked_coverage = stream_lockstep_evaluation(
            locked_paths, verified.index_paths[f'locked_test_{SEQUENCE_LENGTH_CANDIDATE}'],
        )
        locked_metrics, _ = finalize_streaming_metrics(
            locked_state, model_name='causal_tcn', split_role='locked_test',
            threshold_mode='fixed_threshold', stress_condition='clean',
            source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
            threshold_artifact=threshold_artifact,
            threshold_artifact_hash=threshold_artifact_hash,
        )
        assert_unique_metric_rows(locked_metrics)
        locked_path = DL_BENCHMARK_OUTPUT_ROOT / 'locked_test_metrics.parquet'
        locked_metrics.to_parquet(locked_path, index=False)
        write_endpoint_metrics_manifest(
            locked_path, source_dataset_hash=verified.source_dataset_hash,
            split_hash=verified.split_hash, endpoint_coverage_hash=locked_coverage,
            split_role='locked_test',
        )


In [ ]:
def iter_ml_prediction_endpoint_groups(
    prediction_path: Path, *, batch_rows: int = 8192, max_target_rows: int = 16,
):
    columns = [
        'model_family', 'model_name', 'series_id', 'dataset_id', 'run_id',
        'person_key', 'canonical_time', 'split_role', 'target', 'label',
        'probability', 'threshold', 'target_model_id',
    ]
    carry: list[dict[str, Any]] = []
    carry_key: tuple[str, str, str, int] | None = None
    for batch in pq.ParquetFile(prediction_path).iter_batches(batch_size=batch_rows, columns=columns):
        for row in batch.to_pylist():
            key = (
                str(row['person_key']), str(row['run_id']), str(row['dataset_id']),
                int(pd.Timestamp(row['canonical_time']).value),
            )
            if carry_key is not None and key != carry_key:
                yield pd.DataFrame.from_records(carry)
                carry = []
            if carry_key is not None and key < carry_key:
                raise ValueError('ML champion predictions are not endpoint-major sorted')
            carry_key = key
            carry.append(row)
            if len(carry) > max_target_rows:
                raise ValueError('ML endpoint target carry exceeds hierarchical bound')
    if carry:
        yield pd.DataFrame.from_records(carry)


def _endpoint_key_from_expected(expected: Mapping[str, Any]) -> tuple[str, str, str, int]:
    return (
        str(expected['person_key']), str(expected['run_id']), str(expected['dataset_id']),
        int(pd.Timestamp(expected['prediction_time']).value),
    )


def _endpoint_key_from_group(group: pd.DataFrame) -> tuple[str, str, str, int]:
    return (
        str(group['person_key'].iloc[0]), str(group['run_id'].iloc[0]),
        str(group['dataset_id'].iloc[0]), int(pd.Timestamp(group['canonical_time'].iloc[0]).value),
    )


def compare_ml_dl_on_dl600_endpoints(
    *, verified: VerifiedSequenceInputs, dl_shard_paths: Sequence[Path],
    dl_metrics_path: Path, dl_manifest_path: Path,
) -> Path:
    ml_prediction_path = ML_BENCHMARK_OUTPUT_ROOT / 'validation_champion_predictions.parquet'
    ml_manifest_path = ML_BENCHMARK_OUTPUT_ROOT / 'validation_champion_predictions.manifest.json'
    ml_manifest = json.loads(ml_manifest_path.read_text())
    dl_manifest = json.loads(dl_manifest_path.read_text())
    if ml_manifest.get('split_role') != 'validation' or dl_manifest.get('split_role') != 'validation':
        raise ValueError('comparison accepts validation only; locked_test is never reused')
    if ml_manifest.get('source_dataset_hash') != verified.source_dataset_hash or ml_manifest.get('split_hash') != verified.split_hash:
        raise ValueError('ML champion identity differs from DL-600 source/split')
    if ml_manifest.get('prediction_sha256') != sha256_file(ml_prediction_path):
        raise ValueError('ML champion prediction hash mismatch')
    required_targets = {PATTERN_TARGET, *{f'stage::{code}' for code in STAGE_CODES}, *{f'behavior::{code}' for code in BEHAVIOR_CODES}}
    if set(ml_manifest.get('targets', [])) != required_targets:
        raise ValueError('ML champion hierarchical target manifest is incomplete')
    dl_streams = [iter_window_major_prediction_groups(path) for path in dl_shard_paths]
    _ = dl_streams
    dl_groups = iter(merge_window_major_shards(dl_shard_paths))
    ml_groups = iter(iter_ml_prediction_endpoint_groups(ml_prediction_path))
    expected_rows = iter(iter_expected_sequence_windows(
        verified.index_paths[f'validation_{SEQUENCE_LENGTH_CANDIDATE}'],
    ))
    ml_state = StreamingEvaluationState()
    dl_state = StreamingEvaluationState()
    coverage = hashlib.sha256()
    ml_group = next(ml_groups, None)
    for expected in expected_rows:
        expected_key = _endpoint_key_from_expected(expected)
        while ml_group is not None and _endpoint_key_from_group(ml_group) < expected_key:
            ml_group = next(ml_groups, None)
        if ml_group is None or _endpoint_key_from_group(ml_group) != expected_key:
            raise ValueError('ML champion does not cover an exact DL-600 endpoint')
        dl_group = next(dl_groups, None)
        if dl_group is None:
            raise ValueError('DL prediction stream ended before its index')
        validate_window_group_semantics(dl_group, expected)
        ml_group = ml_group.copy()
        ml_group['window_id'] = str(expected['window_id'])
        ml_group['audit_event_binary'] = int(expected[ONSET_EVENT_TARGET])
        validate_window_group_semantics(ml_group, expected)
        label_columns = ['target', 'label']
        if not ml_group[label_columns].sort_values('target').reset_index(drop=True).equals(
            dl_group[label_columns].sort_values('target').reset_index(drop=True)
        ):
            raise ValueError('ML/DL endpoint truth labels differ')
        identity = '|'.join([
            str(expected['person_key']), str(expected['run_id']), str(expected['dataset_id']),
            pd.Timestamp(expected['prediction_time']).isoformat(), str(expected['window_id']),
        ])
        coverage.update((identity + '\n').encode())
        ml_state.update_window(ml_group)
        dl_state.update_window(dl_group)
        ml_group = next(ml_groups, None)
    if next(dl_groups, None) is not None:
        raise ValueError('DL prediction stream has extra endpoints')
    ml_state.finish()
    dl_state.finish()
    endpoint_coverage_hash = coverage.hexdigest()
    if endpoint_coverage_hash != dl_manifest.get('endpoint_coverage_hash'):
        raise ValueError('recomputed DL-600 endpoint coverage hash mismatch')
    ml_metrics, _ = finalize_streaming_metrics(
        ml_state, model_name=str(ml_manifest['model_name']), split_role='validation',
        threshold_mode='select_clean_validation', stress_condition='clean',
        source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
    )
    dl_metrics, _ = finalize_streaming_metrics(
        dl_state, model_name='causal_tcn', split_role='validation',
        threshold_mode='select_clean_validation', stress_condition='clean',
        source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
    )
    keys = ['series_id', 'split_role', 'target', 'metric', 'data_status', 'stress_condition']
    if set(map(tuple, ml_metrics[keys].to_numpy())) != set(map(tuple, dl_metrics[keys].to_numpy())):
        raise ValueError('ML/DL target and common metric grains differ')
    comparison = ml_metrics.merge(dl_metrics, on=keys, how='inner', validate='one_to_one', suffixes=('_ml', '_dl'))
    if not comparison['support_ml'].equals(comparison['support_dl']):
        raise ValueError('ML/DL common endpoint metric support differs')
    comparison['endpoint_coverage_hash'] = endpoint_coverage_hash
    comparison['value_delta_dl_minus_ml'] = comparison['value_dl'] - comparison['value_ml']
    output_path = dl_metrics_path.parent / 'validation_ml_vs_dl600.parquet'
    comparison.to_parquet(output_path, index=False, compression='zstd')
    return output_path


def run_dl_training(epochs: int = 20, batch_size: int = 64) -> None:
    # aggregate_prediction_shards/compute_metrics_from_predictions are superseded by
    # bounded rank0 streaming; threshold_artifact is created only after cleanup_ddp.
    require_exactly_two_cuda_devices()
    if 'LOCAL_RANK' not in os.environ:
        launch_dual_t4_torchrun()
        return
    rank, local_rank, world_size = setup_ddp()
    device = torch.device('cuda', local_rank)
    verified: VerifiedSequenceInputs | None = None
    wandb_module: Any | None = None
    process_group_active = True
    try:
        set_deterministic_seed(SEED, rank)
        verified = verify_sequence_inputs()
        if verified.source_dataset_hash != os.environ.get('GOAL15_EXPECTED_SOURCE_HASH') or verified.split_hash != os.environ.get('GOAL15_EXPECTED_SPLIT_HASH'):
            raise RuntimeError('worker physical identity differs from guarded launcher')
        loss_weights = derive_train_loss_weights(verified.index_paths[f'train_{SEQUENCE_LENGTH_CANDIDATE}'])
        if rank == 0:
            write_train_loss_support(DL_BENCHMARK_OUTPUT_ROOT, loss_weights)
        if rank == 0 and login_wandb_from_kaggle_secret():
            import wandb
            wandb_module = wandb
            safe_wandb_call(wandb.init, project=WANDB_PROJECT, group=WANDB_GROUP, tags=WANDB_TAGS, config={
                'epochs': epochs, 'batch_size': batch_size,
                'sequence_length': SEQUENCE_LENGTH_CANDIDATE, 'data_status': DATA_STATUS,
            })
        train_dataset = Goal15SequenceDataset(
            verified.index_paths[f'train_{SEQUENCE_LENGTH_CANDIDATE}'],
            verified.timeline_paths['train'], verified.normalization, 'train',
        )
        validation_dataset = Goal15SequenceDataset(
            verified.index_paths[f'validation_{SEQUENCE_LENGTH_CANDIDATE}'],
            verified.timeline_paths['validation'], verified.normalization, 'validation',
        )
        train_batch_sampler = PersonBlockBatchSampler(
            train_dataset, batch_size=batch_size, rank=rank,
            num_replicas=world_size, seed=SEED,
        )
        validation_sampler = PersonShardEvalSampler(validation_dataset, rank=rank, num_replicas=world_size)
        train_loader = DataLoader(train_dataset, batch_sampler=train_batch_sampler, num_workers=2, pin_memory=True)
        validation_loader = DataLoader(validation_dataset, batch_size=batch_size, sampler=validation_sampler, num_workers=2, pin_memory=True)
        model = Goal15TCN(input_size=len(ALLOWED_FEATURE_COLUMNS)).to(device)
        parameter_count = assert_parameter_budget(model)
        assert_right_padding_invariance(model, len(ALLOWED_FEATURE_COLUMNS))
        model = DistributedDataParallel(
            model, device_ids=[local_rank], output_device=local_rank,
            broadcast_buffers=False,
        )
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
        scaler = torch.amp.GradScaler('cuda')
        for epoch in range(epochs):
            losses = train_one_epoch(
                model, train_loader, optimizer, scaler, device,
                epoch=epoch, sampler=train_batch_sampler, loss_weights=loss_weights,
            )
            if rank == 0 and wandb_module is not None:
                safe_wandb_call(wandb_module.log, {f'train/{name}': value for name, value in losses.items()}, step=epoch)
        clean_manifest = evaluate_to_prediction_shard(
            model, validation_loader, device, split_role='validation', model_name='causal_tcn',
            rank=rank, output_path=DL_BENCHMARK_OUTPUT_ROOT / 'validation' / 'clean' / f'rank-{rank}.parquet',
            source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
        )
        stress_manifests: dict[str, str] = {}
        for condition in STRESS_CONDITIONS:
            manifest = evaluate_to_prediction_shard(
                model, validation_loader, device, split_role='validation', model_name='causal_tcn',
                rank=rank, output_path=DL_BENCHMARK_OUTPUT_ROOT / 'validation' / 'stress' / condition / f'rank-{rank}.parquet',
                source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
                stress_condition=condition,
            )
            stress_manifests[condition] = str(manifest)
        probability_manifests: dict[str, Any] = {'clean': str(clean_manifest), 'stress': stress_manifests}
        if RUN_LOCKED_TEST:
            locked_dataset = Goal15SequenceDataset(
                verified.index_paths[f'locked_test_{SEQUENCE_LENGTH_CANDIDATE}'],
                verified.timeline_paths['locked_test'], verified.normalization, 'locked_test',
            )
            locked_loader = DataLoader(
                locked_dataset, batch_size=batch_size,
                sampler=PersonShardEvalSampler(locked_dataset, rank=rank, num_replicas=world_size),
                num_workers=2, pin_memory=True,
            )
            locked_manifest = evaluate_to_prediction_shard(
                model, locked_loader, device, split_role='locked_test', model_name='causal_tcn',
                rank=rank, output_path=DL_BENCHMARK_OUTPUT_ROOT / 'locked_test' / f'rank-{rank}.parquet',
                source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
            )
            probability_manifests['locked_test'] = str(locked_manifest)
        if rank == 0:
            export_model_state(model, DL_BENCHMARK_OUTPUT_ROOT)
        write_rank_done_marker(
            DL_BENCHMARK_OUTPUT_ROOT, rank=rank,
            source_dataset_hash=verified.source_dataset_hash,
            split_hash=verified.split_hash,
            probability_manifests=probability_manifests,
        )
        if rank == 0 and wandb_module is not None:
            safe_wandb_call(wandb_module.log, {'model/parameter_count': parameter_count})
            safe_wandb_call(wandb_module.finish)
        cleanup_ddp()
        process_group_active = False
        if rank != 0:
            return
        run_rank0_postprocess(verified, world_size=world_size)
    finally:
        if process_group_active:
            cleanup_ddp()


if RUN_TRAINING:
    print('Round5 nonce/stress/streaming metric 정의를 계속 로드합니다.')
else:
    print('학습 비활성화: T4 x2, W&B, locked test를 실행하지 않았습니다.')


In [ ]:
import uuid


def create_nonce_run_root(output_root: Path, run_nonce: str) -> Path:
    if not run_nonce or run_nonce in {'.', '..'} or '/' in run_nonce or '\\' in run_nonce:
        raise ValueError('run_nonce is not a safe path component')
    run_root = output_root / 'runs' / run_nonce
    run_root.mkdir(parents=True, exist_ok=False)
    return run_root


def requested_stress_conditions(
    enabled: bool, conditions: Sequence[str] | None = None,
) -> tuple[str, ...]:
    selected = STRESS_CONDITIONS if conditions is None else conditions
    return tuple(selected) if enabled else ()


def model_state_sha256(model: DistributedDataParallel) -> str:
    digest = hashlib.sha256()
    for name, tensor in sorted(model.module.state_dict().items()):
        value = tensor.detach().cpu().contiguous()
        digest.update(name.encode())
        digest.update(str(value.dtype).encode())
        digest.update(str(tuple(value.shape)).encode())
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()


def _probability_manifest_receipt(manifest_path: Path, run_root: Path) -> dict[str, str]:
    resolved_manifest = manifest_path.resolve()
    if run_root.resolve() not in resolved_manifest.parents:
        raise ValueError('probability manifest escaped nonce run root')
    manifest = json.loads(resolved_manifest.read_text())
    shard_path = resolved_manifest.parent / manifest['path']
    if run_root.resolve() not in shard_path.resolve().parents:
        raise ValueError('probability shard escaped nonce run root')
    shard_sha256 = sha256_file(shard_path)
    if shard_sha256 != manifest.get('sha256'):
        raise ValueError('probability shard hash differs from manifest')
    return {
        'manifest_path': str(resolved_manifest),
        'manifest_sha256': sha256_file(resolved_manifest),
        'shard_path': str(shard_path.resolve()),
        'shard_sha256': shard_sha256,
    }


def write_rank_done_marker(
    run_root: Path, *, rank: int, run_nonce: str,
    source_dataset_hash: str, split_hash: str, sequence_hash: str,
    model_hash: str, probability_manifests: Mapping[str, Any],
    requested_conditions: Sequence[str], executed_conditions: Sequence[str],
) -> Path:
    if run_root.name != run_nonce:
        raise ValueError('rank marker run_nonce/root mismatch')
    requested = tuple(requested_conditions)
    executed = tuple(executed_conditions)
    if requested != executed:
        raise ValueError('requested/executed stress conditions differ')
    receipts: dict[str, Any] = {
        'clean': _probability_manifest_receipt(Path(probability_manifests['clean']), run_root),
        'stress': {
            condition: _probability_manifest_receipt(Path(probability_manifests['stress'][condition]), run_root)
            for condition in requested
        },
    }
    if 'locked_test' in probability_manifests:
        receipts['locked_test'] = _probability_manifest_receipt(Path(probability_manifests['locked_test']), run_root)
    marker_root = run_root / 'rank_done'
    marker_root.mkdir(parents=True, exist_ok=True)
    path = marker_root / f'rank-{rank}.json'
    temporary = marker_root / f'rank-{rank}.tmp'
    temporary.write_text(json.dumps({
        'schema_version': 'goal1.5/dl-rank-probability-done/v2',
        'rank': rank, 'run_nonce': run_nonce,
        'source_dataset_hash': source_dataset_hash, 'split_hash': split_hash,
        'sequence_hash': sequence_hash, 'model_hash': model_hash,
        'requested_conditions': list(requested),
        'executed_conditions': list(executed),
        'probability_manifests': receipts,
    }, indent=2, sort_keys=True) + '\n')
    temporary.replace(path)
    return path


def wait_for_rank_done_markers(
    run_root: Path, *, world_size: int, run_nonce: str,
    source_dataset_hash: str, split_hash: str, sequence_hash: str,
    model_hash: str, timeout_seconds: float = 120.0,
) -> list[dict[str, Any]]:
    if run_root.name != run_nonce:
        raise ValueError('wait run_nonce/root mismatch')
    deadline = time.monotonic() + timeout_seconds
    paths = [run_root / 'rank_done' / f'rank-{rank}.json' for rank in range(world_size)]
    while not all(path.is_file() for path in paths):
        if time.monotonic() >= deadline:
            raise TimeoutError('current nonce rank marker is missing')
        time.sleep(0.25)
    payloads = [json.loads(path.read_text()) for path in paths]
    expected = {
        'run_nonce': run_nonce, 'source_dataset_hash': source_dataset_hash,
        'split_hash': split_hash, 'sequence_hash': sequence_hash,
        'model_hash': model_hash,
    }
    for rank, payload in enumerate(payloads):
        if payload.get('rank') != rank or any(payload.get(key) != value for key, value in expected.items()):
            raise ValueError('current nonce rank marker identity mismatch')
        receipts = payload.get('probability_manifests', {})
        flat = [receipts.get('clean'), *receipts.get('stress', {}).values()]
        if 'locked_test' in receipts:
            flat.append(receipts['locked_test'])
        for receipt in flat:
            if not isinstance(receipt, dict):
                raise ValueError('rank marker probability receipt is missing')
            manifest_path = Path(receipt['manifest_path'])
            shard_path = Path(receipt['shard_path'])
            if run_root.resolve() not in manifest_path.resolve().parents or run_root.resolve() not in shard_path.resolve().parents:
                raise ValueError('rank marker receipt escaped current nonce root')
            if sha256_file(manifest_path) != receipt['manifest_sha256'] or sha256_file(shard_path) != receipt['shard_sha256']:
                raise ValueError('rank marker shard hashes are stale or changed')
    return payloads


def _marker_manifest_paths(
    payloads: Sequence[Mapping[str, Any]], role: str,
    condition: str | None = None,
) -> list[Path]:
    paths: list[Path] = []
    for payload in payloads:
        receipts = payload['probability_manifests']
        receipt = receipts[role] if condition is None else receipts[role][condition]
        path = Path(receipt['shard_path'])
        if sha256_file(path) != receipt['shard_sha256']:
            raise ValueError('rank marker shard changed after validation')
        paths.append(path)
    return paths


Round4StreamingEvaluationState = StreamingEvaluationState


class StreamingEvaluationState(Round4StreamingEvaluationState):
    '''Compact state; lead = onset minus first fixed-threshold positive inside the causal forecast_60s truth interval.'''
    def __init__(self, *, bins: int = COMMON_THRESHOLD_GRID_SIZE) -> None:
        super().__init__(bins=bins)
        self.person_stage_confusion = np.zeros((len(STAGE_CODES), len(STAGE_CODES)), dtype=np.int64)
        self.forecast_truth_active = False
        self.forecast_alert_active = np.zeros(bins, dtype=bool)
        self.forecast_first_alert_time = np.full(bins, np.nan)
        self.forecast_lead_seconds: list[list[float]] = [[] for _ in range(bins)]
        self.forecast_event_count = 0
        self.forecast_no_prediction_count = np.zeros(bins, dtype=np.int64)
        self.previous_audit_event = False

    def _finish_person(self) -> None:
        if self.current_person is None:
            return
        super()._finish_person()
        self.person_sufficient_statistics[-1]['stage_confusion'] = self.person_stage_confusion.copy()
        self.person_stage_confusion.fill(0)
        self.forecast_truth_active = False
        self.forecast_alert_active.fill(False)
        self.forecast_first_alert_time.fill(np.nan)
        self.previous_audit_event = False

    def update_window(self, group: pd.DataFrame) -> None:
        super().update_window(group)
        stage = group.loc[group['target'].str.startswith('stage::')]
        if not stage.empty:
            actual = str(stage.loc[stage['label'].eq(1), 'target'].iloc[0]).removeprefix('stage::')
            predicted = str(stage.loc[stage['probability'].idxmax(), 'target']).removeprefix('stage::')
            self.person_stage_confusion[STAGE_CODES.index(actual), STAGE_CODES.index(predicted)] += 1
        pattern = group.loc[group['target'].eq(PATTERN_TARGET)].iloc[0]
        timestamp = pd.Timestamp(pattern['canonical_time']).timestamp()
        forecast_truth = bool(pattern['label'])
        audit_event = bool(pattern['audit_event_binary'])
        active = float(pattern['probability']) >= np.linspace(0.0, 1.0, self.bins)
        if forecast_truth:
            self.forecast_truth_active = True
            starting = active & ~self.forecast_alert_active & np.isnan(self.forecast_first_alert_time)
            self.forecast_first_alert_time[starting] = timestamp
            self.forecast_alert_active = active
        onset = audit_event and not self.previous_audit_event
        if onset:
            self.forecast_event_count += 1
            for index, first_time in enumerate(self.forecast_first_alert_time):
                if np.isfinite(first_time):
                    self.forecast_lead_seconds[index].append(max(timestamp - float(first_time), 0.0))
                else:
                    # no predicted alert before onset is retained as explicit missing support
                    self.forecast_no_prediction_count[index] += 1
            self.forecast_truth_active = False
            self.forecast_alert_active.fill(False)
            self.forecast_first_alert_time.fill(np.nan)
        elif not forecast_truth and not audit_event and self.forecast_truth_active:
            self.forecast_truth_active = False
            self.forecast_alert_active.fill(False)
            self.forecast_first_alert_time.fill(np.nan)
        self.previous_audit_event = audit_event


round4_finalize_streaming_metrics = finalize_streaming_metrics


def finalize_streaming_metrics(
    state: StreamingEvaluationState, *, model_name: str, split_role: str,
    threshold_mode: str, stress_condition: str,
    source_dataset_hash: str, split_hash: str,
    threshold_artifact: Path | None = None,
    threshold_artifact_hash: str | None = None,
) -> tuple[pd.DataFrame, float]:
    metrics, threshold = round4_finalize_streaming_metrics(
        state, model_name=model_name, split_role=split_role,
        threshold_mode=threshold_mode, stress_condition=stress_condition,
        source_dataset_hash=source_dataset_hash, split_hash=split_hash,
        threshold_artifact=threshold_artifact,
        threshold_artifact_hash=threshold_artifact_hash,
    )
    inherited_behavior_metrics = {
        'behavior_code_aucpr', 'behavior_code_auroc', 'behavior_code_f1',
        'behavior_positive_support', 'behavior_macro_aucpr',
        'behavior_macro_auroc', 'behavior_macro_f1',
        'behavior_micro_aucpr', 'behavior_micro_auroc', 'behavior_micro_f1',
    }
    for code in BEHAVIOR_CODES:
        code_metrics = set(metrics.loc[metrics['target'].eq(f'behavior::{code}'), 'metric'])
        if not {'behavior_code_aucpr', 'behavior_code_auroc', 'behavior_code_f1', 'behavior_positive_support'}.issubset(code_metrics):
            raise ValueError(f'inherited behavior metric support missing: {code}')
    if not inherited_behavior_metrics.intersection(set(metrics['metric'])):
        raise ValueError('inherited behavior macro/micro metrics are missing')
    rows: list[dict[str, Any]] = []
    confusion = state.stage_confusion
    for actual in STAGE_CODES:
        actual_index = STAGE_CODES.index(actual)
        actual_support = int(confusion[actual_index].sum())
        global_recall = float(confusion[actual_index, actual_index] / actual_support) if actual_support else np.nan
        person_recalls = []
        for person in state.person_sufficient_statistics:
            person_confusion = person['stage_confusion']
            support = int(person_confusion[actual_index].sum())
            if support:
                person_recalls.append(float(person_confusion[actual_index, actual_index] / support))
        rows.append(_round3_metric_row(
            model_name, split_role, f'stage::{actual}', 'stage_global_recall',
            global_recall, actual_support, stress_condition,
        ))
        rows.append(_round3_metric_row(
            model_name, split_role, f'stage::{actual}', 'stage_person_macro_recall',
            float(np.mean(person_recalls)) if person_recalls else np.nan,
            len(person_recalls), stress_condition,
        ))
        for predicted in STAGE_CODES:
            predicted_index = STAGE_CODES.index(predicted)
            count = int(confusion[actual_index, predicted_index])
            rows.append(_round3_metric_row(
                model_name, split_role, f'stage::{actual}',
                f'stage_confusion_count::{predicted}', float(count), count,
                stress_condition,
            ))
    _ = ('stage_macro_f1', 'stage_balanced_accuracy')
    threshold_index = int(round(threshold * (state.bins - 1)))
    leads = state.forecast_lead_seconds[threshold_index]
    lead_support = len(leads)
    for metric, value in (
        ('forecast_lead_mean_seconds', float(np.mean(leads)) if leads else np.nan),
        ('forecast_lead_median_seconds', float(np.median(leads)) if leads else np.nan),
        ('forecast_lead_support', float(lead_support)),
        ('forecast_event_support', float(state.forecast_event_count)),
        ('forecast_no_prediction_support', float(state.forecast_no_prediction_count[threshold_index])),
    ):
        rows.append(_round3_metric_row(
            model_name, split_role, PATTERN_TARGET, metric, value,
            state.forecast_event_count, stress_condition,
        ))
    extended = pd.concat([metrics, pd.DataFrame(rows, columns=ROUND3_METRIC_COLUMNS)], ignore_index=True)
    assert_unique_metric_rows(extended)
    return extended, threshold


def write_run_audit_manifest(
    run_root: Path, *, run_nonce: str, status: str,
    requested_conditions: Sequence[str], executed_conditions: Sequence[str],
    source_dataset_hash: str, split_hash: str,
    sequence_hash: str, model_hash: str,
) -> Path:
    path = run_root / 'run_execution_manifest.json'
    path.write_text(json.dumps({
        'schema_version': 'goal1.5/dl-run-execution/v1',
        'run_nonce': run_nonce, 'status': status,
        'retention_policy': 'RETAIN_FOR_AUDIT',
        'safe_cleanup': 'manual_after_registry_review',
        'requested_conditions': list(requested_conditions),
        'executed_conditions': list(executed_conditions),
        'source_dataset_hash': source_dataset_hash, 'split_hash': split_hash,
        'sequence_hash': sequence_hash, 'model_hash': model_hash,
    }, indent=2, sort_keys=True) + '\n')
    return path


def run_rank0_postprocess(
    verified: VerifiedSequenceInputs, *, world_size: int, run_root: Path,
    run_nonce: str, sequence_hash: str, model_hash: str,
) -> None:
    payloads = wait_for_rank_done_markers(
        run_root, world_size=world_size, run_nonce=run_nonce,
        source_dataset_hash=verified.source_dataset_hash,
        split_hash=verified.split_hash, sequence_hash=sequence_hash,
        model_hash=model_hash,
    )
    requested_conditions = tuple(payloads[0]['requested_conditions'])
    for payload in payloads:
        if tuple(payload['requested_conditions']) != requested_conditions or tuple(payload['executed_conditions']) != requested_conditions:
            raise ValueError('rank stress request/execution manifests differ')
    clean_paths = _marker_manifest_paths(payloads, 'clean')
    clean_state, endpoint_coverage_hash = stream_lockstep_evaluation(
        clean_paths, verified.index_paths[f'validation_{SEQUENCE_LENGTH_CANDIDATE}'],
    )
    clean_metrics, threshold = finalize_streaming_metrics(
        clean_state, model_name='causal_tcn', split_role='validation',
        threshold_mode='select_clean_validation', stress_condition='clean',
        source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
    )
    assert_unique_metric_rows(clean_metrics)
    metrics_path = run_root / 'validation_metrics.parquet'
    clean_metrics.to_parquet(metrics_path, index=False)
    stage_confusion = clean_metrics.loc[clean_metrics['metric'].str.startswith('stage_confusion_count::')].copy()
    stage_confusion.to_parquet(run_root / 'validation_stage_confusion.parquet', index=False)
    metrics_manifest = write_endpoint_metrics_manifest(
        metrics_path, source_dataset_hash=verified.source_dataset_hash,
        split_hash=verified.split_hash, endpoint_coverage_hash=endpoint_coverage_hash,
        split_role='validation',
    )
    threshold_artifact, threshold_artifact_hash = write_clean_validation_threshold(
        run_root, threshold=threshold,
        source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
        model_name='causal_tcn',
    )
    for condition in requested_conditions:
        stress_paths = _marker_manifest_paths(payloads, 'stress', condition)
        stress_state, stress_coverage = stream_lockstep_evaluation(
            stress_paths, verified.index_paths[f'validation_{SEQUENCE_LENGTH_CANDIDATE}'],
        )
        if stress_coverage != endpoint_coverage_hash:
            raise ValueError('stress endpoint coverage differs from clean validation')
        stress_metrics, _ = finalize_streaming_metrics(
            stress_state, model_name='causal_tcn', split_role='validation',
            threshold_mode='fixed_threshold', stress_condition=condition,
            source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
            threshold_artifact=threshold_artifact,
            threshold_artifact_hash=threshold_artifact_hash,
        )
        assert_unique_metric_rows(stress_metrics)
        stress_path = run_root / f'validation_noise_{condition}.parquet'
        stress_metrics.to_parquet(stress_path, index=False)
        degradation = compute_noise_degradation(clean_metrics, stress_metrics, condition)
        degradation.to_parquet(run_root / f'validation_noise_degradation_{condition}.parquet', index=False)
    if RUN_VALIDATION_COMPARISON:
        compare_ml_dl_on_dl600_endpoints(
            verified=verified, dl_shard_paths=clean_paths,
            dl_metrics_path=metrics_path, dl_manifest_path=metrics_manifest,
        )
    if RUN_LOCKED_TEST:
        locked_paths = _marker_manifest_paths(payloads, 'locked_test')
        locked_state, locked_coverage = stream_lockstep_evaluation(
            locked_paths, verified.index_paths[f'locked_test_{SEQUENCE_LENGTH_CANDIDATE}'],
        )
        locked_metrics, _ = finalize_streaming_metrics(
            locked_state, model_name='causal_tcn', split_role='locked_test',
            threshold_mode='fixed_threshold', stress_condition='clean',
            source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
            threshold_artifact=threshold_artifact,
            threshold_artifact_hash=threshold_artifact_hash,
        )
        assert_unique_metric_rows(locked_metrics)
        locked_path = run_root / 'locked_test_metrics.parquet'
        locked_metrics.to_parquet(locked_path, index=False)
        write_endpoint_metrics_manifest(
            locked_path, source_dataset_hash=verified.source_dataset_hash,
            split_hash=verified.split_hash, endpoint_coverage_hash=locked_coverage,
            split_role='locked_test',
        )
    write_run_audit_manifest(
        run_root, run_nonce=run_nonce, status='COMPLETED',
        requested_conditions=requested_conditions,
        executed_conditions=requested_conditions,
        source_dataset_hash=verified.source_dataset_hash,
        split_hash=verified.split_hash, sequence_hash=sequence_hash,
        model_hash=model_hash,
    )


def write_guarded_torchrun_worker(source: str, run_root: Path) -> Path:
    path = run_root / 'goal15_tcn_worker.py'
    path.write_text(source)
    return path


def launch_dual_t4_torchrun() -> None:
    require_exactly_two_cuda_devices()
    verified = verify_sequence_inputs()
    run_nonce = str(uuid.uuid4())
    run_root = create_nonce_run_root(DL_BENCHMARK_OUTPUT_ROOT, run_nonce)
    if Path('/kaggle/working') not in run_root.parents:
        raise RuntimeError('nonce run root must stay under /kaggle/working')
    worker_source = build_torchrun_worker_source(verified.source_dataset_hash, verified.split_hash)
    worker_path = write_guarded_torchrun_worker(worker_source, run_root)
    environment = os.environ.copy()
    environment['GOAL15_EXPECTED_SOURCE_HASH'] = verified.source_dataset_hash
    environment['GOAL15_EXPECTED_SPLIT_HASH'] = verified.split_hash
    environment['GOAL15_RUN_NONCE'] = run_nonce
    environment['GOAL15_RUN_ROOT'] = str(run_root)
    command = ['python', '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2', str(worker_path)]
    try:
        subprocess.run(command, check=True, env=environment)
    except Exception:
        write_run_audit_manifest(
            run_root, run_nonce=run_nonce, status='FAILED_RETAINED',
            requested_conditions=requested_stress_conditions(RUN_NOISE_STRESS),
            executed_conditions=(), source_dataset_hash=verified.source_dataset_hash,
            split_hash=verified.split_hash, sequence_hash='NOT_AVAILABLE',
            model_hash='NOT_AVAILABLE',
        )
        raise


def run_dl_training(epochs: int = 20, batch_size: int = 64) -> None:
    # aggregate_prediction_shards/compute_metrics_from_predictions are replaced by
    # rank0 bounded streaming; threshold_artifact is created only after cleanup_ddp.
    require_exactly_two_cuda_devices()
    if 'LOCAL_RANK' not in os.environ:
        launch_dual_t4_torchrun()
        return
    rank, local_rank, world_size = setup_ddp()
    device = torch.device('cuda', local_rank)
    verified: VerifiedSequenceInputs | None = None
    wandb_module: Any | None = None
    process_group_active = True
    try:
        set_deterministic_seed(SEED, rank)
        verified = verify_sequence_inputs()
        if verified.source_dataset_hash != os.environ.get('GOAL15_EXPECTED_SOURCE_HASH') or verified.split_hash != os.environ.get('GOAL15_EXPECTED_SPLIT_HASH'):
            raise RuntimeError('worker physical identity differs from guarded launcher')
        run_nonce = os.environ.get('GOAL15_RUN_NONCE', '')
        run_root = Path(os.environ.get('GOAL15_RUN_ROOT', ''))
        if not run_nonce or run_root.name != run_nonce or not run_root.is_dir():
            raise RuntimeError('current nonce run root is missing or mismatched')
        sequence_hash = sha256_file(verified.index_paths[f'validation_{SEQUENCE_LENGTH_CANDIDATE}'])
        requested_conditions = requested_stress_conditions(RUN_NOISE_STRESS)
        loss_weights = derive_train_loss_weights(verified.index_paths[f'train_{SEQUENCE_LENGTH_CANDIDATE}'])
        if rank == 0:
            write_train_loss_support(run_root, loss_weights)
        if rank == 0 and login_wandb_from_kaggle_secret():
            import wandb
            wandb_module = wandb
            safe_wandb_call(wandb.init, project=WANDB_PROJECT, group=WANDB_GROUP, tags=WANDB_TAGS, config={
                'epochs': epochs, 'batch_size': batch_size,
                'sequence_length': SEQUENCE_LENGTH_CANDIDATE,
                'run_nonce': run_nonce, 'noise_stress': bool(RUN_NOISE_STRESS),
                'data_status': DATA_STATUS,
            })
        train_dataset = Goal15SequenceDataset(
            verified.index_paths[f'train_{SEQUENCE_LENGTH_CANDIDATE}'],
            verified.timeline_paths['train'], verified.normalization, 'train',
        )
        validation_dataset = Goal15SequenceDataset(
            verified.index_paths[f'validation_{SEQUENCE_LENGTH_CANDIDATE}'],
            verified.timeline_paths['validation'], verified.normalization, 'validation',
        )
        train_batch_sampler = PersonBlockBatchSampler(
            train_dataset, batch_size=batch_size, rank=rank,
            num_replicas=world_size, seed=SEED,
        )
        validation_sampler = PersonShardEvalSampler(validation_dataset, rank=rank, num_replicas=world_size)
        train_loader = DataLoader(train_dataset, batch_sampler=train_batch_sampler, num_workers=2, pin_memory=True)
        validation_loader = DataLoader(validation_dataset, batch_size=batch_size, sampler=validation_sampler, num_workers=2, pin_memory=True)
        model = Goal15TCN(input_size=len(ALLOWED_FEATURE_COLUMNS)).to(device)
        parameter_count = assert_parameter_budget(model)
        assert_right_padding_invariance(model, len(ALLOWED_FEATURE_COLUMNS))
        model = DistributedDataParallel(model, device_ids=[local_rank], output_device=local_rank, broadcast_buffers=False)
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
        scaler = torch.amp.GradScaler('cuda')
        for epoch in range(epochs):
            losses = train_one_epoch(
                model, train_loader, optimizer, scaler, device,
                epoch=epoch, sampler=train_batch_sampler, loss_weights=loss_weights,
            )
            if rank == 0 and wandb_module is not None:
                safe_wandb_call(wandb_module.log, {f'train/{name}': value for name, value in losses.items()}, step=epoch)
        model_hash = model_state_sha256(model)
        clean_manifest = evaluate_to_prediction_shard(
            model, validation_loader, device, split_role='validation', model_name='causal_tcn',
            rank=rank, output_path=run_root / 'validation' / 'clean' / f'rank-{rank}.parquet',
            source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
        )
        stress_manifests: dict[str, str] = {}
        for condition in requested_conditions:
            manifest = evaluate_to_prediction_shard(
                model, validation_loader, device, split_role='validation', model_name='causal_tcn',
                rank=rank, output_path=run_root / 'validation' / 'stress' / condition / f'rank-{rank}.parquet',
                source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
                stress_condition=condition,
            )
            stress_manifests[condition] = str(manifest)
        probability_manifests: dict[str, Any] = {'clean': str(clean_manifest), 'stress': stress_manifests}
        if RUN_LOCKED_TEST:
            locked_dataset = Goal15SequenceDataset(
                verified.index_paths[f'locked_test_{SEQUENCE_LENGTH_CANDIDATE}'],
                verified.timeline_paths['locked_test'], verified.normalization, 'locked_test',
            )
            locked_loader = DataLoader(
                locked_dataset, batch_size=batch_size,
                sampler=PersonShardEvalSampler(locked_dataset, rank=rank, num_replicas=world_size),
                num_workers=2, pin_memory=True,
            )
            probability_manifests['locked_test'] = str(evaluate_to_prediction_shard(
                model, locked_loader, device, split_role='locked_test', model_name='causal_tcn',
                rank=rank, output_path=run_root / 'locked_test' / f'rank-{rank}.parquet',
                source_dataset_hash=verified.source_dataset_hash, split_hash=verified.split_hash,
            ))
        if rank == 0:
            export_model_state(model, run_root)
        write_rank_done_marker(
            run_root, rank=rank, run_nonce=run_nonce,
            source_dataset_hash=verified.source_dataset_hash,
            split_hash=verified.split_hash, sequence_hash=sequence_hash,
            model_hash=model_hash, probability_manifests=probability_manifests,
            requested_conditions=requested_conditions,
            executed_conditions=tuple(stress_manifests),
        )
        if rank == 0 and wandb_module is not None:
            safe_wandb_call(wandb_module.log, {'model/parameter_count': parameter_count})
            safe_wandb_call(wandb_module.finish)
        cleanup_ddp()
        process_group_active = False
        if rank != 0:
            return
        run_rank0_postprocess(
            verified, world_size=world_size, run_root=run_root,
            run_nonce=run_nonce, sequence_hash=sequence_hash,
            model_hash=model_hash,
        )
    finally:
        if process_group_active:
            cleanup_ddp()


if RUN_TRAINING:
    run_dl_training()
else:
    print('학습 비활성화: nonce-isolated T4 x2 실행을 시작하지 않았습니다.')
